# Fermentation model calibration — effective-parameter reformulation

This is a parallel version of the updated calibration notebook. It keeps the revised workflow from the attached base notebook — data preprocessing, Pyomo DAE simulation, Gaussian WSSE/ParmEst calibration, optional leave-one-batch-out CV, local sensitivity/FIM diagnostics, uncertainty quantification, profile likelihood, post-optimization simulation, and optional PSO — but estimates an effective parameter vector instead of the original yield/saturation vector directly.

The reformulation targets practical identifiability issues by replacing several weakly separable physical parameters with combinations that appear directly in the mass balances:

- growth and nitrogen/sugar uptake use effective uptake rates such as `qN = mu0/Yxn`, `qXG = mu0/Yxg`, and `qXF = mu0/Yxf`;
- fermentation sugar uptake uses `qEG = betaG0/Yeg` and `qEF = betaF0/Yef`;
- saturation constants are represented through slope-like ratios such as `sN = mu0/Kn0`, `sG = betaG0/Kg0`, and `sF = betaF0/Kf0`;
- inhibition constants are estimated as inverse sensitivities, `iG = 1/Kig0` and `iE = 1/Kie0`.

The notebook still reports the back-transformed physical parameters for interpretation, but all estimation/diagnostic blocks operate in the effective parameter space by default.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os
import shutil
import sys
import tempfile

_NOTEBOOK_START_DIR = Path.cwd().resolve()
if (_NOTEBOOK_START_DIR / "fermentation_model").is_dir():
    REPO_ROOT = _NOTEBOOK_START_DIR
elif _NOTEBOOK_START_DIR.name == "fermentation_model":
    REPO_ROOT = _NOTEBOOK_START_DIR.parent
else:
    REPO_ROOT = _NOTEBOOK_START_DIR

REPO_SCRATCH_ROOT = Path(os.environ.get("PYOMO_DOE_SCRATCH_DIR", REPO_ROOT / ".scratch")).resolve()
PYOMO_SOLVER_TMP_DIR = REPO_SCRATCH_ROOT / "pyomo-temp"
PYOMO_CACHE_DIR = REPO_SCRATCH_ROOT / "cache"
PYOMO_MPLCONFIG_DIR = REPO_SCRATCH_ROOT / "matplotlib"
PYOMO_JUPYTER_RUNTIME_DIR = REPO_SCRATCH_ROOT / "jupyter-runtime"
for _path in [PYOMO_SOLVER_TMP_DIR, PYOMO_CACHE_DIR, PYOMO_MPLCONFIG_DIR, PYOMO_JUPYTER_RUNTIME_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

for _env_name, _env_path in {
    "TMPDIR": PYOMO_SOLVER_TMP_DIR,
    "TMP": PYOMO_SOLVER_TMP_DIR,
    "TEMP": PYOMO_SOLVER_TMP_DIR,
    "XDG_CACHE_HOME": PYOMO_CACHE_DIR,
    "MPLCONFIGDIR": PYOMO_MPLCONFIG_DIR,
    "JUPYTER_RUNTIME_DIR": PYOMO_JUPYTER_RUNTIME_DIR,
}.items():
    os.environ[_env_name] = str(_env_path)
tempfile.tempdir = str(PYOMO_SOLVER_TMP_DIR)

SOLVER_THREAD_COUNT = str(os.environ.get("PYOMO_DOE_SOLVER_THREADS", "1"))
for _thread_env_name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[_thread_env_name] = SOLVER_THREAD_COUNT

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
import pyomo.dae as dae
import pyomo.contrib.parmest.parmest as parmest
from pyomo.common.tempfiles import TempfileManager
try:
    from scipy.integrate import solve_ivp
except ModuleNotFoundError:
    solve_ivp = None
from pyomo.contrib.parmest.experiment import Experiment
from IPython.display import display

TempfileManager.tempdir = str(PYOMO_SOLVER_TMP_DIR)

IDAES_AVAILABLE = False
IDAES_BIN = None
try:
    from idaes.config import bin_directory as _IDAES_BIN

    IDAES_AVAILABLE = True
    IDAES_BIN = Path(_IDAES_BIN)
except ModuleNotFoundError:
    localappdata = Path(os.environ.get("LOCALAPPDATA", Path.home() / "AppData" / "Local"))
    candidate_idaes_bin = localappdata / "idaes" / "bin"
    if candidate_idaes_bin.exists():
        IDAES_BIN = candidate_idaes_bin
    print("WARNING: the active notebook kernel cannot import idaes.")
    print("Kernel Python:", sys.executable)
    print("Install IDAES in this kernel with:")
    print(f'"{sys.executable}" -m pip install idaes-pse')
    print("Then run: idaes get-extensions")
    print("After installation, restart this notebook kernel.")

def _remove_unstable_solver_library_overrides():
    """Avoid LD_LIBRARY_PATH entries that make external Ipopt binaries segfault."""
    raw_path = os.environ.get("LD_LIBRARY_PATH", "")
    if not raw_path:
        return []
    kept = []
    removed = []
    for entry in raw_path.split(os.pathsep):
        if not entry:
            continue
        parts = {part.lower() for part in Path(entry).expanduser().parts}
        if "hsl" in " ".join(parts) and "opt" in parts:
            removed.append(entry)
        else:
            kept.append(entry)
    if removed:
        if kept:
            os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(kept)
        else:
            os.environ.pop("LD_LIBRARY_PATH", None)
    return removed


SOLVER_LD_LIBRARY_PATH_REMOVED = _remove_unstable_solver_library_overrides()

KERNEL_BIN = Path(sys.executable).resolve().parent
solver_path_entries = [str(KERNEL_BIN)]
if IDAES_BIN is not None:
    solver_path_entries.append(str(IDAES_BIN))
solver_path_entries.append(os.environ.get("PATH", ""))
os.environ["PATH"] = os.pathsep.join(entry for entry in solver_path_entries if entry)


def _solver_executable(name):
    override = os.environ.get(f"PYOMO_DOE_{name.upper()}_EXECUTABLE")
    if override:
        override_path = Path(override).expanduser().resolve()
        if override_path.exists():
            return str(override_path)
        raise FileNotFoundError(f"Configured solver executable for {name} does not exist: {override_path}")
    kernel_candidate = KERNEL_BIN / name
    if kernel_candidate.exists():
        return str(kernel_candidate)
    return shutil.which(name)


IPOPT_EXECUTABLE = _solver_executable("ipopt")
KAUG_EXECUTABLE = _solver_executable("k_aug")
DOT_SENS_EXECUTABLE = _solver_executable("dot_sens")
if IPOPT_EXECUTABLE is None:
    raise RuntimeError("Ipopt was not found. Install IDAES extensions or another Ipopt executable before solving.")

IPOPT_LINEAR_SOLVER = os.environ.get("PYOMO_DOE_IPOPT_LINEAR_SOLVER", "").strip() or None

IPOPT_SOLVER_OPTIONS = {
    "max_iter": 5000,
    "tol": 1e-7,
    "acceptable_tol": 1e-6,
    "acceptable_constr_viol_tol": 1e-6,
    "mu_strategy": "adaptive",
    "max_cpu_time": float(os.environ.get("PYOMO_DOE_IPOPT_MAX_CPU_TIME", "1800")),
}
if IPOPT_LINEAR_SOLVER is not None:
    IPOPT_SOLVER_OPTIONS["linear_solver"] = IPOPT_LINEAR_SOLVER

if not hasattr(pyo, "_fermentation_original_solver_factory"):
    pyo._fermentation_original_solver_factory = pyo.SolverFactory


def fermentation_solver_factory(name, *args, **kwargs):
    if name == "ipopt":
        kwargs.setdefault("executable", IPOPT_EXECUTABLE)
    solver = pyo._fermentation_original_solver_factory(name, *args, **kwargs)
    if name == "ipopt" and solver is not None:
        for key, value in IPOPT_SOLVER_OPTIONS.items():
            solver.options[key] = value
    return solver


pyo.SolverFactory = fermentation_solver_factory

print("IDAES Python package available:", IDAES_AVAILABLE)
print("IDAES bin:", IDAES_BIN)
print("Ipopt executable:", IPOPT_EXECUTABLE)
print("k_aug executable:", KAUG_EXECUTABLE)
print("dot_sens executable:", DOT_SENS_EXECUTABLE)
print("Ipopt linear solver:", IPOPT_LINEAR_SOLVER)
print("Removed solver LD_LIBRARY_PATH entries:", SOLVER_LD_LIBRARY_PATH_REMOVED)
print("Solver thread count:", SOLVER_THREAD_COUNT)
print("Repo scratch root:", REPO_SCRATCH_ROOT)
print("Pyomo/Ipopt tempdir:", PYOMO_SOLVER_TMP_DIR)

plt.rc("font", size=12)
plt.rc("axes", titlesize=13)
plt.rc("axes", labelsize=12)
plt.rc("legend", fontsize=10)
plt.rc("lines", linewidth=2)

## Paths and experimental data

The notebook can be run either from the repository root or from the `fermentation_model` folder. The Excel workbook contains one sheet per experimental batch.

In [ ]:
CWD = Path.cwd()
if (CWD / "data" / "Calibration_data_vl3.xlsx").exists():
    FERMENTATION_DIR = CWD
elif (CWD / "fermentation_model" / "data" / "Calibration_data_vl3.xlsx").exists():
    FERMENTATION_DIR = CWD / "fermentation_model"
else:
    raise FileNotFoundError(
        "Could not locate fermentation_model/data/Calibration_data_vl3.xlsx. "
        "Run this notebook from the repo root or from fermentation_model/."
    )

DATA_FILE = FERMENTATION_DIR / "data" / "Calibration_data_vl3.xlsx"
CONTEXT_DIR = FERMENTATION_DIR / "context"

xl = pd.ExcelFile(DATA_FILE)
available_batches = xl.sheet_names
print("Available batches:", available_batches)

summary_rows = []
for sheet in available_batches:
    df = xl.parse(sheet)
    state_rows = df[["Viability", "YAN", "GLUCOSE", "FRUCTOSE", "ETANOL"]].dropna(how="all").shape[0]
    summary_rows.append(
        {
            "batch": sheet,
            "rows": len(df),
            "t_min_h": df["t"].min(),
            "t_max_h": df["t"].max(),
            "temperature_rows": int(df["temperatura"].notna().sum()),
            "state_rows": state_rows,
        }
    )

data_summary = pd.DataFrame(summary_rows).set_index("batch")
display(data_summary)

## Data preprocessing

The Zenteno model uses kg/m3. For dilute aqueous concentrations, g/L and kg/m3 have the same numerical value, so glucose, fructose, and ethanol are used directly. `YAN` is reported as mg/L N and is converted to kg/m3 by dividing by 1000. Biomass `X` is computed from `Viability`, which is reported as millions of viable cells per mL, using `3e-11 g/cell`. Nutrition pulses from `pulso_nut` are also reported as mg/L YAN and are converted to kg/m3. If a pulse is entered on the first row, it is treated as initial YAN and added to `N0`. Low YAN plateau values are treated as depletion, and late N/X measurements after depletion are excluded from the weighted calibration objective.

In [ ]:
CELL_MASS_G_PER_CELL = 3e-11
VIABILITY_TO_KG_M3 = 1e6 * CELL_MASS_G_PER_CELL * 1000.0  # million cells/mL -> g/L == kg/m3
MG_L_TO_KG_M3 = 1e-3

APPLY_INITIAL_YAN_PULSE_TO_N0 = True
N_STAGNATION_THRESHOLD_KG_M3 = 0.02
N_MEASUREMENT_KEEP_AFTER_DEPLETION_H = 12.0
X_MEASUREMENT_KEEP_AFTER_DEPLETION_H = 24.0
BIOMASS_MEASUREMENT_ERROR_MULTIPLIER = 3.0
BIOMASS_MEASUREMENT_ERROR_MIN_KG_M3 = 0.50

# Internal numerical stabilization for the Pyomo DAE simulation.
SIMULATION_INTERNAL_STEP_H = 3.0
USE_DYNAMIC_INITIALIZATION = True
USE_SMOOTH_MEASURED_PULSES = True
MEASURED_PULSE_WIDTH_H = 3.0
MEASURED_PULSE_AFTER_SAMPLE = True
KINETIC_SOFTPLUS_EPS = 1e-6
MAINTENANCE_SUGAR_CUTOFF_KG_M3 = 0.1

STATE_COLUMNS = {
    "X": "Viability",  # million viable cells/mL -> viable biomass kg/m3
    "N": "YAN",       # assimilable nitrogen, mg/L N -> kg/m3
    "G": "GLUCOSE",   # glucose, g/L == kg/m3
    "F": "FRUCTOSE",  # fructose, g/L == kg/m3
    "E": "ETANOL",    # ethanol, g/L == kg/m3
}

STATE_LABELS = {
    "X": "Viable biomass X (kg/m3)",
    "N": "Assimilable nitrogen N (kg/m3)",
    "G": "Glucose G (kg/m3)",
    "F": "Fructose F (kg/m3)",
    "E": "Ethanol E (kg/m3)",
}

@dataclass
class FermentationBatch:
    batch_id: str
    run_label: str
    raw: pd.DataFrame
    time: np.ndarray
    temperature_c: np.ndarray
    nutrient_pulse_kg_m3: np.ndarray
    nutrient_pulse_rate_kg_m3_h: np.ndarray
    measurements: pd.DataFrame
    initial_guess: pd.DataFrame
    initials: dict


def clean_sheet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = df[pd.notna(df["t"])].sort_values("t")
    df["t"] = pd.to_numeric(df["t"], errors="coerce").astype(float)
    df = df.dropna(subset=["t"]).drop_duplicates("t", keep="first")
    return df.reset_index(drop=True)


def nitrogen_depletion_time(time, nitrogen_kg_m3, nutrient_pulse_kg_m3):
    valid_nitrogen = pd.Series(nitrogen_kg_m3, index=time).dropna()
    if valid_nitrogen.empty:
        return None
    pulse_times = np.asarray(time, dtype=float)[np.asarray(nutrient_pulse_kg_m3, dtype=float) > 0.0]
    last_pulse_time = float(np.max(pulse_times)) if len(pulse_times) else float(valid_nitrogen.index.min())
    post_pulse = valid_nitrogen.loc[valid_nitrogen.index >= last_pulse_time]
    depleted = post_pulse[post_pulse <= N_STAGNATION_THRESHOLD_KG_M3]
    if depleted.empty:
        return None
    return float(depleted.index.min())


def apply_measurement_preprocessing(measurements, time, nutrient_pulse_kg_m3):
    measurements = measurements.copy()
    initial_yan_pulse = float(nutrient_pulse_kg_m3[0]) if len(nutrient_pulse_kg_m3) else 0.0
    if APPLY_INITIAL_YAN_PULSE_TO_N0 and initial_yan_pulse > 0.0:
        first_time = float(time[0])
        if pd.notna(measurements.loc[first_time, "N"]):
            measurements.loc[first_time, "N"] = float(measurements.loc[first_time, "N"]) + initial_yan_pulse

    measurements.loc[measurements["N"] <= N_STAGNATION_THRESHOLD_KG_M3, "N"] = 0.0

    depletion_time = nitrogen_depletion_time(time, measurements["N"].to_numpy(dtype=float), nutrient_pulse_kg_m3)
    if depletion_time is not None:
        n_cutoff = depletion_time + N_MEASUREMENT_KEEP_AFTER_DEPLETION_H
        x_cutoff = depletion_time + X_MEASUREMENT_KEEP_AFTER_DEPLETION_H
        measurements.loc[measurements.index > n_cutoff, "N"] = np.nan
        measurements.loc[measurements.index > x_cutoff, "X"] = np.nan

    return measurements


def load_batch(batch_id: str) -> FermentationBatch:
    raw = clean_sheet(pd.read_excel(DATA_FILE, sheet_name=str(batch_id)))
    time = raw["t"].to_numpy(dtype=float)
    temperature_c = pd.to_numeric(raw["temperatura"], errors="coerce").interpolate(
        limit_direction="both"
    ).to_numpy(dtype=float)
    nutrient_pulse_kg_m3 = pd.to_numeric(raw["pulso_nut"], errors="coerce").fillna(0.0).to_numpy(dtype=float) * MG_L_TO_KG_M3
    nutrient_pulse_rate_kg_m3_h = np.zeros_like(nutrient_pulse_kg_m3, dtype=float)
    for k in range(1, len(time)):
        dt = time[k] - time[k - 1]
        if dt <= 0:
            raise ValueError(f"Batch {batch_id} has non-increasing time at index {k}.")
        nutrient_pulse_rate_kg_m3_h[k] = nutrient_pulse_kg_m3[k] / dt

    measurements = pd.DataFrame(index=time)
    measurements.index.name = "t"
    measurements["X"] = pd.to_numeric(raw["Viability"], errors="coerce").to_numpy(dtype=float) * VIABILITY_TO_KG_M3
    measurements["N"] = pd.to_numeric(raw["YAN"], errors="coerce").to_numpy(dtype=float) / 1000.0
    measurements["G"] = pd.to_numeric(raw["GLUCOSE"], errors="coerce").to_numpy(dtype=float)
    measurements["F"] = pd.to_numeric(raw["FRUCTOSE"], errors="coerce").to_numpy(dtype=float)
    measurements["E"] = pd.to_numeric(raw["ETANOL"], errors="coerce").to_numpy(dtype=float)
    measurements = apply_measurement_preprocessing(measurements, time, nutrient_pulse_kg_m3)

    initial_guess = measurements.interpolate(limit_direction="both")
    initials = initial_guess.iloc[0].to_dict()
    if any(pd.isna(v) for v in initials.values()):
        missing = [k for k, v in initials.items() if pd.isna(v)]
        raise ValueError(f"Batch {batch_id} has no usable initial value for {missing}.")

    ids = raw["ID"].dropna() if "ID" in raw else pd.Series(dtype=object)
    run_label = str(ids.iloc[0]) if len(ids) else str(batch_id)

    return FermentationBatch(
        batch_id=str(batch_id),
        run_label=run_label,
        raw=raw,
        time=time,
        temperature_c=temperature_c,
        nutrient_pulse_kg_m3=nutrient_pulse_kg_m3,
        nutrient_pulse_rate_kg_m3_h=nutrient_pulse_rate_kg_m3_h,
        measurements=measurements,
        initial_guess=initial_guess,
        initials=initials,
    )


BATCH_ID = available_batches[0]
batch = load_batch(BATCH_ID)
print(f"Selected batch: {batch.batch_id} | {batch.run_label}")
print("Initial conditions used by the model:")
display(pd.Series(batch.initials, name="initial value"))
display(batch.raw.head())

In [ ]:
def plot_batch_data(batch: FermentationBatch):
    fig, ax = plt.subplots(4, 1, figsize=(10, 9), sharex=True)

    ax[0].plot(batch.time, batch.temperature_c, color="tab:red")
    ax[0].set_ylabel("Temp. (deg C)")
    ax[0].grid(True, alpha=0.3)

    ax[1].bar(batch.time, batch.nutrient_pulse_kg_m3, width=3.5, color="tab:cyan", alpha=0.75, label="YAN pulse")
    ax[1].set_ylabel("kg/m3")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)

    ax[2].scatter(batch.measurements.index, batch.measurements["G"], label="Glucose", color="tab:blue")
    ax[2].scatter(batch.measurements.index, batch.measurements["F"], label="Fructose", color="tab:orange")
    ax[2].scatter(batch.measurements.index, batch.measurements["E"], label="Ethanol", color="tab:green")
    ax[2].set_ylabel("kg/m3")
    ax[2].legend(ncol=3)
    ax[2].grid(True, alpha=0.3)

    ax[3].scatter(batch.measurements.index, batch.measurements["X"], label="Biomass", color="tab:purple")
    ax2 = ax[3].twinx()
    ax2.scatter(batch.measurements.index, batch.measurements["N"], label="YAN", color="tab:brown", marker="x")
    ax[3].set_ylabel("X (kg/m3)")
    ax2.set_ylabel("N (kg/m3)")
    ax[3].set_xlabel("Time (h)")
    ax[3].grid(True, alpha=0.3)

    handles, labels = ax[3].get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax[3].legend(handles + handles2, labels + labels2, loc="best")
    fig.suptitle(f"Experimental data - batch {batch.batch_id}")
    fig.tight_layout()
    return fig, ax

plot_batch_data(batch);

## Effective parameters and physical back-transformation

The original Zenteno/MATLAB parameters are kept as a physical reference, but the fitted vector is now

\[
\phi = \{\mu_0, s_N, q_N, q_{XG}, q_{XF}, β_{G0}, s_G, β_{F0}, s_F, q_{EG}, q_{EF}, i_G, i_E, K_{d0}, m_0\}.
\]

The main transformations are

\[
s_N = \mu_0/K_{n0}, \quad q_N = \mu_0/Y_{xn}, \quad q_{XG}=\mu_0/Y_{xg}, \quad q_{XF}=\mu_0/Y_{xf},
\]

\[
s_G = β_{G0}/K_{g0}, \quad s_F = β_{F0}/K_{f0}, \quad q_{EG}=β_{G0}/Y_{eg}, \quad q_{EF}=β_{F0}/Y_{ef},
\]

\[
i_G = 1/K_{ig0}, \quad i_E = 1/K_{ie0}.
\]

`Kig0` remains batch-dependent because the original reference uses initial total sugar divided by four. The helper `_theta_for_batch()` therefore returns a numeric effective vector for each batch. It also accepts old-style physical dictionaries and converts them to the effective vector before model construction.


In [ ]:
DEFAULT_PHYSICAL_THETA = {
    "mu0": 0.18,       # 1/h, l in Zenteno et al. Table 2
    "betaG0": 0.225,   # kg E/kg Bio/h
    "betaF0": 0.225,   # kg E/kg Bio/h
    "Kn0": 0.01,       # kg N/m3
    "Kg0": 7.5,        # kg G/m3
    "Kf0": 7.5,        # kg F/m3
    "Kig0": None,      # kg G/m3; default = initial total sugar / 4
    "Kie0": 40.0,      # kg E/m3
    "Kd0": 0.00044,    # 1/h
    "Yxn": 19.69,      # kg Bio/kg N
    "Yxg": 1.60,       # kg Bio/kg G
    "Yxf": 1.60,       # kg Bio/kg F
    "Yeg": 0.49,       # kg E/kg G
    "Yef": 0.49,       # kg E/kg F
    "m0": 0.01,        # kg S/kg Bio/h, maintenance coefficient
}

PHYSICAL_PARAMETER_BOUNDS = {
    "mu0": (5e-2, 1e0),
    "betaG0": (1e-1, 1e1),
    "betaF0": (1e-1, 1e1),
    "Kn0": (1e-2, 1e-1),
    "Kg0": (1e-2, 1e1),
    "Kf0": (1e-2, 1e1),
    "Kig0": (1e1, 1e3),
    "Kie0": (1e1, 1e3),
    "Kd0": (1e-4, 5e-2),
    "Yxn": (1e0, 1e2),
    "Yxg": (1e-1, 1e1),
    "Yxf": (1e-1, 1e1),
    "Yeg": (1e-1, 1e1),
    "Yef": (1e-1, 1e1),
    "m0": (1e-4, 5e-2),
}

# Effective parameters used by estimation/diagnostics.
# iG is batch-specific because Kig0 defaults to initial total sugar / 4.
DEFAULT_THETA = {
    "mu0": DEFAULT_PHYSICAL_THETA["mu0"],
    "sN": DEFAULT_PHYSICAL_THETA["mu0"] / DEFAULT_PHYSICAL_THETA["Kn0"],
    "qN": DEFAULT_PHYSICAL_THETA["mu0"] / DEFAULT_PHYSICAL_THETA["Yxn"],
    "qXG": DEFAULT_PHYSICAL_THETA["mu0"] / DEFAULT_PHYSICAL_THETA["Yxg"],
    "qXF": DEFAULT_PHYSICAL_THETA["mu0"] / DEFAULT_PHYSICAL_THETA["Yxf"],
    "betaG0": DEFAULT_PHYSICAL_THETA["betaG0"],
    "sG": DEFAULT_PHYSICAL_THETA["betaG0"] / DEFAULT_PHYSICAL_THETA["Kg0"],
    "betaF0": DEFAULT_PHYSICAL_THETA["betaF0"],
    "sF": DEFAULT_PHYSICAL_THETA["betaF0"] / DEFAULT_PHYSICAL_THETA["Kf0"],
    "qEG": DEFAULT_PHYSICAL_THETA["betaG0"] / DEFAULT_PHYSICAL_THETA["Yeg"],
    "qEF": DEFAULT_PHYSICAL_THETA["betaF0"] / DEFAULT_PHYSICAL_THETA["Yef"],
    "iG": None,
    "iE": 1.0 / DEFAULT_PHYSICAL_THETA["Kie0"],
    "Kd0": DEFAULT_PHYSICAL_THETA["Kd0"],
    "m0": DEFAULT_PHYSICAL_THETA["m0"],
}
DEFAULT_PHI = DEFAULT_THETA  # backwards-compatible alias for the effective vector

# Bounds are mapped from the original physical bounds. Some are deliberately broad;
# tighten them after inspecting Q/FIM and profile-likelihood diagnostics.
PARAMETER_BOUNDS = {
    "mu0": PHYSICAL_PARAMETER_BOUNDS["mu0"],
    "sN": (
        PHYSICAL_PARAMETER_BOUNDS["mu0"][0] / PHYSICAL_PARAMETER_BOUNDS["Kn0"][1],
        PHYSICAL_PARAMETER_BOUNDS["mu0"][1] / PHYSICAL_PARAMETER_BOUNDS["Kn0"][0],
    ),
    "qN": (
        PHYSICAL_PARAMETER_BOUNDS["mu0"][0] / PHYSICAL_PARAMETER_BOUNDS["Yxn"][1],
        PHYSICAL_PARAMETER_BOUNDS["mu0"][1] / PHYSICAL_PARAMETER_BOUNDS["Yxn"][0],
    ),
    "qXG": (
        PHYSICAL_PARAMETER_BOUNDS["mu0"][0] / PHYSICAL_PARAMETER_BOUNDS["Yxg"][1],
        PHYSICAL_PARAMETER_BOUNDS["mu0"][1] / PHYSICAL_PARAMETER_BOUNDS["Yxg"][0],
    ),
    "qXF": (
        PHYSICAL_PARAMETER_BOUNDS["mu0"][0] / PHYSICAL_PARAMETER_BOUNDS["Yxf"][1],
        PHYSICAL_PARAMETER_BOUNDS["mu0"][1] / PHYSICAL_PARAMETER_BOUNDS["Yxf"][0],
    ),
    "betaG0": PHYSICAL_PARAMETER_BOUNDS["betaG0"],
    "sG": (
        PHYSICAL_PARAMETER_BOUNDS["betaG0"][0] / PHYSICAL_PARAMETER_BOUNDS["Kg0"][1],
        PHYSICAL_PARAMETER_BOUNDS["betaG0"][1] / PHYSICAL_PARAMETER_BOUNDS["Kg0"][0],
    ),
    "betaF0": PHYSICAL_PARAMETER_BOUNDS["betaF0"],
    "sF": (
        PHYSICAL_PARAMETER_BOUNDS["betaF0"][0] / PHYSICAL_PARAMETER_BOUNDS["Kf0"][1],
        PHYSICAL_PARAMETER_BOUNDS["betaF0"][1] / PHYSICAL_PARAMETER_BOUNDS["Kf0"][0],
    ),
    "qEG": (
        PHYSICAL_PARAMETER_BOUNDS["betaG0"][0] / PHYSICAL_PARAMETER_BOUNDS["Yeg"][1],
        PHYSICAL_PARAMETER_BOUNDS["betaG0"][1] / PHYSICAL_PARAMETER_BOUNDS["Yeg"][0],
    ),
    "qEF": (
        PHYSICAL_PARAMETER_BOUNDS["betaF0"][0] / PHYSICAL_PARAMETER_BOUNDS["Yef"][1],
        PHYSICAL_PARAMETER_BOUNDS["betaF0"][1] / PHYSICAL_PARAMETER_BOUNDS["Yef"][0],
    ),
    "iG": (1.0 / PHYSICAL_PARAMETER_BOUNDS["Kig0"][1], 1.0 / PHYSICAL_PARAMETER_BOUNDS["Kig0"][0]),
    "iE": (1.0 / PHYSICAL_PARAMETER_BOUNDS["Kie0"][1], 1.0 / PHYSICAL_PARAMETER_BOUNDS["Kie0"][0]),
    "Kd0": PHYSICAL_PARAMETER_BOUNDS["Kd0"],
    "m0": PHYSICAL_PARAMETER_BOUNDS["m0"],
}

PHYSICAL_PARAMETER_NAMES = set(DEFAULT_PHYSICAL_THETA)
EFFECTIVE_PARAMETER_NAMES = set(DEFAULT_THETA)


def _physical_theta_for_batch(batch: FermentationBatch, theta_initial=None) -> dict:
    theta = DEFAULT_PHYSICAL_THETA.copy()
    if theta_initial is not None:
        theta.update({name: value for name, value in dict(theta_initial).items() if name in PHYSICAL_PARAMETER_NAMES})
    if theta["Kig0"] is None:
        theta["Kig0"] = (batch.initials["G"] + batch.initials["F"]) / 4.0
    return {name: float(value) for name, value in theta.items()}


def effective_theta_from_physical(theta_physical: dict) -> dict:
    theta = {name: float(value) for name, value in dict(theta_physical).items()}
    return {
        "mu0": theta["mu0"],
        "sN": theta["mu0"] / theta["Kn0"],
        "qN": theta["mu0"] / theta["Yxn"],
        "qXG": theta["mu0"] / theta["Yxg"],
        "qXF": theta["mu0"] / theta["Yxf"],
        "betaG0": theta["betaG0"],
        "sG": theta["betaG0"] / theta["Kg0"],
        "betaF0": theta["betaF0"],
        "sF": theta["betaF0"] / theta["Kf0"],
        "qEG": theta["betaG0"] / theta["Yeg"],
        "qEF": theta["betaF0"] / theta["Yef"],
        "iG": 1.0 / theta["Kig0"],
        "iE": 1.0 / theta["Kie0"],
        "Kd0": theta["Kd0"],
        "m0": theta["m0"],
    }


def _theta_for_batch(batch: FermentationBatch, theta_initial=None) -> dict:
    """Return the numeric effective parameter vector for a batch.

    `theta_initial` may contain effective parameters, original physical parameters,
    or a mix. Physical entries are converted first; effective entries then override
    the converted values.
    """
    physical_theta = _physical_theta_for_batch(batch)
    theta = effective_theta_from_physical(physical_theta)
    if theta_initial is None:
        return theta

    incoming = dict(theta_initial)
    unknown_names = sorted(set(incoming) - PHYSICAL_PARAMETER_NAMES - EFFECTIVE_PARAMETER_NAMES)
    if unknown_names:
        raise ValueError(f"Unknown parameter names in theta_initial: {unknown_names}")

    physical_updates = {
        name: value
        for name, value in incoming.items()
        if name in PHYSICAL_PARAMETER_NAMES and value is not None
    }
    if physical_updates:
        updated_physical = dict(physical_theta)
        updated_physical.update(physical_updates)
        theta.update(effective_theta_from_physical(updated_physical))

    effective_updates = {
        name: float(value)
        for name, value in incoming.items()
        if name in EFFECTIVE_PARAMETER_NAMES and value is not None
    }
    theta.update(effective_updates)
    return theta


def physical_theta_from_effective(theta_effective: dict, batch: FermentationBatch | None = None) -> dict:
    """Back-transform an effective parameter vector to the original physical names."""
    if batch is not None:
        theta = _theta_for_batch(batch, theta_effective)
    else:
        theta = dict(DEFAULT_THETA)
        theta.update(dict(theta_effective))
        missing = [name for name, value in theta.items() if value is None]
        if missing:
            raise ValueError(f"Provide a batch or explicit values for batch-dependent parameters: {missing}")
        theta = {name: float(value) for name, value in theta.items()}

    return {
        "mu0": theta["mu0"],
        "betaG0": theta["betaG0"],
        "betaF0": theta["betaF0"],
        "Kn0": theta["mu0"] / theta["sN"],
        "Kg0": theta["betaG0"] / theta["sG"],
        "Kf0": theta["betaF0"] / theta["sF"],
        "Kig0": 1.0 / theta["iG"],
        "Kie0": 1.0 / theta["iE"],
        "Kd0": theta["Kd0"],
        "Yxn": theta["mu0"] / theta["qN"],
        "Yxg": theta["mu0"] / theta["qXG"],
        "Yxf": theta["mu0"] / theta["qXF"],
        "Yeg": theta["betaG0"] / theta["qEG"],
        "Yef": theta["betaF0"] / theta["qEF"],
        "m0": theta["m0"],
    }


def effective_parameter_table(batch: FermentationBatch, theta_initial=None) -> pd.DataFrame:
    theta = _theta_for_batch(batch, theta_initial)
    rows = []
    for name in DEFAULT_THETA:
        lb, ub = PARAMETER_BOUNDS[name]
        rows.append(
            {
                "parameter": name,
                "nominal_or_current": theta[name],
                "lower_bound": lb,
                "upper_bound": ub,
            }
        )
    return pd.DataFrame(rows).set_index("parameter")


def physical_parameter_table(batch: FermentationBatch, theta_effective=None) -> pd.DataFrame:
    theta = physical_theta_from_effective(theta_effective or {}, batch=batch)
    rows = []
    for name in DEFAULT_PHYSICAL_THETA:
        lb, ub = PHYSICAL_PARAMETER_BOUNDS[name]
        rows.append(
            {
                "parameter": name,
                "back_transformed_value": theta[name],
                "physical_lower_bound": lb,
                "physical_upper_bound": ub,
            }
        )
    return pd.DataFrame(rows).set_index("parameter")


def clip_theta_to_bounds(theta, atol: float = 1e-8):
    clipped = {}
    adjustments = []
    for name, value in theta.items():
        if value is None:
            continue
        value = float(value)
        lb, ub = PARAMETER_BOUNDS[name]
        clipped_value = min(max(value, lb), ub)
        clipped[name] = clipped_value
        if abs(clipped_value - value) > atol:
            adjustments.append(
                {
                    "parameter": name,
                    "raw_value": value,
                    "clipped_value": clipped_value,
                    "lower_bound": lb,
                    "upper_bound": ub,
                }
            )
    return clipped, pd.DataFrame(adjustments)

STATE_BOUNDS = {
    "X": (0.0, 20.0),
    # Small negative lower bounds avoid false infeasibility from numerical tolerances at substrate depletion.
    "N": (-1e-6, 5.0),
    "G": (-1e-6, 300.0),
    "F": (-1e-6, 300.0),
    "E": (0.0, 200.0),
}

# Replicate-based measurement-error estimates define the Gaussian objective weights.
# Biomass keeps a conservative floor so X does not dominate solely through a tiny sigma.
MEASUREMENT_ERROR = {
    "X": 0.05,
    "N": 0.01,
    "G": 2.0,
    "F": 2.0,
    "E": 1.0,
}

FIXED_CONSTANTS = {
    "Cde": 0.0415,     # m3/kg E
    "Etd": 130000.0,   # kJ/kmol
    "R": 8.314,        # kJ/kmol/K
    "Eac": 59453.0,    # kJ/kmol
    "Eafe": 11000.0,   # kJ/kmol
    "EaKn": 46055.0,   # kJ/kmol
    "EaKg": 46055.0,   # kJ/kmol
    "EaKf": 46055.0,   # kJ/kmol
    "EaKig": 46055.0,  # kJ/kmol
    "EaKie": 46055.0,  # kJ/kmol
    "Eam": 37681.0,    # kJ/kmol
}

REPLICATE_BATCH_PAIRS = [("25085", "25086"), ("25150", "25151")]

DESIGN_TEMPERATURE_BOUNDS = (15.0, 25.0)  # deg C
DESIGN_TEMPERATURE_SEGMENTS = 4
DESIGN_MAX_PULSES = 5
DESIGN_MAX_YAN_PER_PULSE = 100.0 * MG_L_TO_KG_M3
DESIGN_MAX_TOTAL_YAN = 200.0 * MG_L_TO_KG_M3
DESIGN_PULSE_ALLOWED_FRACTION = 2 / 3
DESIGN_PULSE_WIDTH_H = 3.0


def estimate_measurement_error_from_replicates(batch_pairs=REPLICATE_BATCH_PAIRS):
    rows = []
    errors = {}
    for state in STATE_LABELS:
        diffs = []
        for batch_a, batch_b in batch_pairs:
            data_a = load_batch(batch_a).measurements[[state]]
            data_b = load_batch(batch_b).measurements[[state]]
            joined = data_a.join(data_b, how="inner", lsuffix="_a", rsuffix="_b").dropna()
            diffs.extend((joined[f"{state}_a"] - joined[f"{state}_b"]).to_list())
        diffs = np.asarray(diffs, dtype=float)
        if len(diffs) == 0:
            errors[state] = MEASUREMENT_ERROR[state]
            rows.append({"state": state, "n_pairs": 0, "measurement_error": errors[state]})
            continue
        errors[state] = float(np.sqrt(np.mean(diffs**2)) / np.sqrt(2))
        rows.append({"state": state, "n_pairs": len(diffs), "measurement_error": errors[state]})
    return errors, pd.DataFrame(rows).set_index("state")


REPLICATE_MEASUREMENT_ERROR, replicate_error_summary = estimate_measurement_error_from_replicates()
MEASUREMENT_ERROR.update(REPLICATE_MEASUREMENT_ERROR)
MEASUREMENT_ERROR["X"] = max(
    BIOMASS_MEASUREMENT_ERROR_MIN_KG_M3,
    BIOMASS_MEASUREMENT_ERROR_MULTIPLIER * float(MEASUREMENT_ERROR["X"]),
)
measurement_error_summary = replicate_error_summary.copy()
measurement_error_summary["replicate_error_diagnostic"] = pd.Series(MEASUREMENT_ERROR)
display(measurement_error_summary)

# Gaussian objective: ParmEst uses this suffix as sigma in
# 0.5 * sum(((model - data) / measurement_error)**2).
OBJECTIVE_SCALE_MODE = "measurement_error"  # Options: "measurement_error", "max_abs", "variance", "std", "range".
OBJECTIVE_SCALE_SCOPE = "state"  # Options: "batch_state", "state". Ignored for measurement_error mode.
OBJECTIVE_NORMALIZE_BY_STATE_COUNT = False  # Keep the Gaussian objective as a raw sum over observations.
OBJECTIVE_NORMALIZE_BY_EXPERIMENT_STATE_COUNT = False  # Keep independent observations, not equal experiment/state blocks.
OBJECTIVE_SCALE_FLOOR = 1e-8


def _objective_batch_ids(batch_ids=None):
    if batch_ids is None:
        return list(available_batches)
    if isinstance(batch_ids, (str, int)):
        return [str(batch_ids)]
    return [str(batch_id) for batch_id in batch_ids]


def objective_scale_summary_for_batches(
    batch_ids=None,
    states=None,
    mode=None,
    scope=None,
    normalize_by_state_count=None,
    normalize_by_experiment_state_count=None,
):
    mode = OBJECTIVE_SCALE_MODE if mode is None else mode
    scope = OBJECTIVE_SCALE_SCOPE if scope is None else scope
    if normalize_by_state_count is None:
        normalize_by_state_count = OBJECTIVE_NORMALIZE_BY_STATE_COUNT
    if normalize_by_experiment_state_count is None:
        normalize_by_experiment_state_count = OBJECTIVE_NORMALIZE_BY_EXPERIMENT_STATE_COUNT
    batch_ids = _objective_batch_ids(batch_ids)
    states = list(STATE_LABELS) if states is None else list(states)
    rows = []

    if mode in {"measurement_error", "gaussian"}:
        normalize_by_state_count = False
        normalize_by_experiment_state_count = False

    def _scale_from_values(values, state):
        values = np.asarray(values, dtype=float)
        n_obs = int(values.size)
        if mode in {"measurement_error", "gaussian"}:
            state_scale = float(MEASUREMENT_ERROR[state])
        elif n_obs == 0:
            state_scale = 1.0
        elif mode == "max_abs":
            state_scale = float(np.max(np.abs(values)))
        elif mode in {"variance", "std"}:
            state_scale = float(np.sqrt(np.var(values)))
        elif mode == "range":
            state_scale = float(np.max(values) - np.min(values))
        else:
            raise ValueError(f"Unknown OBJECTIVE_SCALE_MODE: {mode!r}")
        state_scale = max(state_scale, float(OBJECTIVE_SCALE_FLOOR))
        return n_obs, state_scale

    if scope not in {"batch_state", "state"}:
        raise ValueError(f"Unknown OBJECTIVE_SCALE_SCOPE: {scope!r}")

    if scope == "batch_state":
        observed_state_counts = {}
        values_by_key = {}
        for batch_id in batch_ids:
            observed_state_counts[str(batch_id)] = 0
            for state in states:
                series = load_batch(batch_id).measurements[state].dropna().astype(float)
                values = series.to_numpy(dtype=float)
                values_by_key[(str(batch_id), state)] = values
                observed_state_counts[str(batch_id)] += int(values.size > 0)

        for batch_id in batch_ids:
            batch_key = str(batch_id)
            state_block_count = max(int(observed_state_counts[batch_key]), 1)
            state_block_factor = (
                float(np.sqrt(state_block_count))
                if normalize_by_experiment_state_count
                else 1.0
            )
            for state in states:
                n_obs, state_scale = _scale_from_values(values_by_key[(batch_key, state)], state)
                count_factor = float(np.sqrt(n_obs)) if normalize_by_state_count and n_obs > 0 else 1.0
                objective_scale = state_scale * count_factor * state_block_factor
                rows.append(
                    {
                        "batch": batch_key,
                        "state": state,
                        "n_observations": n_obs,
                        "state_scale": state_scale,
                        "count_factor": count_factor,
                        "state_block_count": state_block_count,
                        "state_block_factor": state_block_factor,
                        "objective_scale": objective_scale,
                        "scale_mode": mode,
                        "scale_scope": scope,
                        "normalized_by_state_count": bool(normalize_by_state_count),
                        "normalized_by_experiment_state_count": bool(normalize_by_experiment_state_count),
                    }
                )
        return pd.DataFrame(rows).set_index(["batch", "state"])

    observed_state_count = 0
    values_by_state = {}
    for state in states:
        values = []
        for batch_id in batch_ids:
            series = load_batch(batch_id).measurements[state].dropna().astype(float)
            values.extend(series.to_list())
        values = np.asarray(values, dtype=float)
        values_by_state[state] = values
        observed_state_count += int(values.size > 0)
    observed_state_count = max(int(observed_state_count), 1)
    state_block_factor = float(np.sqrt(observed_state_count)) if normalize_by_experiment_state_count else 1.0
    for state in states:
        n_obs, state_scale = _scale_from_values(values_by_state[state], state)
        count_factor = float(np.sqrt(n_obs)) if normalize_by_state_count and n_obs > 0 else 1.0
        objective_scale = state_scale * count_factor * state_block_factor
        rows.append(
            {
                "state": state,
                "n_observations": n_obs,
                "state_scale": state_scale,
                "count_factor": count_factor,
                "state_block_count": observed_state_count,
                "state_block_factor": state_block_factor,
                "objective_scale": objective_scale,
                "scale_mode": mode,
                "scale_scope": scope,
                "normalized_by_state_count": bool(normalize_by_state_count),
                "normalized_by_experiment_state_count": bool(normalize_by_experiment_state_count),
            }
        )
    return pd.DataFrame(rows).set_index("state")


def objective_scale_records_for_batches(batch_ids=None, states=None):
    batch_ids = _objective_batch_ids(batch_ids)
    states = list(STATE_LABELS) if states is None else list(states)
    summary = objective_scale_summary_for_batches(batch_ids=batch_ids, states=states)
    records = {}
    if isinstance(summary.index, pd.MultiIndex):
        for (batch_id, state), row in summary.iterrows():
            records[(str(batch_id), state)] = row.to_dict()
    else:
        for batch_id in batch_ids:
            for state, row in summary.iterrows():
                records[(str(batch_id), state)] = row.to_dict()
    return records


def objective_scale_lookup_for_batches(batch_ids=None, states=None):
    records = objective_scale_records_for_batches(batch_ids=batch_ids, states=states)
    return {key: float(record["objective_scale"]) for key, record in records.items()}


def objective_measurement_error_for_batches(batch_ids=None, states=None):
    batch_ids = _objective_batch_ids(batch_ids)
    if len(batch_ids) != 1:
        raise ValueError("Use objective_scale_lookup_for_batches for multiple experiments.")
    states = list(STATE_LABELS) if states is None else list(states)
    records = objective_scale_records_for_batches(batch_ids=batch_ids, states=states)
    batch_key = str(batch_ids[0])
    return {state: float(records[(batch_key, state)]["objective_scale"]) for state in states}


objective_scale_summary = objective_scale_summary_for_batches()
display(objective_scale_summary)

print("Effective parameter vector for selected batch:")
display(effective_parameter_table(batch))
print("Back-transformed physical parameters for selected batch:")
display(physical_parameter_table(batch))


## Pyomo model

This is the same kinetic mass-balance structure as the MATLAB/Zenteno model, but the equations are written in effective parameter form. Physical saturation/inhibition/yield parameters are kept as Pyomo expressions for interpretation, while the variables labeled as unknown parameters are the effective parameters in `DEFAULT_THETA`.

The thermal-death switch is still implemented with `pyo.Expr_if`, because the threshold depends on ethanol, which is a state variable.


In [ ]:
def _time_key(t, ndigits: int = 10):
    return round(float(t), ndigits)


def _simulation_time_grid(data_time, max_step_h=SIMULATION_INTERNAL_STEP_H):
    data_time = np.asarray(data_time, dtype=float)
    if max_step_h is None or float(max_step_h) <= 0.0:
        return np.array(sorted({_time_key(t) for t in data_time}), dtype=float)
    points = [float(data_time[0])]
    for left, right in zip(data_time[:-1], data_time[1:]):
        left = float(left)
        right = float(right)
        n_sub = max(1, int(np.ceil((right - left) / float(max_step_h))))
        segment = np.linspace(left, right, n_sub + 1)[1:]
        points.extend(float(_time_key(t)) for t in segment)
    return np.array(sorted({_time_key(t) for t in points}), dtype=float)


def _interp_to_model_time(data_time, data_values, model_time):
    return np.interp(
        np.asarray(model_time, dtype=float),
        np.asarray(data_time, dtype=float),
        np.asarray(data_values, dtype=float),
    )


def _map_data_values_to_model_time(data_time, data_values, model_time, default=0.0):
    values = {float(t): float(default) for t in model_time}
    lookup = {_time_key(t): float(t) for t in model_time}
    for t, value in zip(data_time, data_values):
        key = _time_key(t)
        if key in lookup:
            values[lookup[key]] = float(value)
    return values


def _measured_pulse_rate_profile(model_time, dt_by_time, data_time, pulse_amounts, width_h=MEASURED_PULSE_WIDTH_H):
    rate = {float(t): 0.0 for t in model_time}
    if not USE_SMOOTH_MEASURED_PULSES:
        pulse_init = _map_data_values_to_model_time(data_time, pulse_amounts, model_time, default=0.0)
        t0 = float(model_time[0])
        for t, amount in pulse_init.items():
            rate[t] = 0.0 if float(t) == t0 else float(amount) / float(dt_by_time[t])
        return rate

    t0 = float(model_time[0])
    width_h = max(float(width_h), 1e-6)
    for event_time, amount in zip(data_time, pulse_amounts):
        event_time = float(event_time)
        amount = float(amount)
        if amount <= 0.0 or event_time <= t0:
            continue
        weights = {}
        for t in model_time:
            t = float(t)
            if t <= t0:
                continue
            if MEASURED_PULSE_AFTER_SAMPLE:
                if t <= event_time:
                    continue
                lag = t - event_time
            else:
                if t < event_time:
                    continue
                lag = max(0.0, t - event_time)
            weights[t] = float(np.exp(-((lag / width_h) ** 2)))
        norm = sum(float(dt_by_time[t]) * weight for t, weight in weights.items())
        if norm <= 0.0:
            continue
        for t, weight in weights.items():
            rate[t] += amount * weight / norm
    return rate


def _state_initialization(batch: FermentationBatch, state: str, model_time=None, dynamic_guess=None) -> dict:
    if model_time is None:
        model_time = batch.time
    model_time = np.asarray(model_time, dtype=float)
    if dynamic_guess is not None and state in dynamic_guess.columns:
        values = _interp_to_model_time(dynamic_guess.index.to_numpy(dtype=float), dynamic_guess[state].to_numpy(dtype=float), model_time)
    else:
        values = _interp_to_model_time(batch.initial_guess.index.to_numpy(dtype=float), batch.initial_guess[state].to_numpy(dtype=float), model_time)
    lb, ub = STATE_BOUNDS[state]
    floor = max(float(lb) + 1e-8, 1e-8)
    return {
        float(t): min(max(floor, float(value)), float(ub))
        for t, value in zip(model_time, values)
    }


def _dynamic_initialization(batch: FermentationBatch, theta, model_time, temp_init, pulse_rate_init):
    if not USE_DYNAMIC_INITIALIZATION or solve_ivp is None:
        return None

    constants = FIXED_CONSTANTS
    model_time = np.asarray(model_time, dtype=float)
    temp_time = np.array(sorted(temp_init), dtype=float)
    temp_values = np.array([float(temp_init[t]) for t in temp_time], dtype=float)
    pulse_time = np.array(sorted(pulse_rate_init), dtype=float)
    pulse_values = np.array([float(pulse_rate_init[t]) for t in pulse_time], dtype=float)
    theta = {name: float(value) for name, value in theta.items()}
    y0 = np.array([float(batch.initials[state]) for state in STATE_LABELS], dtype=float)

    def rhs(t, y):
        X, N, G, F, E = [max(0.0, float(v)) for v in y]
        T_K = float(np.interp(t, temp_time, temp_values)) + 273.15
        pulse_rate = float(np.interp(t, pulse_time, pulse_values))
        R = constants["R"]
        eps = 1e-8

        A_mu = np.exp(constants["Eac"] * (T_K - 300.0) / (300.0 * R * T_K))
        A_beta = np.exp(constants["Eafe"] * (T_K - 296.15) / (296.15 * R * T_K))
        A_Kn = np.exp(constants["EaKn"] * (T_K - 293.15) / (293.15 * R * T_K))
        A_Kg = np.exp(constants["EaKg"] * (T_K - 293.15) / (293.15 * R * T_K))
        A_Kf = np.exp(constants["EaKf"] * (T_K - 293.15) / (293.15 * R * T_K))
        A_Kig = np.exp(constants["EaKig"] * (T_K - 293.15) / (293.15 * R * T_K))
        A_Kie = np.exp(constants["EaKie"] * (T_K - 293.15) / (293.15 * R * T_K))

        Kn = (theta["mu0"] / theta["sN"]) * A_Kn
        Kg = (theta["betaG0"] / theta["sG"]) * A_Kg
        Kf = (theta["betaF0"] / theta["sF"]) * A_Kf
        iG_T = theta["iG"] / (A_Kig + eps)
        iE_T = theta["iE"] / (A_Kie + eps)

        N_lim = N / (N + Kn + eps)
        G_lim = G / (G + Kg + eps)
        F_lim = F / (F + Kf + eps)
        E_inhibition = 1.0 / (1.0 + iE_T * E)
        G_inhibition_for_fructose = 1.0 / (1.0 + iG_T * G)

        growth_factor = A_mu * N_lim
        glucose_ferm_factor = A_beta * G_lim * E_inhibition
        fructose_ferm_factor = A_beta * F_lim * G_inhibition_for_fructose * E_inhibition

        mu = theta["mu0"] * growth_factor
        beta_G = theta["betaG0"] * glucose_ferm_factor
        beta_F = theta["betaF0"] * fructose_ferm_factor
        maintenance = theta["m0"] * np.exp(constants["Eam"] * (T_K - 293.3) / (293.3 * R * T_K))

        Td = -0.0001 * E**3 + 0.0049 * E**2 - 0.1279 * E + 315.89
        Kd = theta["Kd0"] * np.exp((constants["Cde"] * E) + (constants["Etd"] * (T_K - 305.65)) / (305.65 * R * T_K)) if T_K >= Td else 0.0
        sugar_total = G + F + eps
        maintenance_availability = sugar_total / (sugar_total + MAINTENANCE_SUGAR_CUTOFF_KG_M3)
        return [
            (mu - Kd) * X,
            -theta["qN"] * growth_factor * X + pulse_rate,
            -(theta["qXG"] * growth_factor + theta["qEG"] * glucose_ferm_factor + maintenance * maintenance_availability * (G / sugar_total)) * X,
            -(theta["qXF"] * growth_factor + theta["qEF"] * fructose_ferm_factor + maintenance * maintenance_availability * (F / sugar_total)) * X,
            (beta_G + beta_F) * X,
        ]

    try:
        sol = solve_ivp(
            rhs,
            (float(model_time[0]), float(model_time[-1])),
            y0,
            t_eval=model_time,
            method="LSODA",
            rtol=1e-6,
            atol=1e-8,
        )
    except Exception:
        return None
    if not sol.success or sol.y.shape[1] != len(model_time):
        return None

    values = np.asarray(sol.y.T, dtype=float)
    for col_idx, state in enumerate(STATE_LABELS):
        lb, ub = STATE_BOUNDS[state]
        floor = max(float(lb) + 1e-8, 1e-8)
        values[:, col_idx] = np.clip(values[:, col_idx], floor, float(ub))
    return pd.DataFrame(values, index=model_time, columns=list(STATE_LABELS))


def build_zenteno_pyomo_model(
    batch: FermentationBatch,
    theta_initial=None,
    fix_parameters: bool = True,
    label_model: bool = True,
    input_mode: str = "measured",
    temperature_segments: int = DESIGN_TEMPERATURE_SEGMENTS,
    pulse_max_count: int = DESIGN_MAX_PULSES,
    pulse_width_h: float = DESIGN_PULSE_WIDTH_H,
    fix_design_inputs: bool = False,
):
    theta = _theta_for_batch(batch, theta_initial)
    temperature_segments = int(temperature_segments)
    pulse_max_count = int(pulse_max_count)
    if input_mode not in {"measured", "design"}:
        raise ValueError("input_mode must be 'measured' or 'design'.")
    if temperature_segments < 1:
        raise ValueError("temperature_segments must be at least 1.")
    if not 1 <= pulse_max_count <= DESIGN_MAX_PULSES:
        raise ValueError(f"pulse_max_count must be between 1 and {DESIGN_MAX_PULSES}.")
    if pulse_width_h <= 0:
        raise ValueError("pulse_width_h must be positive.")

    data_time = np.asarray(batch.time, dtype=float)
    model_time = _simulation_time_grid(data_time)

    m = pyo.ConcreteModel(f"Zenteno fermentation effective-parameter model - batch {batch.batch_id}")
    m.t = dae.ContinuousSet(
        initialize=[float(t) for t in model_time],
        bounds=(float(model_time[0]), float(model_time[-1])),
    )

    temp_values = _interp_to_model_time(data_time, batch.temperature_c, model_time)
    temp_init = {float(t): float(temp) for t, temp in zip(model_time, temp_values)}
    m.TempC = pyo.Var(m.t, bounds=(-5.0, 45.0), initialize=temp_init)
    if input_mode == "measured":
        for t, temp in temp_init.items():
            m.TempC[t].fix(temp)
    else:
        m.temperature_segments = pyo.RangeSet(0, temperature_segments - 1)
        t0_data = float(batch.time[0])
        tf_data = float(batch.time[-1])
        segment_edges = np.linspace(t0_data, tf_data, temperature_segments + 1)
        segment_for_time = {}
        segment_init = {}
        for s in range(temperature_segments):
            if s == temperature_segments - 1:
                mask = (batch.time >= segment_edges[s]) & (batch.time <= segment_edges[s + 1])
            else:
                mask = (batch.time >= segment_edges[s]) & (batch.time < segment_edges[s + 1])
            segment_init[s] = float(np.clip(np.mean(batch.temperature_c[mask]), *DESIGN_TEMPERATURE_BOUNDS))
        for t in model_time:
            s = int(np.searchsorted(segment_edges[1:-1], float(t), side="right"))
            segment_for_time[float(t)] = min(s, temperature_segments - 1)
        m.T_set = pyo.Var(
            m.temperature_segments,
            bounds=DESIGN_TEMPERATURE_BOUNDS,
            initialize=lambda m, s: segment_init[int(s)],
        )
        if fix_design_inputs:
            for s in m.temperature_segments:
                m.T_set[s].fix(segment_init[int(s)])

        @m.Constraint(m.t)
        def temperature_setpoint_profile(m, t):
            return m.TempC[t] == m.T_set[segment_for_time[float(t)]]

    pulse_init = _map_data_values_to_model_time(data_time, batch.nutrient_pulse_kg_m3, model_time, default=0.0)
    dt_init = {float(model_time[0]): 1.0}
    for k in range(1, len(model_time)):
        dt_init[float(model_time[k])] = float(model_time[k] - model_time[k - 1])
    pulse_rate_init = _measured_pulse_rate_profile(
        model_time,
        dt_init,
        data_time,
        batch.nutrient_pulse_kg_m3,
    )
    dynamic_initial_guess = _dynamic_initialization(batch, theta, model_time, temp_init, pulse_rate_init)
    m.dt = pyo.Param(m.t, initialize=dt_init)
    if input_mode == "measured":
        m.N_pulse = pyo.Var(m.t, bounds=(0.0, 5.0), initialize=pulse_init)
        for t, pulse in pulse_init.items():
            m.N_pulse[t].fix(pulse)
    else:
        m.pulse_slots = pyo.RangeSet(0, pulse_max_count - 1)
        t0_data = float(batch.time[0])
        tf_data = float(batch.time[-1])
        pulse_time_lb = t0_data
        pulse_time_ub = t0_data + DESIGN_PULSE_ALLOWED_FRACTION * (tf_data - t0_data)
        observed_pulse_times = [float(t) for t, p in pulse_init.items() if p > 0]
        observed_pulse_amounts = [float(p) for p in pulse_init.values() if p > 0]
        fallback_times = np.linspace(pulse_time_lb, pulse_time_ub, pulse_max_count + 2)[1:-1]
        pulse_time_init = {}
        pulse_amount_init = {}
        for p in range(pulse_max_count):
            pulse_time_init[p] = observed_pulse_times[p] if p < len(observed_pulse_times) else float(fallback_times[p])
            pulse_amount_init[p] = observed_pulse_amounts[p] if p < len(observed_pulse_amounts) else 0.0
        m.pulse_time = pyo.Var(
            m.pulse_slots,
            bounds=(pulse_time_lb, pulse_time_ub),
            initialize=lambda m, p: pulse_time_init[int(p)],
        )
        m.pulse_amount = pyo.Var(
            m.pulse_slots,
            bounds=(0.0, DESIGN_MAX_YAN_PER_PULSE),
            initialize=lambda m, p: pulse_amount_init[int(p)],
        )
        if fix_design_inputs:
            for p in m.pulse_slots:
                m.pulse_time[p].fix(pulse_time_init[int(p)])
                m.pulse_amount[p].fix(pulse_amount_init[int(p)])

        @m.Constraint()
        def total_nutrition_limit(m):
            return sum(m.pulse_amount[p] for p in m.pulse_slots) <= DESIGN_MAX_TOTAL_YAN

        @m.Constraint(m.pulse_slots)
        def pulse_order(m, p):
            if int(p) == pulse_max_count - 1:
                return pyo.Constraint.Skip
            return m.pulse_time[p] <= m.pulse_time[int(p) + 1]

    m.X = pyo.Var(m.t, bounds=STATE_BOUNDS["X"], initialize=_state_initialization(batch, "X", model_time, dynamic_initial_guess))
    m.N = pyo.Var(m.t, bounds=STATE_BOUNDS["N"], initialize=_state_initialization(batch, "N", model_time, dynamic_initial_guess))
    m.G = pyo.Var(m.t, bounds=STATE_BOUNDS["G"], initialize=_state_initialization(batch, "G", model_time, dynamic_initial_guess))
    m.F = pyo.Var(m.t, bounds=STATE_BOUNDS["F"], initialize=_state_initialization(batch, "F", model_time, dynamic_initial_guess))
    m.E = pyo.Var(m.t, bounds=STATE_BOUNDS["E"], initialize=_state_initialization(batch, "E", model_time, dynamic_initial_guess))

    m.dX = dae.DerivativeVar(m.X, wrt=m.t)
    m.dN = dae.DerivativeVar(m.N, wrt=m.t)
    m.dG = dae.DerivativeVar(m.G, wrt=m.t)
    m.dF = dae.DerivativeVar(m.F, wrt=m.t)
    m.dE = dae.DerivativeVar(m.E, wrt=m.t)

    for name, value in theta.items():
        var = pyo.Var(initialize=float(value), bounds=PARAMETER_BOUNDS[name])
        setattr(m, name, var)
        if fix_parameters:
            var.fix(float(value))

    for name, value in FIXED_CONSTANTS.items():
        setattr(m, name, pyo.Param(initialize=float(value)))

    eps = 1e-8
    def _nonnegative_effective(var, t):
        return pyo.Expr_if(IF=(var[t] >= 0.0), THEN=var[t], ELSE=0.0)

    m.N_eff = pyo.Expression(m.t, rule=lambda m, t: _nonnegative_effective(m.N, t))
    m.G_eff = pyo.Expression(m.t, rule=lambda m, t: _nonnegative_effective(m.G, t))
    m.F_eff = pyo.Expression(m.t, rule=lambda m, t: _nonnegative_effective(m.F, t))
    m.sugar_total_eff = pyo.Expression(m.t, rule=lambda m, t: m.G_eff[t] + m.F_eff[t])
    m.maintenance_sugar_availability = pyo.Expression(
        m.t,
        rule=lambda m, t: m.sugar_total_eff[t] / (m.sugar_total_eff[t] + MAINTENANCE_SUGAR_CUTOFF_KG_M3),
    )
    t0 = float(model_time[0])
    if input_mode == "measured":
        m.N_pulse_rate = pyo.Param(m.t, initialize=pulse_rate_init, within=pyo.Reals)
    else:
        m.pulse_shape = pyo.Expression(
            m.pulse_slots,
            m.t,
            rule=lambda m, p, t: pyo.exp(-((float(t) - m.pulse_time[p]) / pulse_width_h) ** 2),
        )
        m.pulse_norm = pyo.Expression(
            m.pulse_slots,
            rule=lambda m, p: sum(m.dt[t] * m.pulse_shape[p, t] for t in m.t if float(t) != t0) + eps,
        )
        m.N_pulse_rate = pyo.Expression(
            m.t,
            rule=lambda m, t: 0.0
            if float(t) == t0
            else sum(m.pulse_amount[p] * m.pulse_shape[p, t] / m.pulse_norm[p] for p in m.pulse_slots),
        )
        m.N_pulse = pyo.Expression(m.t, rule=lambda m, t: m.N_pulse_rate[t] * m.dt[t])
    m.T_K = pyo.Expression(m.t, rule=lambda m, t: m.TempC[t] + 273.15)

    # Temperature multipliers. Keeping them explicit makes the effective-rate balances easier to inspect.
    m.A_mu = pyo.Expression(
        m.t, rule=lambda m, t: pyo.exp(m.Eac * (m.T_K[t] - 300.0) / (300.0 * m.R * m.T_K[t]))
    )
    m.A_beta = pyo.Expression(
        m.t, rule=lambda m, t: pyo.exp(m.Eafe * (m.T_K[t] - 296.15) / (296.15 * m.R * m.T_K[t]))
    )
    m.A_Kn = pyo.Expression(
        m.t, rule=lambda m, t: pyo.exp(m.EaKn * (m.T_K[t] - 293.15) / (293.15 * m.R * m.T_K[t]))
    )
    m.A_Kg = pyo.Expression(
        m.t, rule=lambda m, t: pyo.exp(m.EaKg * (m.T_K[t] - 293.15) / (293.15 * m.R * m.T_K[t]))
    )
    m.A_Kf = pyo.Expression(
        m.t, rule=lambda m, t: pyo.exp(m.EaKf * (m.T_K[t] - 293.15) / (293.15 * m.R * m.T_K[t]))
    )
    m.A_Kig = pyo.Expression(
        m.t, rule=lambda m, t: pyo.exp(m.EaKig * (m.T_K[t] - 293.15) / (293.15 * m.R * m.T_K[t]))
    )
    m.A_Kie = pyo.Expression(
        m.t, rule=lambda m, t: pyo.exp(m.EaKie * (m.T_K[t] - 293.15) / (293.15 * m.R * m.T_K[t]))
    )

    # Back-transformed physical quantities for reporting and interpretation.
    # These are expressions, not estimated parameters.
    m.Kn0 = pyo.Expression(rule=lambda m: m.mu0 / m.sN)
    m.Kg0 = pyo.Expression(rule=lambda m: m.betaG0 / m.sG)
    m.Kf0 = pyo.Expression(rule=lambda m: m.betaF0 / m.sF)
    m.Kig0 = pyo.Expression(rule=lambda m: 1.0 / m.iG)
    m.Kie0 = pyo.Expression(rule=lambda m: 1.0 / m.iE)
    m.Yxn = pyo.Expression(rule=lambda m: m.mu0 / m.qN)
    m.Yxg = pyo.Expression(rule=lambda m: m.mu0 / m.qXG)
    m.Yxf = pyo.Expression(rule=lambda m: m.mu0 / m.qXF)
    m.Yeg = pyo.Expression(rule=lambda m: m.betaG0 / m.qEG)
    m.Yef = pyo.Expression(rule=lambda m: m.betaF0 / m.qEF)

    m.mu_max = pyo.Expression(m.t, rule=lambda m, t: m.mu0 * m.A_mu[t])
    m.betaG_max = pyo.Expression(m.t, rule=lambda m, t: m.betaG0 * m.A_beta[t])
    m.betaF_max = pyo.Expression(m.t, rule=lambda m, t: m.betaF0 * m.A_beta[t])
    m.Kn = pyo.Expression(m.t, rule=lambda m, t: m.Kn0 * m.A_Kn[t])
    m.Kg = pyo.Expression(m.t, rule=lambda m, t: m.Kg0 * m.A_Kg[t])
    m.Kf = pyo.Expression(m.t, rule=lambda m, t: m.Kf0 * m.A_Kf[t])
    m.Kig = pyo.Expression(m.t, rule=lambda m, t: m.Kig0 * m.A_Kig[t])
    m.Kie = pyo.Expression(m.t, rule=lambda m, t: m.Kie0 * m.A_Kie[t])
    m.iG_T = pyo.Expression(m.t, rule=lambda m, t: m.iG / m.A_Kig[t])
    m.iE_T = pyo.Expression(m.t, rule=lambda m, t: m.iE / m.A_Kie[t])
    m.maintenance = pyo.Expression(
        m.t, rule=lambda m, t: m.m0 * pyo.exp(m.Eam * (m.T_K[t] - 293.3) / (293.3 * m.R * m.T_K[t]))
    )

    m.N_limitation = pyo.Expression(m.t, rule=lambda m, t: m.N_eff[t] / (m.N_eff[t] + m.Kn[t]))
    m.G_limitation = pyo.Expression(m.t, rule=lambda m, t: m.G_eff[t] / (m.G_eff[t] + m.Kg[t]))
    m.F_limitation = pyo.Expression(m.t, rule=lambda m, t: m.F_eff[t] / (m.F_eff[t] + m.Kf[t]))
    m.E_inhibition = pyo.Expression(m.t, rule=lambda m, t: 1.0 / (1.0 + m.iE_T[t] * m.E[t]))
    m.G_inhibition_for_fructose = pyo.Expression(
        m.t, rule=lambda m, t: 1.0 / (1.0 + m.iG_T[t] * m.G_eff[t])
    )

    m.growth_factor = pyo.Expression(m.t, rule=lambda m, t: m.A_mu[t] * m.N_limitation[t])
    m.glucose_ferm_factor = pyo.Expression(
        m.t, rule=lambda m, t: m.A_beta[t] * m.G_limitation[t] * m.E_inhibition[t]
    )
    m.fructose_ferm_factor = pyo.Expression(
        m.t,
        rule=lambda m, t: m.A_beta[t]
        * m.F_limitation[t]
        * m.G_inhibition_for_fructose[t]
        * m.E_inhibition[t],
    )

    m.mu = pyo.Expression(m.t, rule=lambda m, t: m.mu0 * m.growth_factor[t])
    m.beta_G = pyo.Expression(m.t, rule=lambda m, t: m.betaG0 * m.glucose_ferm_factor[t])
    m.beta_F = pyo.Expression(m.t, rule=lambda m, t: m.betaF0 * m.fructose_ferm_factor[t])

    m.qN_growth = pyo.Expression(m.t, rule=lambda m, t: m.qN * m.growth_factor[t])
    m.qXG_growth = pyo.Expression(m.t, rule=lambda m, t: m.qXG * m.growth_factor[t])
    m.qXF_growth = pyo.Expression(m.t, rule=lambda m, t: m.qXF * m.growth_factor[t])
    m.qEG_ferm = pyo.Expression(m.t, rule=lambda m, t: m.qEG * m.glucose_ferm_factor[t])
    m.qEF_ferm = pyo.Expression(m.t, rule=lambda m, t: m.qEF * m.fructose_ferm_factor[t])

    m.Td = pyo.Expression(
        m.t, rule=lambda m, t: -0.0001 * m.E[t] ** 3 + 0.0049 * m.E[t] ** 2 - 0.1279 * m.E[t] + 315.89
    )
    m.Kd = pyo.Expression(
        m.t,
        rule=lambda m, t: pyo.Expr_if(
            IF=(m.T_K[t] >= m.Td[t]),
            THEN=m.Kd0 * pyo.exp((m.Cde * m.E[t]) + (m.Etd * (m.T_K[t] - 305.65)) / (305.65 * m.R * m.T_K[t])),
            ELSE=0.0,
        ),
    )

    @m.Constraint(m.t)
    def X_balance(m, t):
        return m.dX[t] == m.mu[t] * m.X[t] - m.Kd[t] * m.X[t]

    @m.Constraint(m.t)
    def N_balance(m, t):
        return m.dN[t] == -m.qN_growth[t] * m.X[t] + m.N_pulse_rate[t]

    @m.Constraint(m.t)
    def G_balance(m, t):
        return m.dG[t] == -(
            m.qXG_growth[t]
            + m.qEG_ferm[t]
            + m.maintenance[t] * m.maintenance_sugar_availability[t] * (m.G_eff[t] / (m.G_eff[t] + m.F_eff[t] + eps))
        ) * m.X[t]

    @m.Constraint(m.t)
    def F_balance(m, t):
        return m.dF[t] == -(
            m.qXF_growth[t]
            + m.qEF_ferm[t]
            + m.maintenance[t] * m.maintenance_sugar_availability[t] * (m.F_eff[t] / (m.G_eff[t] + m.F_eff[t] + eps))
        ) * m.X[t]

    @m.Constraint(m.t)
    def E_balance(m, t):
        return m.dE[t] == (m.beta_G[t] + m.beta_F[t]) * m.X[t]

    m.X[t0].fix(float(batch.initials["X"]))
    m.N[t0].fix(float(batch.initials["N"]))
    m.G[t0].fix(float(batch.initials["G"]))
    m.F[t0].fix(float(batch.initials["F"]))
    m.E[t0].fix(float(batch.initials["E"]))

    m.obj = pyo.Objective(expr=0.0)

    if label_model:
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        for name in theta:
            var = getattr(m, name)
            m.unknown_parameters[var] = pyo.value(var)

        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        if input_mode == "measured":
            for t in m.t:
                m.experiment_inputs[m.TempC[t]] = None
                m.experiment_inputs[m.N_pulse[t]] = None
        else:
            for s in m.temperature_segments:
                m.experiment_inputs[m.T_set[s]] = None
            for p in m.pulse_slots:
                m.experiment_inputs[m.pulse_time[p]] = None
                m.experiment_inputs[m.pulse_amount[p]] = None

        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        objective_scale = objective_measurement_error_for_batches([batch.batch_id])
        state_vars = {"X": m.X, "N": m.N, "G": m.G, "F": m.F, "E": m.E}
        for state, var in state_vars.items():
            for t, measurement in batch.measurements[state].dropna().items():
                t = float(t)
                if t in m.t:
                    m.experiment_outputs[var[t]] = float(measurement)
                    m.measurement_error[var[t]] = float(objective_scale[state])

    return m

## Experiment abstraction

Following `notebooks/experiment_abstraction.ipynb`, the fermentation batch is wrapped as a Pyomo `Experiment`. The abstraction separates the workflow into three operations:

- `create_model()`: build the unlabelled Pyomo DAE model;
- `finalize_model()`: impose experimental conditions and discretize the DAE;
- `label_experiment()`: attach the suffixes required by ParmEst and Pyomo.DoE.

The measured temperature profile and the YAN nutrition pulse profile are the current experimental inputs. They are fixed for simulation and parameter estimation, but labeled as `experiment_inputs` so they can later be promoted to design decisions.

In [ ]:
class FermentationExperiment(Experiment):
    def __init__(
        self,
        data: FermentationBatch,
        theta_initial=None,
        parameters_to_estimate=None,
        measurement_error=None,
        input_mode: str = "measured",
        temperature_segments: int = DESIGN_TEMPERATURE_SEGMENTS,
        pulse_max_count: int = DESIGN_MAX_PULSES,
        pulse_width_h: float = DESIGN_PULSE_WIDTH_H,
        fix_design_inputs: bool = False,
    ):
        """
        Pyomo experiment abstraction for one fermentation batch.

        Arguments
        ---------
        data:
            FermentationBatch instance with time, measured temperature, YAN pulses, and observations.
        theta_initial:
            Optional dictionary overriding nominal effective parameters. Original physical names are also accepted and converted.
        parameters_to_estimate:
            Optional iterable with a subset of parameter names to label as unknown.
            If omitted, all parameters in DEFAULT_THETA are labeled.
        measurement_error:
            Optional dictionary overriding objective scales by state name.
        input_mode:
            "measured" fixes the experimental temperature and pulse profiles; "design" uses
            temperature setpoints and continuous pulse timing/magnitude decision variables.
        """
        self.data = data
        self.theta_initial = _theta_for_batch(data, theta_initial)

        if parameters_to_estimate is None:
            self.parameter_names = list(self.theta_initial)
        else:
            self.parameter_names = list(parameters_to_estimate)

        unknown_names = sorted(set(self.parameter_names) - set(self.theta_initial))
        if unknown_names:
            raise ValueError(f"Unknown parameter names: {unknown_names}")

        self.measurement_error = objective_measurement_error_for_batches([data.batch_id])
        if measurement_error is not None:
            self.measurement_error.update(measurement_error)
        self.input_mode = input_mode
        self.temperature_segments = temperature_segments
        self.pulse_max_count = pulse_max_count
        self.pulse_width_h = pulse_width_h
        self.fix_design_inputs = fix_design_inputs

        self.model = None

    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

    def create_model(self):
        """Create the unlabelled fermentation model."""
        self.model = build_zenteno_pyomo_model(
            self.data,
            theta_initial=self.theta_initial,
            fix_parameters=True,
            label_model=False,
            input_mode=self.input_mode,
            temperature_segments=self.temperature_segments,
            pulse_max_count=self.pulse_max_count,
            pulse_width_h=self.pulse_width_h,
            fix_design_inputs=self.fix_design_inputs,
        )
        return self.model

    def finalize_model(self):
        """Apply experimental conditions and discretize the DAE model."""
        m = self.model
        if m is None:
            raise RuntimeError("Call create_model() before finalize_model().")
        if getattr(m, "_dae_discretized", False):
            return m

        # The model may contain internal grid points between measured samples.
        nfe = len(list(m.t)) - 1
        pyo.TransformationFactory("dae.finite_difference").apply_to(
            m,
            scheme="BACKWARD",
            nfe=nfe,
            wrt=m.t,
        )
        m._dae_discretized = True
        return m

    def label_experiment(self):
        """Label outputs, parameters, design inputs, and measurement errors."""
        m = self.model
        if m is None:
            raise RuntimeError("Call create_model() before label_experiment().")

        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        state_vars = {"X": m.X, "N": m.N, "G": m.G, "F": m.F, "E": m.E}
        for state, var in state_vars.items():
            for t, measurement in self.data.measurements[state].dropna().items():
                t = float(t)
                if t in m.t:
                    m.experiment_outputs[var[t]] = float(measurement)
                    m.measurement_error[var[t]] = float(self.measurement_error[state])

        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        for name in self.parameter_names:
            var = getattr(m, name)
            m.unknown_parameters[var] = pyo.value(var)

        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        if self.input_mode == "measured":
            for t in self.data.time:
                t = float(t)
                if t in m.t:
                    m.experiment_inputs[m.TempC[t]] = None
                    m.experiment_inputs[m.N_pulse[t]] = None
        else:
            for s in m.temperature_segments:
                m.experiment_inputs[m.T_set[s]] = None
            for p in m.pulse_slots:
                m.experiment_inputs[m.pulse_time[p]] = None
                m.experiment_inputs[m.pulse_amount[p]] = None

        return m

## Simulation

The simulation now goes through `FermentationExperiment.get_labeled_model()`, matching the abstraction used by Dowling's examples. `finalize_model()` discretizes the DAE; the solve step only solves the resulting square algebraic system.


In [ ]:
def solve_dynamic_model(m, tee: bool = False):
    solver = pyo.SolverFactory("ipopt", executable=IPOPT_EXECUTABLE)
    if not solver.available(False):
        raise RuntimeError("Ipopt is not available. Install Ipopt/IDAES before running this notebook.")
    for key, value in IPOPT_SOLVER_OPTIONS.items():
        solver.options[key] = value
    solver.options["print_level"] = 5 if tee else 0
    return solver.solve(m, tee=tee)


def extract_simulation_results(m) -> pd.DataFrame:
    rows = []
    physical_scalar_names = ["Kn0", "Kg0", "Kf0", "Kig0", "Kie0", "Yxn", "Yxg", "Yxf", "Yeg", "Yef"]
    physical_scalars = {
        name: pyo.value(getattr(m, name))
        for name in physical_scalar_names
        if hasattr(m, name)
    }
    for t in m.t:
        row = {
            "t": float(pyo.value(t)),
            "temperature_c": pyo.value(m.TempC[t]),
            "N_pulse": pyo.value(m.N_pulse[t]),
            "N_pulse_rate": pyo.value(m.N_pulse_rate[t]),
            "X": pyo.value(m.X[t]),
            "N": pyo.value(m.N[t]),
            "G": pyo.value(m.G[t]),
            "F": pyo.value(m.F[t]),
            "E": pyo.value(m.E[t]),
            "mu": pyo.value(m.mu[t]),
            "beta_G": pyo.value(m.beta_G[t]),
            "beta_F": pyo.value(m.beta_F[t]),
            "growth_factor": pyo.value(m.growth_factor[t]),
            "glucose_ferm_factor": pyo.value(m.glucose_ferm_factor[t]),
            "fructose_ferm_factor": pyo.value(m.fructose_ferm_factor[t]),
            "qN_growth": pyo.value(m.qN_growth[t]),
            "qXG_growth": pyo.value(m.qXG_growth[t]),
            "qXF_growth": pyo.value(m.qXF_growth[t]),
            "qEG_ferm": pyo.value(m.qEG_ferm[t]),
            "qEF_ferm": pyo.value(m.qEF_ferm[t]),
            "Kn": pyo.value(m.Kn[t]),
            "Kg": pyo.value(m.Kg[t]),
            "Kf": pyo.value(m.Kf[t]),
            "Kig": pyo.value(m.Kig[t]),
            "Kie": pyo.value(m.Kie[t]),
            "iG_T": pyo.value(m.iG_T[t]),
            "iE_T": pyo.value(m.iE_T[t]),
            "Kd": pyo.value(m.Kd[t]),
            "maintenance": pyo.value(m.maintenance[t]),
            "Td_K": pyo.value(m.Td[t]),
        }
        row.update(physical_scalars)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("t").reset_index(drop=True)


def simulate_batch(
    batch: FermentationBatch,
    theta_initial=None,
    parameters_to_estimate=None,
    measurement_error=None,
    input_mode: str = "measured",
    temperature_segments: int = DESIGN_TEMPERATURE_SEGMENTS,
    pulse_max_count: int = DESIGN_MAX_PULSES,
    pulse_width_h: float = DESIGN_PULSE_WIDTH_H,
    fix_design_inputs: bool = False,
    tee: bool = False,
):
    experiment = FermentationExperiment(
        batch,
        theta_initial=theta_initial,
        parameters_to_estimate=parameters_to_estimate,
        measurement_error=measurement_error,
        input_mode=input_mode,
        temperature_segments=temperature_segments,
        pulse_max_count=pulse_max_count,
        pulse_width_h=pulse_width_h,
        fix_design_inputs=fix_design_inputs,
    )
    m = experiment.get_labeled_model()
    results = solve_dynamic_model(m, tee=tee)
    sim = extract_simulation_results(m)
    return m, results, sim


model, solve_results, sim = simulate_batch(batch)
print("Solver status:", solve_results.solver.status)
print("Termination:", solve_results.solver.termination_condition)
print("Labeled unknown parameters:", len(model.unknown_parameters))
print("Labeled experiment inputs:", len(model.experiment_inputs))
print("Labeled experiment outputs:", len(model.experiment_outputs))
display(sim.tail())


## Design-mode experiment inputs

The design version keeps the same dynamic model and output labels, but promotes temperature and nutrition inputs to decision variables. Temperature is represented by setpoints over fixed process segments; nutrition is represented by up to five pulse slots with continuous timing and YAN magnitude. Pulse timing is bounded to the first two thirds of the observed fermentation horizon. The pulse count is represented by active slots: a slot with zero YAN magnitude is inactive, which keeps this abstraction continuous for Pyomo DoE.

In [ ]:
design_experiment = FermentationExperiment(batch, input_mode="design", fix_design_inputs=False)
design_model = design_experiment.get_labeled_model()
pulse_time_upper = float(batch.time[0] + DESIGN_PULSE_ALLOWED_FRACTION * (batch.time[-1] - batch.time[0]))

print("Design-mode unknown parameters:", len(design_model.unknown_parameters))
print("Design-mode experiment inputs:", len(design_model.experiment_inputs))
print("Temperature setpoints:", len(design_model.T_set))
print("Pulse slots:", len(design_model.pulse_slots))
print("Pulse time upper bound (h):", round(pulse_time_upper, 2))
print("YAN per-pulse upper bound (kg/m3):", DESIGN_MAX_YAN_PER_PULSE)
print("YAN total upper bound (kg/m3):", DESIGN_MAX_TOTAL_YAN)


In [ ]:
def plot_simulation(batch: FermentationBatch, sim: pd.DataFrame):
    fig, ax = plt.subplots(4, 1, figsize=(11, 10), sharex=True)

    ax[0].plot(batch.time, batch.temperature_c, color="tab:red", label="Measured temperature")
    ax0b = ax[0].twinx()
    ax0b.bar(sim["t"], sim["N_pulse"], width=3.5, color="tab:cyan", alpha=0.45, label="YAN pulse")
    ax[0].set_ylabel("deg C")
    ax0b.set_ylabel("Pulse (kg/m3)")
    handles, labels = ax[0].get_legend_handles_labels()
    handles2, labels2 = ax0b.get_legend_handles_labels()
    ax[0].legend(handles + handles2, labels + labels2, loc="best")
    ax[0].grid(True, alpha=0.3)

    for state, color in [("G", "tab:blue"), ("F", "tab:orange"), ("E", "tab:green")]:
        ax[1].plot(sim["t"], sim[state], color=color, label=f"{state} simulated")
        y = batch.measurements[state].dropna()
        ax[1].scatter(y.index, y.values, color=color, marker="o", s=28, alpha=0.75, label=f"{state} measured")
    ax[1].set_ylabel("kg/m3")
    ax[1].legend(ncol=3)
    ax[1].grid(True, alpha=0.3)

    ax[2].plot(sim["t"], sim["X"], color="tab:purple", label="X simulated")
    y = batch.measurements["X"].dropna()
    ax[2].scatter(y.index, y.values, color="tab:purple", s=28, alpha=0.75, label="X measured")
    ax[2].set_ylabel("X (kg/m3)")
    ax[2].legend()
    ax[2].grid(True, alpha=0.3)

    ax[3].plot(sim["t"], sim["N"], color="tab:brown", label="N simulated")
    y = batch.measurements["N"].dropna()
    ax[3].scatter(y.index, y.values, color="tab:brown", marker="x", s=34, alpha=0.85, label="N measured")
    ax[3].set_ylabel("N (kg/m3)")
    ax[3].set_xlabel("Time (h)")
    ax[3].legend()
    ax[3].grid(True, alpha=0.3)

    fig.suptitle(f"Zenteno kinetic model simulation - batch {batch.batch_id}")
    fig.tight_layout()
    return fig, ax

plot_simulation(batch, sim);

## Optional: simulate all batches

The following helper is useful for checking the nominal model across the complete calibration workbook. It is intentionally not called by default because later parameter-estimation runs will be more expensive, and the uncalibrated nominal model can drive some long/pulsed batches to substrate-depletion bounds.

In [ ]:
def simulate_all_batches(batch_ids=None, tee: bool = False) -> pd.DataFrame:
    if batch_ids is None:
        batch_ids = available_batches

    rows = []
    for batch_id in batch_ids:
        b = load_batch(batch_id)
        try:
            m, res, s = simulate_batch(b, tee=tee)
            final = s.iloc[-1]
            rows.append(
                {
                    "batch": batch_id,
                    "status": str(res.solver.status),
                    "termination": str(res.solver.termination_condition),
                    "final_X": final["X"],
                    "final_N": final["N"],
                    "final_G": final["G"],
                    "final_F": final["F"],
                    "final_E": final["E"],
                    "total_N_pulse": s["N_pulse"].sum(),
                }
            )
        except Exception as err:
            rows.append({"batch": batch_id, "status": "failed", "termination": repr(err)})
    return pd.DataFrame(rows).set_index("batch")


# all_batch_summary = simulate_all_batches()
# display(all_batch_summary)

## ParmEst helper functions

This section defines the common `parmest.Estimator` wrappers used later for local WSSE polishing, covariance calculations, and profile-likelihood solves. The first actual parameter-estimation pass is intentionally delayed until after the nominal Q/eigen screening and the global PSO search, so IPOPT starts from a filtered global candidate rather than from the nominal reference point.


In [ ]:
PARAMETER_ESTIMATION_BATCHES = available_batches
PARMEST_SOLVER_OPTIONS = {
    **IPOPT_SOLVER_OPTIONS,
    "print_level": 5,
    "hessian_approximation": "limited-memory",
    # IDAES Ipopt 3.13.x exits on newer/unsupported acceptable_* options.
    "acceptable_tol": 1e-2,
    "acceptable_dual_inf_tol": 1e8,
    "acceptable_constr_viol_tol": 1e-6,
    "acceptable_compl_inf_tol": 1.0,
}


def normalize_batch_ids(batch_ids=PARAMETER_ESTIMATION_BATCHES):
    if batch_ids is None:
        return list(PARAMETER_ESTIMATION_BATCHES)
    if isinstance(batch_ids, (str, int)):
        return [str(batch_ids)]
    return list(batch_ids)


def build_parmest_experiments(batch_ids=PARAMETER_ESTIMATION_BATCHES, parameters_to_estimate=None, theta_initial=None):
    batch_ids = normalize_batch_ids(batch_ids)
    objective_scale_lookup = objective_scale_lookup_for_batches(batch_ids)
    return [
        FermentationExperiment(
            load_batch(batch_id),
            theta_initial=theta_initial,
            parameters_to_estimate=parameters_to_estimate,
            measurement_error={
                state: objective_scale_lookup[(str(batch_id), state)]
                for state in STATE_LABELS
            },
            input_mode="measured",
        )
        for batch_id in batch_ids
    ]


def run_parmest_estimation(
    obj_function: str,
    batch_ids=PARAMETER_ESTIMATION_BATCHES,
    tee: bool = False,
    theta_initial=None,
    parameters_to_estimate=None,
):
    batch_ids = normalize_batch_ids(batch_ids)
    experiments = build_parmest_experiments(
        batch_ids=batch_ids,
        parameters_to_estimate=parameters_to_estimate,
        theta_initial=theta_initial,
    )
    estimator = parmest.Estimator(
        experiments,
        obj_function=obj_function,
        tee=tee,
        solver_options=PARMEST_SOLVER_OPTIONS,
    )
    obj_value_per_scenario, theta = estimator.theta_est()
    # ParmEst builds an extensive form with equal scenario probabilities, so
    # theta_est() reports the average scenario objective. The direct Gaussian
    # WSSE diagnostics and likelihood-ratio tests use the total objective over all
    # batches.
    obj_value = float(obj_value_per_scenario) * len(batch_ids)
    estimator.obj_value_per_scenario = float(obj_value_per_scenario)
    estimator.obj_value_total = float(obj_value)
    return obj_value, theta, estimator


def clip_theta_to_bounds(theta, atol: float = 1e-8):
    clipped = {}
    adjustments = []
    for name, value in theta.items():
        if value is None:
            continue
        if name not in PARAMETER_BOUNDS:
            continue
        value = float(value)
        lb, ub = PARAMETER_BOUNDS[name]
        clipped_value = min(max(value, lb), ub)
        clipped[name] = clipped_value
        if abs(clipped_value - value) > atol:
            adjustments.append(
                {
                    "parameter": name,
                    "raw_value": value,
                    "clipped_value": clipped_value,
                    "lower_bound": lb,
                    "upper_bound": ub,
                }
            )
    return clipped, pd.DataFrame(adjustments)


def evaluate_fit_objectives(theta, batch_ids=PARAMETER_ESTIMATION_BATCHES):
    batch_ids = normalize_batch_ids(batch_ids)
    scale_records = objective_scale_records_for_batches(batch_ids)
    rows = []
    solve_rows = []
    for batch_id in batch_ids:
        batch_eval = load_batch(batch_id)
        try:
            model_eval, solve_result, sim_eval = simulate_batch(batch_eval, theta_initial=theta)
        except Exception as err:
            solve_rows.append(
                {
                    "batch": batch_id,
                    "status": "failed",
                    "termination": repr(err),
                }
            )
            continue
        solve_rows.append(
            {
                "batch": batch_id,
                "status": str(solve_result.solver.status),
                "termination": str(solve_result.solver.termination_condition),
            }
        )
        sim_by_time = sim_eval.set_index("t")
        for state in STATE_LABELS:
            scale_record = scale_records[(str(batch_id), state)]
            sigma = float(scale_record["objective_scale"])
            state_scale = float(scale_record["state_scale"])
            state_n_observations = int(scale_record["n_observations"])
            for t, observed in batch_eval.measurements[state].dropna().items():
                predicted = float(sim_by_time.loc[float(t), state])
                residual = predicted - float(observed)
                rows.append(
                    {
                        "batch": batch_id,
                        "state": state,
                        "t": float(t),
                        "observed": float(observed),
                        "predicted": predicted,
                        "residual": residual,
                        "measurement_error": sigma,
                        "objective_scale": sigma,
                        "state_scale": state_scale,
                        "state_n_observations": state_n_observations,
                        "experiment_state_count": int(scale_record["state_block_count"]),
                        "squared_error": residual**2,
                        "weighted_squared_error": 0.5 * (residual / sigma) ** 2,
                    }
                )
    residuals = pd.DataFrame(rows)
    solve_summary = pd.DataFrame(solve_rows).set_index("batch")
    objective_summary = {
        "n_observations": len(residuals),
        "SSE_raw_sum": residuals["squared_error"].sum() if not residuals.empty else np.nan,
        "WSSE_raw_sum": residuals["weighted_squared_error"].sum() if not residuals.empty else np.nan,
        "objective_scale_mode": OBJECTIVE_SCALE_MODE,
        "objective_scale_scope": OBJECTIVE_SCALE_SCOPE,
        "objective_normalized_by_state_count": OBJECTIVE_NORMALIZE_BY_STATE_COUNT,
        "objective_normalized_by_experiment_state_count": OBJECTIVE_NORMALIZE_BY_EXPERIMENT_STATE_COUNT,
    }
    return objective_summary, residuals, solve_summary


def leave_one_batch_out_folds(batch_ids):
    batch_ids = normalize_batch_ids(batch_ids)
    if len(batch_ids) < 2:
        raise ValueError("Cross-validation requires at least two experiments.")
    folds = []
    for fold, validation_batch in enumerate(batch_ids, start=1):
        validation_batches = [validation_batch]
        train_batches = [batch_id for batch_id in batch_ids if batch_id not in validation_batches]
        folds.append(
            {
                "fold": fold,
                "train_batches": train_batches,
                "validation_batches": validation_batches,
            }
        )
    return folds


def run_leave_one_batch_out_cv(
    obj_function: str,
    batch_ids,
    theta_initial=None,
    parameters_to_estimate=None,
    tee: bool = False,
    continue_on_failure: bool = False,
):
    batch_ids = normalize_batch_ids(batch_ids)
    folds = leave_one_batch_out_folds(batch_ids)
    reference_batch = load_batch(batch_ids[0])
    if theta_initial is None:
        theta_reference = _theta_for_batch(reference_batch, DEFAULT_THETA)
    else:
        theta_reference = _theta_for_batch(reference_batch, theta_initial)

    fold_rows = []
    theta_rows = []
    residual_tables = {}
    solve_tables = {}
    for fold_info in folds:
        fold = int(fold_info["fold"])
        train_batches = fold_info["train_batches"]
        validation_batches = fold_info["validation_batches"]
        try:
            train_objective, theta_fold_raw, _ = run_parmest_estimation(
                obj_function,
                batch_ids=train_batches,
                tee=tee,
                theta_initial=theta_reference,
                parameters_to_estimate=parameters_to_estimate,
            )
            theta_fold_complete = dict(theta_reference)
            theta_fold_complete.update(theta_fold_raw)
            theta_fold, theta_adjustments = clip_theta_to_bounds(theta_fold_complete)
            train_summary, train_residuals, train_solve = evaluate_fit_objectives(
                theta_fold,
                batch_ids=train_batches,
            )
            validation_summary, validation_residuals, validation_solve = evaluate_fit_objectives(
                theta_fold,
                batch_ids=validation_batches,
            )
            train_objective_direct = float(train_summary.get("WSSE_raw_sum", np.nan))
            validation_objective = float(validation_summary.get("WSSE_raw_sum", np.nan))
            fold_rows.append(
                {
                    "fold": fold,
                    "status": "ok",
                    "train_batches": ", ".join(map(str, train_batches)),
                    "validation_batches": ", ".join(map(str, validation_batches)),
                    "train_objective_parmest": float(train_objective),
                    "train_objective_direct": train_objective_direct,
                    "train_objective_per_experiment": train_objective_direct / max(len(train_batches), 1),
                    "validation_objective": validation_objective,
                    "validation_objective_per_experiment": validation_objective / max(len(validation_batches), 1),
                    "n_train_observations": int(train_summary.get("n_observations", 0)),
                    "n_validation_observations": int(validation_summary.get("n_observations", 0)),
                    "n_bound_adjustments": int(len(theta_adjustments)),
                    "error": "",
                }
            )
            theta_rows.append(pd.Series(theta_fold, name=fold))
            residual_tables[(fold, "train")] = train_residuals
            residual_tables[(fold, "validation")] = validation_residuals
            solve_tables[(fold, "train")] = train_solve
            solve_tables[(fold, "validation")] = validation_solve
        except Exception as err:
            fold_rows.append(
                {
                    "fold": fold,
                    "status": "failed",
                    "train_batches": ", ".join(map(str, train_batches)),
                    "validation_batches": ", ".join(map(str, validation_batches)),
                    "train_objective_parmest": np.nan,
                    "train_objective_direct": np.nan,
                    "train_objective_per_experiment": np.nan,
                    "validation_objective": np.nan,
                    "validation_objective_per_experiment": np.nan,
                    "n_train_observations": np.nan,
                    "n_validation_observations": np.nan,
                    "n_bound_adjustments": np.nan,
                    "error": f"{type(err).__name__}: {err}",
                }
            )
            if not continue_on_failure:
                raise

    fold_summary = pd.DataFrame(fold_rows).set_index("fold")
    theta_by_fold = pd.DataFrame(theta_rows) if theta_rows else pd.DataFrame()
    success = fold_summary[fold_summary["status"].eq("ok")]
    cv_summary = pd.Series(
        {
            "n_folds": int(len(fold_summary)),
            "n_success": int(len(success)),
            "cv_validation_objective_mean": float(success["validation_objective_per_experiment"].mean()) if not success.empty else np.nan,
            "cv_validation_objective_std": float(success["validation_objective_per_experiment"].std(ddof=0)) if not success.empty else np.nan,
            "cv_train_objective_mean": float(success["train_objective_per_experiment"].mean()) if not success.empty else np.nan,
            "objective_scale_mode": OBJECTIVE_SCALE_MODE,
            "objective_scale_scope": OBJECTIVE_SCALE_SCOPE,
            "objective_normalized_by_state_count": OBJECTIVE_NORMALIZE_BY_STATE_COUNT,
            "objective_normalized_by_experiment_state_count": OBJECTIVE_NORMALIZE_BY_EXPERIMENT_STATE_COUNT,
        },
        name="leave_one_batch_out_cv",
    )
    return {
        "summary": cv_summary,
        "fold_summary": fold_summary,
        "theta_by_fold": theta_by_fold,
        "residual_tables": residual_tables,
        "solve_tables": solve_tables,
    }


print("ParmEst batches:", PARAMETER_ESTIMATION_BATCHES)
print("Unknown parameters:", list(DEFAULT_THETA))


## Nominal sensitivity / estimability pre-screen in effective-parameter space

Evaluate the weighted finite-difference sensitivity matrix around the nominal effective parameter vector before the initial ParmEst fit. Use the weakest eigen-directions and loadings from this block to manually populate `INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN` in the next cell. Those fixed parameters stay fixed during the initial Gaussian WSSE ParmEst polish, post-fit Q/eigen-analysis, covariance, and profile likelihood.


In [ ]:
RUN_Q_EIGEN_ANALYSIS = True
ESTIMABILITY_ANALYSIS_BATCHES = ["25026", "25086", "25150", "25170"]
Q_ANALYSIS_LABEL = "nominal"
Q_REFERENCE_BATCH = ESTIMABILITY_ANALYSIS_BATCHES[0]
Q_ANALYSIS_BATCHES = ESTIMABILITY_ANALYSIS_BATCHES
Q_EXCLUDED_PARAMETERS = set()
Q_ANALYSIS_PARAMETERS = [p for p in DEFAULT_THETA if p not in Q_EXCLUDED_PARAMETERS]
Q_ANALYSIS_OUTPUT_STATES = list(STATE_LABELS)
Q_PERTURBATION_FRACTION = 0.10
Q_EIGEN_TOP_N_LOADINGS = 5
Q_EIGEN_NEAR_NULL_RTOL = 1e-8
Q_DROP_INCOMPLETE_PARAMETERS = True
Q_MIN_EFFECTIVE_PARAMETERS = 2
Q_ANALYSIS_RESULTS_DIR = FERMENTATION_DIR / "results"


def get_q_analysis_theta(fit_label=Q_ANALYSIS_LABEL):
    theta = _theta_for_batch(load_batch(Q_REFERENCE_BATCH), DEFAULT_THETA)
    theta_clipped, _ = clip_theta_to_bounds(theta)
    return {name: float(theta_clipped[name]) for name in Q_ANALYSIS_PARAMETERS}


def profile_perturbation_points(theta, parameter, fraction=Q_PERTURBATION_FRACTION):
    nominal = float(theta[parameter])
    lb, ub = PARAMETER_BOUNDS[parameter]
    lb = float(lb)
    ub = float(ub)

    lower = max(lb, nominal * (1.0 - fraction))
    upper = min(ub, nominal * (1.0 + fraction))
    mode = "central_clipped"

    if np.isclose(lower, upper, rtol=1e-12, atol=1e-14):
        span = max(abs(nominal) * fraction, 0.01 * (ub - lb), 1e-10)
        if nominal <= lb + 1e-12 * max(1.0, abs(lb)):
            lower = nominal
            upper = min(ub, nominal + span)
            mode = "forward_at_lower_bound"
        elif nominal >= ub - 1e-12 * max(1.0, abs(ub)):
            lower = max(lb, nominal - span)
            upper = nominal
            mode = "backward_at_upper_bound"
        else:
            lower = max(lb, nominal - span)
            upper = min(ub, nominal + span)
            mode = "fallback_central"

    if not upper > lower:
        raise RuntimeError(
            f"Cannot build a finite-difference perturbation for {parameter}: "
            f"lower={lower}, upper={upper}, bounds=({lb}, {ub})."
        )

    return lower, upper, mode


def measurement_index_for_batches(batch_ids, states=Q_ANALYSIS_OUTPUT_STATES):
    batch_ids = normalize_batch_ids(batch_ids)
    scale_records = objective_scale_records_for_batches(batch_ids=batch_ids, states=states)
    rows = []
    for batch_id in batch_ids:
        batch = load_batch(batch_id)
        for state in states:
            scale_record = scale_records[(str(batch_id), state)]
            sigma = float(scale_record["objective_scale"])
            for t, observed in batch.measurements[state].dropna().items():
                rows.append(
                    {
                        "batch": str(batch_id),
                        "state": state,
                        "t": float(t),
                        "measurement_error": sigma,
                        "objective_scale": sigma,
                        "state_scale": float(scale_record["state_scale"]),
                        "state_n_observations": int(scale_record["n_observations"]),
                        "experiment_state_count": int(scale_record["state_block_count"]),
                    }
                )
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError("No measured outputs found for Q analysis.")
    return frame.set_index(["batch", "state", "t"])


def simulate_prediction_series(theta, batch_ids, states=Q_ANALYSIS_OUTPUT_STATES):
    values = {}
    failures = []
    for batch_id in batch_ids:
        batch = load_batch(batch_id)
        try:
            _, result, sim = simulate_batch(batch, theta_initial=theta)
        except Exception as err:
            failures.append(
                {
                    "batch": str(batch_id),
                    "status": "failed",
                    "termination": f"{type(err).__name__}: {err}",
                }
            )
            continue

        failures.append(
            {
                "batch": str(batch_id),
                "status": str(result.solver.status),
                "termination": str(result.solver.termination_condition),
            }
        )
        if not pyo.check_optimal_termination(result):
            continue
        sim_by_time = sim.set_index("t")
        for state in states:
            for t in batch.measurements[state].dropna().index:
                key = (str(batch_id), state, float(t))
                values[key] = float(sim_by_time.loc[float(t), state])
    series = pd.Series(values, name="prediction")
    if not series.empty:
        series.index = pd.MultiIndex.from_tuples(series.index, names=["batch", "state", "t"])
    return series, pd.DataFrame(failures)


def build_weighted_relative_sensitivity_matrix(theta, parameter_names, batch_ids, fraction=Q_PERTURBATION_FRACTION):
    measurement_frame = measurement_index_for_batches(batch_ids)
    full_index = measurement_frame.index
    q_raw = pd.DataFrame(index=full_index, columns=parameter_names, dtype=float)
    perturbation_rows = []
    solve_rows = []

    for j, parameter in enumerate(parameter_names, start=1):
        lower, upper, mode = profile_perturbation_points(theta, parameter, fraction=fraction)
        delta = upper - lower
        theta_minus = dict(theta)
        theta_plus = dict(theta)
        theta_minus[parameter] = lower
        theta_plus[parameter] = upper
        print(
            f"[Q] {parameter} {j}/{len(parameter_names)}: "
            f"minus={lower:.6g}, plus={upper:.6g}, delta={delta:.6g}, mode={mode}",
            flush=True,
        )

        y_minus, failures_minus = simulate_prediction_series(theta_minus, batch_ids)
        y_plus, failures_plus = simulate_prediction_series(theta_plus, batch_ids)
        failures_minus["parameter"] = parameter
        failures_minus["side"] = "minus"
        failures_plus["parameter"] = parameter
        failures_plus["side"] = "plus"
        solve_rows.extend(failures_minus.to_dict("records"))
        solve_rows.extend(failures_plus.to_dict("records"))

        common_index = y_minus.index.intersection(y_plus.index)
        sensitivity = (y_plus.loc[common_index] - y_minus.loc[common_index]) / delta
        q_raw.loc[common_index, parameter] = sensitivity

        nominal = float(theta[parameter])
        perturbation_rows.append(
            {
                "parameter": parameter,
                "nominal": nominal,
                "lower_bound": float(PARAMETER_BOUNDS[parameter][0]),
                "upper_bound": float(PARAMETER_BOUNDS[parameter][1]),
                "theta_minus": lower,
                "theta_plus": upper,
                "delta": delta,
                "minus_pct": (lower / nominal - 1.0) if nominal != 0.0 else np.nan,
                "plus_pct": (upper / nominal - 1.0) if nominal != 0.0 else np.nan,
                "mode": mode,
                "n_common_predictions": int(len(common_index)),
            }
        )

    missing_by_parameter = q_raw.isna().sum().rename("n_missing_sensitivities")
    active_parameters = [name for name in parameter_names if q_raw[name].notna().any()]
    dropped_parameters = []
    q_complete = pd.DataFrame()

    if Q_DROP_INCOMPLETE_PARAMETERS:
        complete_parameters = [name for name in active_parameters if int(missing_by_parameter.get(name, 0)) == 0]
        incomplete_parameters = [name for name in active_parameters if int(missing_by_parameter.get(name, 0)) > 0]
        if len(complete_parameters) >= int(Q_MIN_EFFECTIVE_PARAMETERS):
            dropped_parameters.extend(incomplete_parameters)
            active_parameters = complete_parameters

    while active_parameters:
        q_complete = q_raw.loc[:, active_parameters].dropna(axis=0, how="any").astype(float)
        if not q_complete.empty or not Q_DROP_INCOMPLETE_PARAMETERS:
            break
        missing_active = q_raw.loc[:, active_parameters].isna().sum()
        drop_name = str(missing_active.sort_values(ascending=False).index[0])
        dropped_parameters.append(drop_name)
        active_parameters.remove(drop_name)

    if not active_parameters or q_complete.empty:
        diagnostic = pd.DataFrame(perturbation_rows).set_index("parameter") if perturbation_rows else pd.DataFrame()
        if not diagnostic.empty:
            diagnostic = diagnostic.join(missing_by_parameter)
        solve_summary = pd.DataFrame(solve_rows)
        if not diagnostic.empty:
            display(diagnostic)
        if not solve_summary.empty:
            display(solve_summary.groupby(["parameter", "side", "status", "termination"]).size().rename("count").reset_index())
        raise RuntimeError(
            "The sensitivity matrix is empty after selecting complete rows. "
            "Inspect q_perturbation_summary/q_solve_summary above; at least one perturbation failed or produced no common predictions."
        )

    if len(active_parameters) < int(Q_MIN_EFFECTIVE_PARAMETERS):
        raise RuntimeError(
            f"Only {len(active_parameters)} parameter(s) have a complete sensitivity matrix: {active_parameters}. "
            "Increase solver robustness, reduce Q_ANALYSIS_PARAMETERS, or inspect failed perturbations."
        )

    if dropped_parameters:
        print(
            "WARNING: Q analysis excluded parameters with incomplete sensitivities: "
            + ", ".join(dropped_parameters),
            flush=True,
        )

    sigma = measurement_frame.loc[q_complete.index, "measurement_error"].astype(float)
    q_weighted = q_complete.div(sigma, axis=0)

    parameter_scales = pd.Series(
        {name: max(abs(float(theta[name])), 1e-12) for name in active_parameters},
        name="parameter_scale",
    )
    q_relative = q_weighted.mul(parameter_scales, axis=1)

    perturbation_summary = pd.DataFrame(perturbation_rows).set_index("parameter")
    perturbation_summary = perturbation_summary.join(missing_by_parameter)
    perturbation_summary["used_in_q_matrix"] = perturbation_summary.index.isin(active_parameters)
    perturbation_summary["dropped_from_q_reason"] = ""
    perturbation_summary.loc[perturbation_summary.index.isin(dropped_parameters), "dropped_from_q_reason"] = "incomplete_sensitivity_column"
    perturbation_summary.loc[~perturbation_summary.index.isin(active_parameters) & perturbation_summary["dropped_from_q_reason"].eq(""), "dropped_from_q_reason"] = "no_sensitivities"
    solve_summary = pd.DataFrame(solve_rows)
    return q_raw, q_weighted, q_relative, perturbation_summary, solve_summary


def eigen_analysis_from_q(q_matrix, parameter_names, top_n=Q_EIGEN_TOP_N_LOADINGS, near_null_rtol=Q_EIGEN_NEAR_NULL_RTOL):
    q_array = q_matrix.loc[:, parameter_names].to_numpy(dtype=float)
    fim = pd.DataFrame(q_array.T @ q_array, index=parameter_names, columns=parameter_names)
    fim_symmetric = 0.5 * (fim.to_numpy(dtype=float) + fim.to_numpy(dtype=float).T)
    eigvals, eigvecs = np.linalg.eigh(fim_symmetric)
    order = np.argsort(eigvals)
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    max_eig = float(np.max(eigvals)) if len(eigvals) else np.nan
    positive = eigvals[eigvals > max(max_eig * near_null_rtol, 0.0)] if np.isfinite(max_eig) and max_eig > 0 else np.array([])
    condition_number = float(max_eig / np.min(positive)) if len(positive) else np.inf

    eigen_summary = pd.DataFrame(
        {
            "direction": np.arange(1, len(eigvals) + 1),
            "fim_eigenvalue": eigvals,
            "relative_eigenvalue": eigvals / max_eig if np.isfinite(max_eig) and max_eig > 0 else np.nan,
            "singular_value": np.sqrt(np.maximum(eigvals, 0.0)),
            "near_null": eigvals <= near_null_rtol * max_eig if np.isfinite(max_eig) and max_eig > 0 else False,
            "condition_number": condition_number,
        }
    )

    loading_rows = []
    for direction_idx in range(eigvecs.shape[1]):
        vector = eigvecs[:, direction_idx]
        abs_vector = np.abs(vector)
        top_order = np.argsort(abs_vector)[::-1][:top_n]
        loading_rows.append(
            {
                "direction": direction_idx + 1,
                "fim_eigenvalue": float(eigvals[direction_idx]),
                "relative_eigenvalue": float(eigvals[direction_idx] / max_eig) if np.isfinite(max_eig) and max_eig > 0 else np.nan,
                "dominant_parameters": ", ".join(parameter_names[i] for i in top_order),
                "dominant_loadings": ", ".join(f"{vector[i]:+.3f}" for i in top_order),
                "dominant_abs_loadings": ", ".join(f"{abs_vector[i]:.3f}" for i in top_order),
            }
        )
    loading_summary = pd.DataFrame(loading_rows)
    return fim, eigen_summary, loading_summary


if RUN_Q_EIGEN_ANALYSIS:
    q_analysis_theta = get_q_analysis_theta()
    display(pd.Series(q_analysis_theta, name="q_analysis_theta"))
    print("Q analysis batches:", Q_ANALYSIS_BATCHES)
    print("Q perturbation fraction:", Q_PERTURBATION_FRACTION)

    q_raw, q_weighted, q_relative, q_perturbation_summary, q_solve_summary = build_weighted_relative_sensitivity_matrix(
        q_analysis_theta,
        Q_ANALYSIS_PARAMETERS,
        Q_ANALYSIS_BATCHES,
        fraction=Q_PERTURBATION_FRACTION,
    )
    q_effective_parameters = list(q_relative.columns)
    q_dropped_parameters = [name for name in Q_ANALYSIS_PARAMETERS if name not in q_effective_parameters]
    if q_dropped_parameters:
        print("Q/eigen excluded parameters:", q_dropped_parameters)

    fim_relative, q_eigen_summary, q_loading_summary = eigen_analysis_from_q(
        q_relative,
        q_effective_parameters,
    )
    fim_weighted_unscaled, q_eigen_summary_unscaled, q_loading_summary_unscaled = eigen_analysis_from_q(
        q_weighted,
        q_effective_parameters,
    )

    q_diagnostic_summary = pd.Series(
        {
            "n_measurement_rows_total": int(q_raw.shape[0]),
            "n_measurement_rows_complete": int(q_relative.shape[0]),
            "n_parameters": int(q_relative.shape[1]),
            "effective_parameters": ", ".join(q_effective_parameters),
            "dropped_parameters": ", ".join(q_dropped_parameters) or "none",
            "relative_condition_number": float(q_eigen_summary["condition_number"].iloc[0]),
            "unscaled_condition_number": float(q_eigen_summary_unscaled["condition_number"].iloc[0]),
        },
        name="q_eigen_diagnostic",
    )

    display(q_diagnostic_summary)
    display(q_perturbation_summary)
    if not q_solve_summary.empty:
        display(q_solve_summary.groupby(["parameter", "side", "status", "termination"]).size().rename("count").reset_index())

    print("Eigen-analysis of weighted, relative-scaled Q (primary diagnostic):")
    display(q_eigen_summary)
    display(q_loading_summary)

    print("Weakest directions only:")
    display(q_loading_summary.head(min(5, len(q_loading_summary))))

    q_eigen_plot_df = q_eigen_summary.copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].semilogy(
        q_eigen_plot_df["direction"],
        q_eigen_plot_df["fim_eigenvalue"].clip(lower=np.finfo(float).tiny),
        marker="o",
    )
    axes[0].set_title("FIM eigenvalues from weighted relative Q")
    axes[0].set_xlabel("Eigen-direction (weak to strong)")
    axes[0].set_ylabel("Eigenvalue (log scale)")
    axes[0].grid(True, which="both", alpha=0.3)

    axes[1].semilogy(
        q_eigen_plot_df["direction"],
        q_eigen_plot_df["relative_eigenvalue"].clip(lower=np.finfo(float).tiny),
        marker="o",
        color="tab:orange",
    )
    axes[1].axhline(Q_EIGEN_NEAR_NULL_RTOL, color="tab:red", linestyle="--", label="near-null tolerance")
    axes[1].set_title("Relative eigenvalues")
    axes[1].set_xlabel("Eigen-direction (weak to strong)")
    axes[1].set_ylabel("Eigenvalue / max eigenvalue")
    axes[1].grid(True, which="both", alpha=0.3)
    axes[1].legend(loc="best")
    fig.tight_layout()
    plt.show()

    Q_ANALYSIS_RESULTS_DIR.mkdir(exist_ok=True)
    q_raw.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_raw_nominal_pm10.csv")
    q_weighted.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_weighted_nominal_pm10.csv")
    q_relative.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_weighted_relative_nominal_pm10.csv")
    fim_relative.to_csv(Q_ANALYSIS_RESULTS_DIR / "fim_weighted_relative_nominal_pm10.csv")
    q_eigen_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "fim_weighted_relative_eigenvalues_nominal_pm10.csv", index=False)
    q_loading_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "fim_weighted_relative_eigendirections_nominal_pm10.csv", index=False)
    q_perturbation_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_perturbation_summary_nominal_pm10.csv")
    q_solve_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_solve_summary_nominal_pm10.csv", index=False)


if RUN_Q_EIGEN_ANALYSIS:
    nominal_q_eigen_summary = q_eigen_summary
    nominal_q_loading_summary = q_loading_summary
    nominal_q_perturbation_summary = q_perturbation_summary
    nominal_q_solve_summary = q_solve_summary


## Manual filter from the nominal eigen-analysis

After inspecting the nominal eigen-directions, edit `INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN` using effective parameter names. Parameters listed there are kept fixed at nominal values during the local Gaussian WSSE ParmEst polish, post-fit Q/eigen-analysis, covariance, and profile likelihood. The PSO helper keeps a backwards-compatible alias but is no longer part of the initial fit path.


In [ ]:
# Final identifiable reduced model after the deep model-selection run.
# qXG/qXF remain weak near-null/active-bound directions; iG passed FIM and profile-likelihood checks.
INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN = {"Kd0", "sN", "sG", "sF", "iE", "m0", "qXG", "qXF"}
# Backwards-compatible aliases used by the deferred PSO helper defaults.
PSO_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN = set(INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN)
# Estimated subset after reduction: mu0, qN, betaG0, betaF0, qEG, qEF, iG.

unknown_filter_names = sorted(set(INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN) - set(DEFAULT_THETA))
if unknown_filter_names:
    raise ValueError(f"Unknown fixed parameter names: {unknown_filter_names}")

INITIAL_ESTIMATED_PARAMETERS = [
    name for name in DEFAULT_THETA
    if name not in INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN
]
PSO_ESTIMATED_PARAMETERS = list(INITIAL_ESTIMATED_PARAMETERS)

display(pd.Series(
    {
        "n_total_parameters": len(DEFAULT_THETA),
        "n_initial_estimated_parameters": len(INITIAL_ESTIMATED_PARAMETERS),
        "fixed_parameters": ", ".join(sorted(INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN)) or "none",
        "estimated_parameters": ", ".join(INITIAL_ESTIMATED_PARAMETERS),
    },
    name="manual_filter_summary",
))


## Deferred custom PSO helpers

The particle-swarm implementation is defined here but not run before the first ParmEst fit. The initial calibration path now goes directly through ParmEst/IPOPT; PSO is reserved for a later post-profile stage, where it can be restricted to parameters that the profile-likelihood screen marks as estimable.


In [ ]:
RUN_CUSTOM_PSO = False  # Deferred: PSO is run after profile-likelihood screening.

if "ESTIMABILITY_ANALYSIS_BATCHES" not in globals():
    ESTIMABILITY_ANALYSIS_BATCHES = ["25026", "25086", "25150", "25170"]

CUSTOM_PSO_CONFIG = {
    "epoch": 180,
    "pop_size": 12,
    "w": 0.5,
    "c1": 1.5,
    "c2": 1.5,
    "seed": 123,
    "verbose": False,
    "save_history": False,
    "relative_gap_threshold": 5e-4,
}
CUSTOM_PSO_FIXED_PARAMETERS = set(PSO_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN) if "PSO_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN" in globals() else set()
CUSTOM_PSO_PARAMETERS = list(PSO_ESTIMATED_PARAMETERS) if "PSO_ESTIMATED_PARAMETERS" in globals() else [
    name for name in DEFAULT_THETA if name not in CUSTOM_PSO_FIXED_PARAMETERS
]
CUSTOM_PSO_BATCHES = ESTIMABILITY_ANALYSIS_BATCHES
CUSTOM_PSO_LOG_PARAMETERS = [
    name for name in CUSTOM_PSO_PARAMETERS if PARAMETER_BOUNDS[name][0] > 0.0
]
CUSTOM_PSO_FAILED_OBJECTIVE = 1e30
CUSTOM_PSO_CHECK_INTERVAL = 5
CUSTOM_PSO_PATIENCE_CHECKS = 3
CUSTOM_PSO_PRINT_PROGRESS = True
CUSTOM_PSO_PROGRESS_EVERY_EVALUATIONS = CUSTOM_PSO_CONFIG["pop_size"]
CUSTOM_PSO_SUPPRESS_PYOMO_WARNINGS = True
CUSTOM_PSO_INCLUDE_WSSE_SEED = False  # Scaled-SSE local polish happens in the next section.
CUSTOM_PSO_POLISH_WITH_IPOPT = False
CUSTOM_PSO_RESULTS_DIR = FERMENTATION_DIR / "results"
CUSTOM_PSO_RESULT_PREFIX = "custom_pso"
CUSTOM_PSO_REFERENCE_THETA = None

import logging
from contextlib import contextmanager


@contextmanager
def custom_pso_pyomo_warning_filter(enabled=CUSTOM_PSO_SUPPRESS_PYOMO_WARNINGS):
    if not enabled:
        yield
        return
    logger_names = ["pyomo.core", "pyomo.opt", "pyomo.solvers"]
    loggers = [logging.getLogger(name) for name in logger_names]
    previous_levels = [logger.level for logger in loggers]
    try:
        for logger in loggers:
            logger.setLevel(logging.ERROR)
        yield
    finally:
        for logger, level in zip(loggers, previous_levels):
            logger.setLevel(level)


def custom_pso_failure_summary(solve_rows):
    if solve_rows is None or solve_rows.empty:
        return "no solve rows"
    row = solve_rows.iloc[-1]
    return f"batch={row.get('batch', '')}, status={row.get('status', '')}, termination={row.get('termination', '')}"


def custom_pso_reference_theta(reference_batch_id=None):
    if reference_batch_id is None:
        reference_batch_id = CUSTOM_PSO_BATCHES[0]
    theta = _theta_for_batch(load_batch(reference_batch_id), DEFAULT_THETA)
    if CUSTOM_PSO_REFERENCE_THETA is not None:
        theta.update({name: float(value) for name, value in CUSTOM_PSO_REFERENCE_THETA.items() if name in DEFAULT_THETA})
    theta_clipped, _ = clip_theta_to_bounds(theta)
    return {name: float(theta_clipped[name]) for name in DEFAULT_THETA}


def custom_pso_complete_theta(theta_update):
    theta = custom_pso_reference_theta()
    theta.update({name: float(value) for name, value in theta_update.items() if name in DEFAULT_THETA})
    theta_clipped, _ = clip_theta_to_bounds(theta)
    return {name: float(theta_clipped[name]) for name in DEFAULT_THETA}


def custom_pso_decode_position(position):
    theta = {}
    for coordinate, name in zip(position, CUSTOM_PSO_PARAMETERS):
        coordinate = float(np.clip(coordinate, 0.0, 1.0))
        lb, ub = PARAMETER_BOUNDS[name]
        lb = float(lb)
        ub = float(ub)
        if name in CUSTOM_PSO_LOG_PARAMETERS:
            theta[name] = 10.0 ** (np.log10(lb) + coordinate * (np.log10(ub) - np.log10(lb)))
        else:
            theta[name] = lb + coordinate * (ub - lb)
    return theta


def custom_pso_encode_theta(theta):
    position = []
    for name in CUSTOM_PSO_PARAMETERS:
        lb, ub = PARAMETER_BOUNDS[name]
        lb = float(lb)
        ub = float(ub)
        value = float(theta[name])
        value = min(max(value, lb), ub)
        if name in CUSTOM_PSO_LOG_PARAMETERS:
            coordinate = (np.log10(value) - np.log10(lb)) / (np.log10(ub) - np.log10(lb))
        else:
            coordinate = (value - lb) / (ub - lb)
        position.append(float(np.clip(coordinate, 0.0, 1.0)))
    return np.array(position, dtype=float)


def custom_pso_weighted_sse(theta, batch_ids=None, penalty=CUSTOM_PSO_FAILED_OBJECTIVE):
    if batch_ids is None:
        batch_ids = CUSTOM_PSO_BATCHES
    batch_ids = normalize_batch_ids(batch_ids)
    objective_scale = objective_scale_lookup_for_batches(batch_ids)
    total = 0.0
    solve_rows = []
    try:
        theta = custom_pso_complete_theta({name: float(theta[name]) for name in CUSTOM_PSO_PARAMETERS})
    except Exception as err:
        return float(penalty), pd.DataFrame(
            [{"batch": "parameter_decode", "status": "failed", "termination": f"{type(err).__name__}: {err}"}]
        )

    for batch_id in batch_ids:
        batch_eval = load_batch(batch_id)
        try:
            with custom_pso_pyomo_warning_filter():
                _, solve_result, sim_eval = simulate_batch(batch_eval, theta_initial=theta)
            status = str(solve_result.solver.status)
            termination = str(solve_result.solver.termination_condition)
            solve_rows.append({"batch": str(batch_id), "status": status, "termination": termination})
            if not pyo.check_optimal_termination(solve_result):
                return float(penalty), pd.DataFrame(solve_rows)
        except Exception as err:
            solve_rows.append(
                {
                    "batch": str(batch_id),
                    "status": "failed",
                    "termination": f"{type(err).__name__}: {err}",
                }
            )
            return float(penalty), pd.DataFrame(solve_rows)

        sim_by_time = sim_eval.set_index("t")
        for state in STATE_LABELS:
            sigma = float(objective_scale[(str(batch_id), state)])
            for t, observed in batch_eval.measurements[state].dropna().items():
                try:
                    predicted = float(sim_by_time.loc[float(t), state])
                except Exception:
                    return float(penalty), pd.DataFrame(solve_rows)
                residual = predicted - float(observed)
                total += 0.5 * (residual / sigma) ** 2

    if not np.isfinite(total):
        return float(penalty), pd.DataFrame(solve_rows)
    return float(total), pd.DataFrame(solve_rows)


def custom_pso_initial_positions(rng):
    pop_size = int(CUSTOM_PSO_CONFIG["pop_size"])
    n_parameters = len(CUSTOM_PSO_PARAMETERS)
    positions = rng.random((pop_size, n_parameters))

    nominal_theta = custom_pso_reference_theta()
    positions[0, :] = custom_pso_encode_theta(nominal_theta)

    if CUSTOM_PSO_INCLUDE_WSSE_SEED and pop_size > 1 and "theta_WSSE" in globals():
        seeded_theta = dict(nominal_theta)
        seeded_theta.update({name: float(theta_WSSE[name]) for name in CUSTOM_PSO_PARAMETERS if name in theta_WSSE})
        positions[1, :] = custom_pso_encode_theta(seeded_theta)

    return positions


def custom_pso_save_best(theta, objective, progress):
    CUSTOM_PSO_RESULTS_DIR.mkdir(exist_ok=True)
    theta = custom_pso_complete_theta(theta)
    pd.Series(theta, name=f"{CUSTOM_PSO_RESULT_PREFIX}_best_theta").to_csv(
        CUSTOM_PSO_RESULTS_DIR / f"{CUSTOM_PSO_RESULT_PREFIX}_best_theta.csv"
    )
    pd.Series({"objective": float(objective)}, name=f"{CUSTOM_PSO_RESULT_PREFIX}_best_objective").to_csv(
        CUSTOM_PSO_RESULTS_DIR / f"{CUSTOM_PSO_RESULT_PREFIX}_best_objective.csv"
    )
    if progress:
        pd.DataFrame(progress).to_csv(CUSTOM_PSO_RESULTS_DIR / f"{CUSTOM_PSO_RESULT_PREFIX}_progress.csv", index=False)


def run_custom_pso():
    rng = np.random.default_rng(int(CUSTOM_PSO_CONFIG["seed"]))
    n_epoch = int(CUSTOM_PSO_CONFIG["epoch"])
    pop_size = int(CUSTOM_PSO_CONFIG["pop_size"])
    n_parameters = len(CUSTOM_PSO_PARAMETERS)

    positions = custom_pso_initial_positions(rng)
    velocities = rng.uniform(-0.10, 0.10, size=(pop_size, n_parameters))
    personal_best_positions = positions.copy()
    personal_best_objectives = np.full(pop_size, np.inf, dtype=float)
    global_best_position = positions[0, :].copy()
    global_best_objective = np.inf
    global_best_theta = custom_pso_decode_position(global_best_position)

    progress_rows = []
    objective_cache = {}
    total_evaluations = 0
    total_cache_hits = 0
    total_successful_evaluations = 0
    total_failed_evaluations = 0
    last_failure_summary = ""
    no_improvement_checks = 0
    previous_checkpoint_best = np.inf

    print("Custom PSO batches:", CUSTOM_PSO_BATCHES)
    print("Custom PSO estimated parameters:", CUSTOM_PSO_PARAMETERS)
    print("Custom PSO fixed parameters:", sorted(CUSTOM_PSO_FIXED_PARAMETERS))
    print("Custom PSO config:", CUSTOM_PSO_CONFIG)

    for epoch in range(1, n_epoch + 1):
        for particle_idx in range(pop_size):
            position_signature = tuple(np.round(positions[particle_idx, :], 10))
            used_cache = False
            if position_signature in objective_cache:
                objective, theta, evaluation_failed, failure_summary = objective_cache[position_signature]
                total_cache_hits += 1
                used_cache = True
            else:
                theta = custom_pso_decode_position(positions[particle_idx, :])
                objective, solve_rows = custom_pso_weighted_sse(theta)
                evaluation_failed = (not np.isfinite(objective)) or objective >= CUSTOM_PSO_FAILED_OBJECTIVE
                failure_summary = custom_pso_failure_summary(solve_rows) if evaluation_failed else ""
                objective_cache[position_signature] = (objective, theta, evaluation_failed, failure_summary)
                total_evaluations += 1
                if evaluation_failed:
                    total_failed_evaluations += 1
                    last_failure_summary = failure_summary
                else:
                    total_successful_evaluations += 1

            if (
                CUSTOM_PSO_PRINT_PROGRESS
                and not used_cache
                and (
                    total_evaluations == 1
                    or total_evaluations % CUSTOM_PSO_PROGRESS_EVERY_EVALUATIONS == 0
                )
            ):
                current_display = "penalty" if evaluation_failed else f"{objective:.6g}"
                print(
                    f"[CUSTOM PSO] eval={total_evaluations} | epoch={epoch}/{n_epoch} "
                    f"| particle={particle_idx + 1}/{pop_size} | current={current_display} "
                    f"| best={global_best_objective:.6g} | ok={total_successful_evaluations} "
                    f"| failed={total_failed_evaluations} | cache={total_cache_hits}",
                    flush=True,
                )
                if last_failure_summary:
                    print(f"[CUSTOM PSO] last failed evaluation: {last_failure_summary}", flush=True)

            if objective < personal_best_objectives[particle_idx]:
                personal_best_objectives[particle_idx] = objective
                personal_best_positions[particle_idx, :] = positions[particle_idx, :].copy()

            if objective < global_best_objective:
                global_best_objective = float(objective)
                global_best_position = positions[particle_idx, :].copy()
                global_best_theta = custom_pso_complete_theta(theta)
                custom_pso_save_best(global_best_theta, global_best_objective, progress_rows)
                if CUSTOM_PSO_PRINT_PROGRESS:
                    print(
                        f"[CUSTOM PSO] new best | eval={total_evaluations} | epoch={epoch}/{n_epoch} "
                        f"| particle={particle_idx + 1}/{pop_size} | best={global_best_objective:.6g}",
                        flush=True,
                    )

        if epoch == 1 or epoch % CUSTOM_PSO_CHECK_INTERVAL == 0 or epoch == n_epoch:
            if np.isfinite(previous_checkpoint_best):
                rel_improvement = max(previous_checkpoint_best - global_best_objective, 0.0) / max(
                    abs(previous_checkpoint_best), 1.0
                )
            else:
                rel_improvement = np.inf

            checkpoint = {
                "epoch": epoch,
                "evaluations": total_evaluations,
                "cache_hits": total_cache_hits,
                "successful_evaluations": total_successful_evaluations,
                "failed_evaluations": total_failed_evaluations,
                "best_objective": float(global_best_objective),
                "relative_improvement_since_checkpoint": float(rel_improvement),
            }
            progress_rows.append(checkpoint)
            custom_pso_save_best(global_best_theta, global_best_objective, progress_rows)

            if CUSTOM_PSO_PRINT_PROGRESS or CUSTOM_PSO_CONFIG.get("verbose", False):
                print(
                    f"[CUSTOM PSO] checkpoint epoch {epoch}/{n_epoch} | evaluations={total_evaluations} "
                    f"| ok={total_successful_evaluations} | failed={total_failed_evaluations} "
                    f"| cache={total_cache_hits} | best={global_best_objective:.6g} "
                    f"| rel_improvement={rel_improvement:.3g}",
                    flush=True,
                )

            if np.isfinite(previous_checkpoint_best):
                if rel_improvement < float(CUSTOM_PSO_CONFIG["relative_gap_threshold"]):
                    no_improvement_checks += 1
                else:
                    no_improvement_checks = 0
                if no_improvement_checks >= CUSTOM_PSO_PATIENCE_CHECKS:
                    print(
                        "[CUSTOM PSO] Early stop: relative improvement stayed below "
                        f"{CUSTOM_PSO_CONFIG['relative_gap_threshold']} for "
                        f"{CUSTOM_PSO_PATIENCE_CHECKS} checkpoints.",
                        flush=True,
                    )
                    break
            previous_checkpoint_best = float(global_best_objective)

        r1 = rng.random((pop_size, n_parameters))
        r2 = rng.random((pop_size, n_parameters))
        velocities = (
            float(CUSTOM_PSO_CONFIG["w"]) * velocities
            + float(CUSTOM_PSO_CONFIG["c1"]) * r1 * (personal_best_positions - positions)
            + float(CUSTOM_PSO_CONFIG["c2"]) * r2 * (global_best_position - positions)
        )
        positions = np.clip(positions + velocities, 0.0, 1.0)

    result = {
        "best_objective": float(global_best_objective),
        "best_theta": global_best_theta,
        "progress": pd.DataFrame(progress_rows),
        "personal_best_objectives": pd.Series(personal_best_objectives, name="personal_best_objective"),
        "n_evaluations": total_evaluations,
        "n_cache_hits": total_cache_hits,
        "n_successful_evaluations": total_successful_evaluations,
        "n_failed_evaluations": total_failed_evaluations,
    }
    return result


custom_pso_results = {}
theta_WSSE_pso_best = None
obj_WSSE_pso_best = np.nan
theta_WSSE_pso_polished = None
obj_WSSE_pso_polished = np.nan

if RUN_CUSTOM_PSO:
    custom_pso_results = run_custom_pso()
    theta_WSSE_pso_best = custom_pso_results["best_theta"]
    obj_WSSE_pso_best = float(custom_pso_results["best_objective"])

    display(pd.Series(theta_WSSE_pso_best, name="theta_WSSE_pso_best"))
    display(pd.Series({
        "obj_WSSE_pso_best": obj_WSSE_pso_best,
        "n_evaluations": custom_pso_results["n_evaluations"],
        "n_successful_evaluations": custom_pso_results["n_successful_evaluations"],
        "n_failed_evaluations": custom_pso_results["n_failed_evaluations"],
        "n_cache_hits": custom_pso_results["n_cache_hits"],
    }))
    if not custom_pso_results["progress"].empty:
        display(custom_pso_results["progress"].tail(10))

    if CUSTOM_PSO_POLISH_WITH_IPOPT and np.isfinite(obj_WSSE_pso_best):
        print("Polishing best PSO point with ParmEst/Ipopt on CUSTOM_PSO_BATCHES...")
        with custom_pso_pyomo_warning_filter():
            obj_WSSE_pso_polished, theta_WSSE_pso_polished_raw, _ = run_parmest_estimation(
                "SSE_weighted",
                batch_ids=CUSTOM_PSO_BATCHES,
                theta_initial=theta_WSSE_pso_best,
                parameters_to_estimate=CUSTOM_PSO_PARAMETERS,
            )
        theta_WSSE_pso_polished, theta_WSSE_pso_polished_adjustments = clip_theta_to_bounds(
            theta_WSSE_pso_polished_raw
        )
        display(pd.Series(theta_WSSE_pso_polished, name="theta_WSSE_pso_polished"))
        print("Polished PSO/Ipopt objective:", obj_WSSE_pso_polished)
        if not theta_WSSE_pso_polished_adjustments.empty:
            display(theta_WSSE_pso_polished_adjustments)

        CUSTOM_PSO_RESULTS_DIR.mkdir(exist_ok=True)
        pd.Series(theta_WSSE_pso_polished, name="theta_WSSE_pso_polished").to_csv(
            CUSTOM_PSO_RESULTS_DIR / "custom_pso_polished_theta.csv"
        )
        pd.Series({"objective": float(obj_WSSE_pso_polished)}, name="custom_pso_polished_objective").to_csv(
            CUSTOM_PSO_RESULTS_DIR / "custom_pso_polished_objective.csv"
        )
else:
    print("Custom PSO deferred. The initial Gaussian WSSE fit below uses ParmEst directly; run post-profile PSO after profile-likelihood screening.")


### Gaussian WSSE local polish with ParmEst in effective-parameter space

Use the nominal/reference parameter vector as the initial point for the local `SSE_weighted` ParmEst/IPOPT solve. PSO outputs are intentionally ignored here so stale global-search results cannot silently seed the baseline fit used by covariance and profile-likelihood diagnostics.


In [ ]:
# Initial local ParmEst calibration. PSO is intentionally deferred until after profile-likelihood screening.
BASE_FIT_OBJECTIVE = "SSE_weighted"
BASE_FIT_LABEL = "WSSE_fit"
BASE_FIT_BATCHES = ESTIMABILITY_ANALYSIS_BATCHES if "ESTIMABILITY_ANALYSIS_BATCHES" in globals() else PARAMETER_ESTIMATION_BATCHES
BASE_FIT_FIXED_PARAMETERS = sorted(INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN) if "INITIAL_FIXED_PARAMETERS_FROM_NOMINAL_EIGEN" in globals() else []
BASE_FIT_PARAMETERS = [name for name in DEFAULT_THETA if name not in set(BASE_FIT_FIXED_PARAMETERS)]
BASE_FIT_FIXED_PARAMETERS = [name for name in DEFAULT_THETA if name not in BASE_FIT_PARAMETERS]
BASE_FIT_TEE = False  # Set True only when debugging the local Ipopt solve.
BASE_FIT_REFERENCE_BATCH = BASE_FIT_BATCHES[0] if len(BASE_FIT_BATCHES) else "25026"
BASE_FIT_CONTINUE_ON_LOCAL_FAILURE = False


base_fit_reference_batch = load_batch(BASE_FIT_REFERENCE_BATCH)
base_fit_nominal_theta = _theta_for_batch(base_fit_reference_batch, DEFAULT_THETA)
base_fit_initial_theta = dict(base_fit_nominal_theta)
if "BASE_FIT_INITIAL_THETA_OVERRIDE" in globals() and BASE_FIT_INITIAL_THETA_OVERRIDE is not None:
    base_fit_initial_theta = _theta_for_batch(
        base_fit_reference_batch,
        {**base_fit_initial_theta, **dict(BASE_FIT_INITIAL_THETA_OVERRIDE)},
    )
    base_fit_initial_source = "BASE_FIT_INITIAL_THETA_OVERRIDE"
else:
    base_fit_initial_source = "nominal_reference"

base_fit_status = "optimal"
base_fit_error = ""
pest_WSSE = None

print("Gaussian WSSE local polish batches:", BASE_FIT_BATCHES)
print("Gaussian WSSE local polish initial source:", base_fit_initial_source)
print("Gaussian WSSE local polish estimated parameters:", BASE_FIT_PARAMETERS)
print("Gaussian WSSE local polish fixed parameters:", BASE_FIT_FIXED_PARAMETERS)

try:
    obj_WSSE, theta_WSSE_raw, pest_WSSE = run_parmest_estimation(
        BASE_FIT_OBJECTIVE,
        batch_ids=BASE_FIT_BATCHES,
        tee=BASE_FIT_TEE,
        theta_initial=base_fit_initial_theta,
        parameters_to_estimate=BASE_FIT_PARAMETERS,
    )
except Exception as err:
    if not BASE_FIT_CONTINUE_ON_LOCAL_FAILURE:
        raise
    base_fit_status = f"fallback_{base_fit_initial_source}_after_local_failure"
    base_fit_error = f"{type(err).__name__}: {err}"
    print(
        "WARNING: Gaussian WSSE ParmEst local polish failed; "
        f"continuing with {base_fit_initial_source} theta."
    )
    print(base_fit_error)
    theta_WSSE_raw = dict(base_fit_initial_theta)
    fallback_objective_summary, _, fallback_solve_summary = evaluate_fit_objectives(
        theta_WSSE_raw,
        batch_ids=BASE_FIT_BATCHES,
    )
    obj_WSSE = float(fallback_objective_summary.get("WSSE_raw_sum", np.nan))
    if not fallback_solve_summary.empty:
        display(fallback_solve_summary)

theta_WSSE_complete = dict(base_fit_initial_theta)
theta_WSSE_complete.update(theta_WSSE_raw)
theta_WSSE, theta_WSSE_adjustments = clip_theta_to_bounds(theta_WSSE_complete)
theta_WSSE_full_fit = dict(theta_WSSE)
obj_WSSE_full_fit = float(obj_WSSE)
theta_WSSE_source = "full_data_fit_from_nominal"

base_fit_diagnostic = pd.Series(
    {
        "status": base_fit_status,
        "initial_source": base_fit_initial_source,
        "batches": ", ".join(map(str, BASE_FIT_BATCHES)),
        "estimated_parameters": ", ".join(BASE_FIT_PARAMETERS),
        "fixed_parameters": ", ".join(BASE_FIT_FIXED_PARAMETERS) or "none",
        "objective": obj_WSSE,
        "objective_source": "ParmEst" if base_fit_status == "optimal" else "direct_simulation_fallback",
        "error": base_fit_error,
    },
    name="base_fit_diagnostic",
)
display(base_fit_diagnostic)

theta_comparison_columns = {
    f"nominal_reference_{BASE_FIT_REFERENCE_BATCH}": base_fit_nominal_theta,
}
if base_fit_initial_source != "nominal_reference":
    theta_comparison_columns[base_fit_initial_source] = base_fit_initial_theta
theta_comparison_columns["WSSE"] = theta_WSSE

theta_comparison = pd.DataFrame(theta_comparison_columns)
display(theta_comparison)

print("Back-transformed physical-parameter view for the reference batch:")
theta_physical_comparison = pd.DataFrame(
    {
        label: physical_theta_from_effective(theta_values, batch=base_fit_reference_batch)
        for label, theta_values in theta_comparison_columns.items()
    }
)
display(theta_physical_comparison)
if not theta_WSSE_adjustments.empty:
    display(pd.concat({"WSSE": theta_WSSE_adjustments}, names=["fit", "row"]))
print("ParmEst/direct Gaussian WSSE objective:", obj_WSSE)


In [ ]:
def display_fit_diagnostics(theta, objective, label="WSSE_fit", heading=""):
    if heading:
        print(heading)
    fit_summaries = []
    residual_tables = {}
    solve_tables = {}
    objective_summary, residuals, solve_summary = evaluate_fit_objectives(theta, batch_ids=BASE_FIT_BATCHES)
    objective_summary["fit"] = label
    objective_summary["ParmEst_objective"] = objective
    objective_summary["batches"] = ", ".join(map(str, BASE_FIT_BATCHES))
    fit_summaries.append(objective_summary)
    residual_tables[label] = residuals
    solve_tables[label] = solve_summary

    fit_comparison = pd.DataFrame(fit_summaries).set_index("fit")
    display(fit_comparison[["batches", "n_observations", "ParmEst_objective", "SSE_raw_sum", "WSSE_raw_sum"]])
    print("Objective scaling mode:", OBJECTIVE_SCALE_MODE)
    print("Objective scaling scope:", OBJECTIVE_SCALE_SCOPE)
    print("Objective normalized by state observation count:", OBJECTIVE_NORMALIZE_BY_STATE_COUNT)
    print("Objective normalized by experiment state count:", OBJECTIVE_NORMALIZE_BY_EXPERIMENT_STATE_COUNT)
    print("Raw SSE is reported only as a diagnostic, not as the fitted objective.")
    print("ParmEst objective is reported on the same total-over-batches scale as the direct Gaussian WSSE diagnostics.")

    state_tables = {
        fit_label: fit_residuals.groupby("state")[["squared_error", "weighted_squared_error"]].sum()
        for fit_label, fit_residuals in residual_tables.items()
        if not fit_residuals.empty
    }
    state_fit_comparison = pd.concat(state_tables, names=["fit", "state"]) if state_tables else pd.DataFrame()
    if not state_fit_comparison.empty:
        display(state_fit_comparison)
    if solve_tables:
        display(pd.concat(solve_tables, names=["fit", "batch"]))
    return fit_comparison, residual_tables, solve_tables, state_fit_comparison


pre_cv_fit_comparison, pre_cv_residual_tables, pre_cv_solve_tables, pre_cv_state_fit_comparison = display_fit_diagnostics(
    theta_WSSE,
    obj_WSSE,
    heading="Full-data fit diagnostics before optional CV.",
)

RUN_LEAVE_ONE_BATCH_OUT_CV = globals().get("RUN_LEAVE_ONE_BATCH_OUT_CV", False)
CV_CONTINUE_ON_FAILURE = globals().get("CV_CONTINUE_ON_FAILURE", False)

cv_results = None
if RUN_LEAVE_ONE_BATCH_OUT_CV:
    print("Leave-one-experiment-out CV batches:", BASE_FIT_BATCHES)
    cv_results = run_leave_one_batch_out_cv(
        BASE_FIT_OBJECTIVE,
        batch_ids=BASE_FIT_BATCHES,
        theta_initial=base_fit_initial_theta,
        parameters_to_estimate=BASE_FIT_PARAMETERS,
        tee=BASE_FIT_TEE,
        continue_on_failure=CV_CONTINUE_ON_FAILURE,
    )
    display(cv_results["summary"])
    display(cv_results["fold_summary"])
    if not cv_results["theta_by_fold"].empty:
        display(cv_results["theta_by_fold"])

    REFIT_FULL_DATA_FROM_CV_MEAN = globals().get("REFIT_FULL_DATA_FROM_CV_MEAN", True)
    theta_CV_fold_mean_raw = {
        name: float(cv_results["theta_by_fold"][name].mean())
        for name in DEFAULT_THETA
        if name in cv_results["theta_by_fold"].columns
    }
    theta_CV_fold_mean_complete = _theta_for_batch(base_fit_reference_batch, DEFAULT_THETA)
    theta_CV_fold_mean_complete.update(theta_CV_fold_mean_raw)
    theta_CV_fold_mean, theta_CV_fold_mean_adjustments = clip_theta_to_bounds(theta_CV_fold_mean_complete)
    cv_mean_objective_summary, cv_mean_residuals, cv_mean_solve_summary = evaluate_fit_objectives(
        theta_CV_fold_mean,
        batch_ids=BASE_FIT_BATCHES,
    )
    obj_CV_fold_mean = float(cv_mean_objective_summary.get("WSSE_raw_sum", np.nan))
    display(pd.Series(theta_CV_fold_mean, name="theta_CV_fold_mean"))
    display(pd.Series(physical_theta_from_effective(theta_CV_fold_mean, batch=base_fit_reference_batch), name="theta_CV_fold_mean_physical_view"))
    display(
        pd.Series(
            {
                "obj_pre_cv_full_fit": float(obj_WSSE_full_fit),
                "obj_CV_fold_mean_on_all_batches": obj_CV_fold_mean,
                "cv_validation_objective_mean": float(cv_results["summary"]["cv_validation_objective_mean"]),
                "refit_full_data_from_cv_mean": bool(REFIT_FULL_DATA_FROM_CV_MEAN),
            },
            name="cv_mean_theta_summary",
        )
    )
    if not theta_CV_fold_mean_adjustments.empty:
        display(theta_CV_fold_mean_adjustments)
    if not cv_mean_solve_summary.empty:
        display(cv_mean_solve_summary)

    theta_WSSE_pre_cv_full_fit = dict(theta_WSSE_full_fit)
    obj_WSSE_pre_cv_full_fit = float(obj_WSSE_full_fit)
    if REFIT_FULL_DATA_FROM_CV_MEAN:
        print("Refitting full-data Gaussian WSSE from theta_CV_fold_mean...")
        obj_WSSE_refit, theta_WSSE_refit_raw, pest_WSSE_refit = run_parmest_estimation(
            BASE_FIT_OBJECTIVE,
            batch_ids=BASE_FIT_BATCHES,
            tee=BASE_FIT_TEE,
            theta_initial=theta_CV_fold_mean,
            parameters_to_estimate=BASE_FIT_PARAMETERS,
        )
        theta_WSSE_refit_complete = dict(theta_CV_fold_mean)
        theta_WSSE_refit_complete.update(theta_WSSE_refit_raw)
        theta_WSSE_refit, theta_WSSE_refit_adjustments = clip_theta_to_bounds(theta_WSSE_refit_complete)
        refit_objective_summary, refit_residuals, refit_solve_summary = evaluate_fit_objectives(
            theta_WSSE_refit,
            batch_ids=BASE_FIT_BATCHES,
        )
        obj_WSSE_refit_direct = float(refit_objective_summary.get("WSSE_raw_sum", np.nan))
        theta_WSSE = dict(theta_WSSE_refit)
        obj_WSSE = obj_WSSE_refit_direct if np.isfinite(obj_WSSE_refit_direct) else float(obj_WSSE_refit)
        pest_WSSE = pest_WSSE_refit
        theta_WSSE_full_fit = dict(theta_WSSE)
        obj_WSSE_full_fit = float(obj_WSSE)
        theta_WSSE_source = "full_data_fit_from_cv_fold_mean"
        display(pd.Series(theta_WSSE, name="theta_WSSE_full_data_refit_from_cv_mean"))
        display(pd.Series(physical_theta_from_effective(theta_WSSE, batch=base_fit_reference_batch), name="theta_WSSE_full_data_refit_physical_view"))
        display(
            pd.Series(
                {
                    "obj_pre_cv_full_fit": obj_WSSE_pre_cv_full_fit,
                    "obj_cv_fold_mean_on_all_batches": obj_CV_fold_mean,
                    "obj_refit_parmest": float(obj_WSSE_refit),
                    "obj_refit_direct": obj_WSSE_refit_direct,
                    "theta_WSSE_source": theta_WSSE_source,
                },
                name="cv_initialized_full_data_refit_summary",
            )
        )
        if not theta_WSSE_refit_adjustments.empty:
            display(theta_WSSE_refit_adjustments)
        if not refit_solve_summary.empty:
            display(refit_solve_summary)
    else:
        print("Full-data refit from theta_CV_fold_mean skipped; keeping pre-CV full-data fit.")

    theta_post_optimization_default = dict(theta_WSSE)
    obj_post_optimization_default = float(obj_WSSE)
    theta_post_optimization_source = theta_WSSE_source
    print("Post-optimization, FIM/covariance, and profile likelihood will use the final full-data theta_WSSE.")
else:
    theta_post_optimization_default = dict(theta_WSSE)
    obj_post_optimization_default = float(obj_WSSE)
    theta_post_optimization_source = theta_WSSE_source
    print("Leave-one-experiment-out CV skipped.")

fit_comparison, residual_tables, solve_tables, state_fit_comparison = display_fit_diagnostics(
    theta_WSSE,
    obj_WSSE,
    heading="Final full-data fit diagnostics for downstream estimability.",
)
display(
    pd.Series(
        {
            "theta_WSSE_source": theta_WSSE_source,
            "obj_WSSE": float(obj_WSSE),
            "theta_post_optimization_source": theta_post_optimization_source,
            "obj_post_optimization": float(obj_post_optimization_default),
        },
        name="final_theta_source_summary",
    )
)

print("Final back-transformed physical-parameter view for the reference batch:")
display(pd.Series(physical_theta_from_effective(theta_WSSE, batch=base_fit_reference_batch), name="theta_WSSE_physical_view"))


## Post-fit sensitivity eigen-analysis

Repeat the local Q/FIM eigen-analysis around the working WSSE parameter vector after PSO and local ParmEst polishing. This block uses the same fixed parameter set selected after the nominal eigen-analysis, so downstream covariance and profile likelihood are evaluated on the reduced parameter set.

In [ ]:
RUN_POSTFIT_Q_EIGEN_ANALYSIS = True

ANALYSIS_BATCHES = BASE_FIT_BATCHES if "BASE_FIT_BATCHES" in globals() else ESTIMABILITY_ANALYSIS_BATCHES
POSTFIT_Q_BATCHES = ANALYSIS_BATCHES
POSTFIT_Q_FIXED_PARAMETERS = set(BASE_FIT_FIXED_PARAMETERS) if "BASE_FIT_FIXED_PARAMETERS" in globals() else set()
POSTFIT_Q_PARAMETERS = [name for name in DEFAULT_THETA if name not in POSTFIT_Q_FIXED_PARAMETERS]
POSTFIT_Q_OUTPUT_STATES = list(STATE_LABELS)

# Canonical names reused by covariance/profile-likelihood sections.
Q_ANALYSIS_LABEL = "WSSE_fit"
Q_ANALYSIS_BATCHES = POSTFIT_Q_BATCHES
Q_EXCLUDED_PARAMETERS = POSTFIT_Q_FIXED_PARAMETERS
Q_ANALYSIS_PARAMETERS = POSTFIT_Q_PARAMETERS
Q_ANALYSIS_OUTPUT_STATES = POSTFIT_Q_OUTPUT_STATES
Q_REFERENCE_BATCH = POSTFIT_Q_BATCHES[0] if len(POSTFIT_Q_BATCHES) else "25026"

if RUN_POSTFIT_Q_EIGEN_ANALYSIS:
    if "theta_WSSE" not in globals():
        raise RuntimeError("Run the Gaussian WSSE local polish block before post-fit Q/eigen-analysis.")
    print("Post-fit Q/FIM theta source:", globals().get("theta_WSSE_source", "theta_WSSE"))
    def get_q_analysis_theta(fit_label=Q_ANALYSIS_LABEL):
        theta = dict(DEFAULT_THETA)
        theta.update(theta_WSSE)
        theta_clipped, _ = clip_theta_to_bounds(theta)
        return {name: float(theta_clipped[name]) for name in Q_ANALYSIS_PARAMETERS}

    q_analysis_theta = get_q_analysis_theta()
    display(pd.Series(q_analysis_theta, name="postfit_q_analysis_theta"))
    print("Post-fit Q analysis batches:", Q_ANALYSIS_BATCHES)
    print("Post-fit Q fixed parameters:", sorted(Q_EXCLUDED_PARAMETERS))
    print("Post-fit Q estimated parameters:", Q_ANALYSIS_PARAMETERS)
    print("Q perturbation fraction:", Q_PERTURBATION_FRACTION)

    q_raw, q_weighted, q_relative, q_perturbation_summary, q_solve_summary = build_weighted_relative_sensitivity_matrix(
        q_analysis_theta,
        Q_ANALYSIS_PARAMETERS,
        Q_ANALYSIS_BATCHES,
        fraction=Q_PERTURBATION_FRACTION,
    )
    q_effective_parameters = list(q_relative.columns)
    q_dropped_parameters = [name for name in Q_ANALYSIS_PARAMETERS if name not in q_effective_parameters]
    if q_dropped_parameters:
        print("Post-fit Q/eigen excluded parameters:", q_dropped_parameters)
    fim_relative, q_eigen_summary, q_loading_summary = eigen_analysis_from_q(q_relative, q_effective_parameters)
    fim_weighted_unscaled, q_eigen_summary_unscaled, q_loading_summary_unscaled = eigen_analysis_from_q(
        q_weighted,
        q_effective_parameters,
    )
    q_diagnostic_summary = pd.Series(
        {
            "n_measurement_rows_total": int(q_raw.shape[0]),
            "n_measurement_rows_complete": int(q_relative.shape[0]),
            "n_parameters": int(q_relative.shape[1]),
            "effective_parameters": ", ".join(q_effective_parameters),
            "dropped_parameters": ", ".join(q_dropped_parameters) or "none",
            "relative_condition_number": float(q_eigen_summary["condition_number"].iloc[0]),
            "unscaled_condition_number": float(q_eigen_summary_unscaled["condition_number"].iloc[0]),
        },
        name="postfit_q_eigen_diagnostic",
    )
    display(q_diagnostic_summary)
    display(q_perturbation_summary)
    if not q_solve_summary.empty:
        display(q_solve_summary.groupby(["parameter", "side", "status", "termination"]).size().rename("count").reset_index())
    print("Post-fit eigen-analysis of weighted, relative-scaled Q:")
    display(q_eigen_summary)
    display(q_loading_summary)
    print("Post-fit weakest directions only:")
    display(q_loading_summary.head(min(5, len(q_loading_summary))))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].semilogy(q_eigen_summary["direction"], q_eigen_summary["fim_eigenvalue"].clip(lower=np.finfo(float).tiny), marker="o")
    axes[0].set_title("Post-fit FIM eigenvalues")
    axes[0].set_xlabel("Eigen-direction (weak to strong)")
    axes[0].set_ylabel("Eigenvalue (log scale)")
    axes[0].grid(True, which="both", alpha=0.3)
    axes[1].semilogy(q_eigen_summary["direction"], q_eigen_summary["relative_eigenvalue"].clip(lower=np.finfo(float).tiny), marker="o", color="tab:orange")
    axes[1].axhline(Q_EIGEN_NEAR_NULL_RTOL, color="tab:red", linestyle="--", label="near-null tolerance")
    axes[1].set_title("Post-fit relative eigenvalues")
    axes[1].set_xlabel("Eigen-direction (weak to strong)")
    axes[1].set_ylabel("Eigenvalue / max eigenvalue")
    axes[1].grid(True, which="both", alpha=0.3)
    axes[1].legend(loc="best")
    fig.tight_layout()
    plt.show()

    Q_ANALYSIS_RESULTS_DIR.mkdir(exist_ok=True)
    q_raw.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_raw_wsse_fit_pm10.csv")
    q_weighted.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_weighted_wsse_fit_pm10.csv")
    q_relative.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_weighted_relative_wsse_fit_pm10.csv")
    fim_relative.to_csv(Q_ANALYSIS_RESULTS_DIR / "fim_weighted_relative_wsse_fit_pm10.csv")
    q_eigen_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "fim_weighted_relative_eigenvalues_wsse_fit_pm10.csv", index=False)
    q_loading_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "fim_weighted_relative_eigendirections_wsse_fit_pm10.csv", index=False)
    q_perturbation_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_perturbation_summary_wsse_fit_pm10.csv")
    q_solve_summary.to_csv(Q_ANALYSIS_RESULTS_DIR / "q_sensitivity_solve_summary_wsse_fit_pm10.csv", index=False)
else:
    print("Post-fit Q/eigen-analysis skipped. Set RUN_POSTFIT_Q_EIGEN_ANALYSIS = True to execute it.")


## Post-optimization simulation

The fitted objectives are scalar summaries, so the next diagnostic is to simulate with the estimated parameter sets and inspect trajectories against the measurements. The default batch below is `25026`; change `POST_OPTIMIZATION_BATCH` to inspect any calibration batch.

In [ ]:
POST_OPTIMIZATION_BATCH = "25026"


def simulate_post_optimization_batch(batch_id=POST_OPTIMIZATION_BATCH, parameter_sets=None):
    if parameter_sets is None:
        post_theta = globals().get("theta_post_optimization_default", theta_WSSE)
        post_source = globals().get("theta_post_optimization_source", globals().get("theta_WSSE_source", "theta_WSSE"))
        post_label = "CV_fold_mean" if post_source == "cv_fold_mean" else "WSSE_fit"
        parameter_sets = {
            "nominal_reference": None,
            post_label: post_theta,
        }
        if post_source == "cv_fold_mean" and "theta_WSSE_full_fit" in globals():
            parameter_sets["full_data_fit"] = theta_WSSE_full_fit
        if "theta_WSSE_pso_best" in globals() and theta_WSSE_pso_best is not None:
            parameter_sets["WSSE_PSO_best"] = theta_WSSE_pso_best
        if "theta_WSSE_pso_polished" in globals() and theta_WSSE_pso_polished is not None:
            parameter_sets["WSSE_PSO_polished"] = theta_WSSE_pso_polished

    batch_post = load_batch(batch_id)
    simulations = {}
    solve_rows = []
    for label, theta in parameter_sets.items():
        try:
            model_post, result_post, sim_post = simulate_batch(batch_post, theta_initial=theta)
            simulations[label] = sim_post
            solve_rows.append(
                {
                    "parameter_set": label,
                    "status": str(result_post.solver.status),
                    "termination": str(result_post.solver.termination_condition),
                    "final_X": sim_post.iloc[-1]["X"],
                    "final_N": sim_post.iloc[-1]["N"],
                    "final_G": sim_post.iloc[-1]["G"],
                    "final_F": sim_post.iloc[-1]["F"],
                    "final_E": sim_post.iloc[-1]["E"],
                }
            )
        except Exception as err:
            solve_rows.append({"parameter_set": label, "status": "failed", "termination": repr(err)})
    return batch_post, simulations, pd.DataFrame(solve_rows).set_index("parameter_set")


def summarize_post_optimization_residuals(batch_post: FermentationBatch, simulations: dict):
    objective_scale = objective_measurement_error_for_batches([batch_post.batch_id])
    rows = []
    for label, sim_post in simulations.items():
        sim_by_time = sim_post.set_index("t")
        for state in STATE_LABELS:
            sigma = float(objective_scale[state])
            for t, observed in batch_post.measurements[state].dropna().items():
                predicted = float(sim_by_time.loc[float(t), state])
                residual = predicted - float(observed)
                rows.append(
                    {
                        "parameter_set": label,
                        "state": state,
                        "objective_scale": sigma,
                        "squared_error": residual**2,
                        "weighted_squared_error": 0.5 * (residual / sigma) ** 2,
                    }
                )
    residuals = pd.DataFrame(rows)
    if residuals.empty:
        return residuals
    return residuals.groupby(["parameter_set", "state"])[["squared_error", "weighted_squared_error"]].sum()


def plot_post_optimization_simulations(batch_post: FermentationBatch, simulations: dict):
    fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True)
    axes = axes.ravel()
    colors = {
        "nominal_reference": "tab:gray",
        "WSSE_fit": "tab:orange",
        "CV_fold_mean": "tab:orange",
        "full_data_fit": "tab:cyan",
        "WSSE_PSO_best": "tab:blue",
        "WSSE_PSO_polished": "tab:purple",
    }

    for ax, state in zip(axes[:5], STATE_LABELS):
        measured = batch_post.measurements[state].dropna()
        ax.scatter(measured.index, measured.values, color="black", s=28, alpha=0.75, label="measured")
        for label, sim_post in simulations.items():
            ax.plot(sim_post["t"], sim_post[state], color=colors.get(label), label=label)
        ax.set_title(STATE_LABELS[state])
        ax.set_ylabel("kg/m3")
        ax.grid(True, alpha=0.3)

    input_ax = axes[-1]
    pulse_ax = input_ax.twinx()
    temp_line = input_ax.plot(batch_post.time, batch_post.temperature_c, color="tab:red", label="temperature")
    pulse_bars = pulse_ax.bar(
        batch_post.time,
        batch_post.nutrient_pulse_kg_m3 / MG_L_TO_KG_M3,
        width=2.0,
        alpha=0.35,
        color="tab:green",
        label="YAN pulse",
    )
    input_ax.set_title("Measured inputs")
    input_ax.set_ylabel("Temperature (deg C)")
    pulse_ax.set_ylabel("YAN pulse (ppm)")
    input_ax.set_xlabel("Time (h)")
    input_ax.grid(True, alpha=0.3)
    input_ax.legend([temp_line[0], pulse_bars], ["temperature", "YAN pulse"], loc="best")

    axes[0].legend(ncol=2)
    fig.suptitle(f"Post-optimization simulation - batch {batch_post.batch_id}")
    fig.tight_layout()
    return fig, axes


display(
    pd.Series(
        {
            "theta_WSSE_source": globals().get("theta_WSSE_source", "unknown"),
            "obj_WSSE": float(obj_WSSE) if "obj_WSSE" in globals() else np.nan,
            "theta_post_optimization_source": globals().get("theta_post_optimization_source", globals().get("theta_WSSE_source", "unknown")),
            "obj_post_optimization": float(globals().get("obj_post_optimization_default", obj_WSSE if "obj_WSSE" in globals() else np.nan)),
            "post_optimization_batch": POST_OPTIMIZATION_BATCH,
        },
        name="post_optimization_theta_source",
    )
)
post_batch, post_simulations, post_solve_summary = simulate_post_optimization_batch(POST_OPTIMIZATION_BATCH)
display(post_solve_summary)
display(summarize_post_optimization_residuals(post_batch, post_simulations))
plot_post_optimization_simulations(post_batch, post_simulations);


## Parameter uncertainty quantification in effective-parameter space

This block follows `parmest_uncertainty_quantification.ipynb`: after parameter estimation, ParmEst computes covariance matrices and parameter standard deviations from the Fisher information matrix. The three ParmEst covariance methods are attempted: finite differences, reduced Hessian, and automatic differentiation with k_aug. Parameters estimated exactly on a bound are reported separately and excluded from the covariance calculation, because the usual unconstrained covariance interpretation is not valid for active-bound estimates and central finite differences would step outside the allowed domain. The parameter estimates are still obtained from all calibration experiments; covariance can be evaluated on a solver-stable subset when some batches fail under finite-difference perturbations or k_aug sensitivity solves.

In [ ]:
UQ_FITS = {
    "WSSE_fit": ("SSE_weighted", theta_WSSE),
}
print("UQ/covariance theta source:", globals().get("theta_WSSE_source", "theta_WSSE"))
UQ_FIXED_PARAMETERS = set(BASE_FIT_FIXED_PARAMETERS) if "BASE_FIT_FIXED_PARAMETERS" in globals() else set()
# Keep notebook execution robust on Windows/IDAES: the reduced_hessian and
# automatic_differentiation_kaug paths can terminate the kernel in some
# Pynumero/k_aug builds. Add them back here only when that toolchain is verified.
UQ_METHODS = [
    "finite_difference",
]
UQ_STEP = 1e-4
UQ_EXCLUDED_BATCHES_FOR_COVARIANCE = {"25085", "25171"}
UQ_COVARIANCE_SOURCE_BATCHES = BASE_FIT_BATCHES if "BASE_FIT_BATCHES" in globals() else PARAMETER_ESTIMATION_BATCHES
UQ_COVARIANCE_BATCHES = [
    batch_id for batch_id in UQ_COVARIANCE_SOURCE_BATCHES if batch_id not in UQ_EXCLUDED_BATCHES_FOR_COVARIANCE
]
DISPLAY_COVARIANCE_MATRICES = False
UQ_REL_STD_WARNING_PCT = 30.0
UQ_STD_RATIO_WARNING = 2.0
ACTIVE_BOUND_RTOL = 1e-4
ACTIVE_BOUND_ATOL = 1e-8
REDUCED_HESSIAN_MAX_RETRIES = 4


def identify_active_bound_parameters(theta, rtol=ACTIVE_BOUND_RTOL, atol=ACTIVE_BOUND_ATOL):
    rows = []
    for name, value in theta.items():
        value = float(value)
        lb, ub = PARAMETER_BOUNDS[name]
        scale = max(abs(value), abs(lb), abs(ub), 1.0)
        near_lower = abs(value - lb) <= atol + rtol * scale
        near_upper = abs(value - ub) <= atol + rtol * scale
        if near_lower and near_upper:
            status = "fixed_interval"
        elif near_lower:
            status = "active_lower_bound"
        elif near_upper:
            status = "active_upper_bound"
        else:
            status = "free_for_covariance"
        rows.append(
            {
                "parameter": name,
                "estimate": value,
                "lower_bound": lb,
                "upper_bound": ub,
                "bound_status": status,
            }
        )
    return pd.DataFrame(rows).set_index("parameter")


def covariance_diagnostics(cov: pd.DataFrame, theta: dict, fit_label: str, method: str):
    cov = cov.astype(float)
    variances = pd.Series(np.diag(cov), index=cov.index, name="variance")
    std_dev = variances.where(variances >= 0.0).pow(0.5).rename("std_dev")
    estimates = pd.Series({name: float(theta[name]) for name in cov.index}, name="estimate")
    rel_std = (100.0 * std_dev / estimates.abs().replace(0.0, np.nan)).rename("relative_std_pct")
    summary = pd.concat([estimates, variances, std_dev, rel_std], axis=1)
    cov_array = cov.to_numpy(dtype=float)
    cov_symmetric = 0.5 * (cov_array + cov_array.T)
    eigvals = np.linalg.eigvalsh(cov_symmetric)
    summary["fit"] = fit_label
    summary["method"] = method
    diagnostics = {
        "fit": fit_label,
        "method": method,
        "n_parameters": len(cov),
        "trace": float(np.trace(cov_array)),
        "min_eigenvalue": float(np.min(eigvals)),
        "max_eigenvalue": float(np.max(eigvals)),
        "is_psd": bool(np.min(eigvals) >= -1e-10),
        "condition_number": float(np.linalg.cond(cov_array)),
    }
    return summary, diagnostics


def covariance_parameter_names(bound_summary: pd.DataFrame):
    free_names = bound_summary.index[bound_summary["bound_status"].eq("free_for_covariance")].to_list()
    return [name for name in free_names if name not in UQ_FIXED_PARAMETERS]


def build_covariance_estimator(obj_function: str, theta: dict, covariance_parameters: list[str]):
    experiments = build_parmest_experiments(
        batch_ids=UQ_COVARIANCE_BATCHES,
        parameters_to_estimate=covariance_parameters,
        theta_initial=theta,
    )
    estimator = parmest.Estimator(
        experiments,
        obj_function=obj_function,
        tee=False,
        solver_options=PARMEST_SOLVER_OPTIONS,
    )
    estimator.estimated_theta = {name: float(theta[name]) for name in covariance_parameters}
    estimator.covariance_objective = parmest.SSE_weighted if obj_function == "SSE_weighted" else parmest.SSE
    return estimator


def covariance_method_unavailable_reason(method: str):
    if method == "automatic_differentiation_kaug":
        missing = [tool for tool in ["k_aug", "dot_sens"] if shutil.which(tool) is None]
        if missing:
            return f"Missing executable(s): {', '.join(missing)}"
    return None


def run_reduced_hessian_covariance(
    fit_label: str,
    obj_function: str,
    theta: dict,
    covariance_parameters: list[str],
    step=UQ_STEP,
):
    working_parameters = list(covariance_parameters)
    theta_current = {name: float(value) for name, value in theta.items()}
    removed_parameters = []
    max_abs_theta_shift = 0.0

    for attempt in range(1, REDUCED_HESSIAN_MAX_RETRIES + 1):
        if len(working_parameters) == 0:
            diagnostics = {
                "fit": fit_label,
                "method": "reduced_hessian",
                "n_parameters": 0,
                "max_abs_theta_shift_from_fit": max_abs_theta_shift,
                "reduced_hessian_attempts": attempt,
                "reduced_hessian_removed_parameters": ", ".join(removed_parameters),
                "error": "All candidate parameters moved to active bounds during reduced-Hessian retries.",
            }
            return None, pd.DataFrame(), diagnostics

        estimator = build_covariance_estimator(obj_function, theta_current, working_parameters)
        try:
            _, theta_reestimated = estimator.theta_est()
            theta_for_summary = {name: float(theta_reestimated[name]) for name in working_parameters}
            max_abs_theta_shift = max(
                max_abs_theta_shift,
                max(abs(theta_for_summary[name] - float(theta[name])) for name in working_parameters),
            )

            reestimated_bound_summary = identify_active_bound_parameters(theta_for_summary)
            newly_active = reestimated_bound_summary.index[
                ~reestimated_bound_summary["bound_status"].eq("free_for_covariance")
            ].to_list()
            if newly_active:
                removed_parameters.extend(newly_active)
                theta_current.update(theta_for_summary)
                working_parameters = [name for name in working_parameters if name not in newly_active]
                continue

            estimator.estimated_theta = theta_for_summary
            cov = estimator.cov_est(method="reduced_hessian", solver="ipopt", step=float(step))
            uncertainty_summary, diagnostics = covariance_diagnostics(cov, theta_for_summary, fit_label, "reduced_hessian")
            diagnostics["max_abs_theta_shift_from_fit"] = max_abs_theta_shift
            diagnostics["reduced_hessian_attempts"] = attempt
            diagnostics["reduced_hessian_removed_parameters"] = ", ".join(removed_parameters)
            return cov, uncertainty_summary, diagnostics
        except Exception as err:
            diagnostics = {
                "fit": fit_label,
                "method": "reduced_hessian",
                "n_parameters": len(working_parameters),
                "max_abs_theta_shift_from_fit": max_abs_theta_shift,
                "reduced_hessian_attempts": attempt,
                "reduced_hessian_removed_parameters": ", ".join(removed_parameters),
                "error": f"{type(err).__name__}: {err}",
            }
            return None, pd.DataFrame(), diagnostics

    diagnostics = {
        "fit": fit_label,
        "method": "reduced_hessian",
        "n_parameters": len(working_parameters),
        "max_abs_theta_shift_from_fit": max_abs_theta_shift,
        "reduced_hessian_attempts": REDUCED_HESSIAN_MAX_RETRIES,
        "reduced_hessian_removed_parameters": ", ".join(removed_parameters),
        "error": "Reduced-Hessian retry limit reached.",
    }
    return None, pd.DataFrame(), diagnostics


def run_covariance_estimation(
    fit_label: str,
    obj_function: str,
    theta: dict,
    covariance_parameters: list[str],
    method: str,
    step=UQ_STEP,
):
    if len(covariance_parameters) == 0:
        return None, pd.DataFrame(), {"fit": fit_label, "method": method, "error": "No free parameters."}

    unavailable_reason = covariance_method_unavailable_reason(method)
    if unavailable_reason is not None:
        diagnostics = {
            "fit": fit_label,
            "method": method,
            "n_parameters": len(covariance_parameters),
            "max_abs_theta_shift_from_fit": 0.0,
            "error": unavailable_reason,
        }
        return None, pd.DataFrame(), diagnostics

    if method == "reduced_hessian":
        return run_reduced_hessian_covariance(fit_label, obj_function, theta, covariance_parameters, step)

    estimator = build_covariance_estimator(obj_function, theta, covariance_parameters)
    theta_reference = {name: float(theta[name]) for name in covariance_parameters}
    try:
        cov = estimator.cov_est(method=method, solver="ipopt", step=float(step))
        uncertainty_summary, diagnostics = covariance_diagnostics(cov, theta_reference, fit_label, method)
        diagnostics["max_abs_theta_shift_from_fit"] = 0.0
        return cov, uncertainty_summary, diagnostics
    except Exception as err:
        diagnostics = {
            "fit": fit_label,
            "method": method,
            "n_parameters": len(covariance_parameters),
            "max_abs_theta_shift_from_fit": 0.0,
            "error": f"{type(err).__name__}: {err}",
        }
        return None, pd.DataFrame(), diagnostics


def build_method_comparison_table(covariance_results: dict, uncertainty_tables: dict):
    rows = []
    fit_labels = sorted({fit_label for fit_label, _ in covariance_results.keys()})
    for fit_label in fit_labels:
        available_methods = [
            method
            for method in UQ_METHODS
            if covariance_results.get((fit_label, method)) is not None
        ]
        if len(available_methods) < 2:
            continue
        baseline_method = "finite_difference" if "finite_difference" in available_methods else available_methods[0]
        base_cov = covariance_results[(fit_label, baseline_method)]
        base_uncertainty = uncertainty_tables[(fit_label, baseline_method)]
        for method in available_methods:
            if method == baseline_method:
                continue
            cov = covariance_results[(fit_label, method)]
            uncertainty = uncertainty_tables[(fit_label, method)]
            common_parameters = base_cov.index.intersection(cov.index)
            base_array = base_cov.loc[common_parameters, common_parameters].to_numpy(dtype=float)
            method_array = cov.loc[common_parameters, common_parameters].to_numpy(dtype=float)
            base_norm = max(np.linalg.norm(base_array, ord="fro"), np.finfo(float).eps)
            trace_base = float(np.trace(base_array))
            trace_method = float(np.trace(method_array))
            std_base = base_uncertainty.loc[common_parameters, "std_dev"].astype(float)
            std_method = uncertainty.loc[common_parameters, "std_dev"].astype(float)
            std_relative_delta = (std_method - std_base).abs() / std_base.abs().replace(0.0, np.nan)
            rows.append(
                {
                    "fit": fit_label,
                    "baseline_method": baseline_method,
                    "comparison_method": method,
                    "n_common_parameters": len(common_parameters),
                    "relative_frobenius_diff": float(np.linalg.norm(method_array - base_array, ord="fro") / base_norm),
                    "trace_baseline": trace_base,
                    "trace_comparison": trace_method,
                    "trace_ratio": trace_method / trace_base if trace_base != 0 else np.nan,
                    "max_std_relative_delta_pct": float(100.0 * std_relative_delta.max()),
                    "median_std_relative_delta_pct": float(100.0 * std_relative_delta.median()),
                }
            )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index(["fit", "baseline_method", "comparison_method"])


def build_parameter_method_variability(uncertainty_tables: dict):
    pieces = []
    for (fit_label, method), table in uncertainty_tables.items():
        if table.empty:
            continue
        piece = table[["estimate", "std_dev", "relative_std_pct"]].copy().reset_index()
        piece = piece.rename(columns={piece.columns[0]: "parameter"})
        piece["fit"] = fit_label
        piece["method"] = method
        pieces.append(piece)
    if not pieces:
        return pd.DataFrame(), pd.DataFrame()

    all_uncertainty = pd.concat(pieces, ignore_index=True)
    summary = all_uncertainty.groupby(["fit", "parameter"]).agg(
        n_methods=("method", "nunique"),
        methods=("method", lambda values: ", ".join(sorted(values))),
        estimate_min=("estimate", "min"),
        estimate_max=("estimate", "max"),
        std_dev_min=("std_dev", "min"),
        std_dev_max=("std_dev", "max"),
        relative_std_pct_min=("relative_std_pct", "min"),
        relative_std_pct_max=("relative_std_pct", "max"),
    )
    summary["std_dev_ratio_max_min"] = summary["std_dev_max"] / summary["std_dev_min"].replace(0.0, np.nan)
    summary["problem_flag"] = (
        (summary["relative_std_pct_max"] >= UQ_REL_STD_WARNING_PCT)
        | (summary["std_dev_ratio_max_min"] >= UQ_STD_RATIO_WARNING)
    )
    summary = summary.sort_values(
        ["problem_flag", "relative_std_pct_max", "std_dev_ratio_max_min"],
        ascending=[False, False, False],
    )
    return all_uncertainty, summary


def build_problem_parameter_diagnostics(bound_tables: dict, uncertainty_comparison: pd.DataFrame):
    rows = []
    for fit_label, bound_table in bound_tables.items():
        active = bound_table[~bound_table["bound_status"].eq("free_for_covariance")]
        for parameter, row in active.iterrows():
            rows.append(
                {
                    "fit": fit_label,
                    "parameter": parameter,
                    "issue": "active_bound",
                    "detail": row["bound_status"],
                    "estimate": row["estimate"],
                    "relative_std_pct_max": np.nan,
                    "std_dev_ratio_max_min": np.nan,
                    "methods": "",
                }
            )

    if not uncertainty_comparison.empty:
        flagged = uncertainty_comparison[uncertainty_comparison["problem_flag"]]
        for (fit_label, parameter), row in flagged.iterrows():
            rows.append(
                {
                    "fit": fit_label,
                    "parameter": parameter,
                    "issue": "high_uncertainty_or_method_disagreement",
                    "detail": f"relative_std_pct >= {UQ_REL_STD_WARNING_PCT:g} or std_dev_ratio >= {UQ_STD_RATIO_WARNING:g}",
                    "estimate": row["estimate_max"],
                    "relative_std_pct_max": row["relative_std_pct_max"],
                    "std_dev_ratio_max_min": row["std_dev_ratio_max_min"],
                    "methods": row["methods"],
                }
            )

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index(["fit", "parameter", "issue"]).sort_index()


covariance_results = {}
uncertainty_tables = {}
bound_tables = {}
covariance_diagnostic_rows = []
print("Covariance batches:", UQ_COVARIANCE_BATCHES)
print("Covariance fixed parameters excluded:", sorted(UQ_FIXED_PARAMETERS))
print("Excluded from covariance solves:", sorted(UQ_EXCLUDED_BATCHES_FOR_COVARIANCE))
for fit_label, (obj_function, theta) in UQ_FITS.items():
    bound_summary = identify_active_bound_parameters(theta)
    covariance_parameters = covariance_parameter_names(bound_summary)
    bound_tables[fit_label] = bound_summary
    for method in UQ_METHODS:
        cov, uncertainty_summary, diagnostics = run_covariance_estimation(
            fit_label, obj_function, theta, covariance_parameters, method
        )
        diagnostics["n_active_bound_parameters"] = len(bound_summary) - len(covariance_parameters)
        covariance_results[(fit_label, method)] = cov
        uncertainty_tables[(fit_label, method)] = uncertainty_summary
        covariance_diagnostic_rows.append(diagnostics)

display(pd.concat(bound_tables, names=["fit", "parameter"]))
covariance_diagnostic_table = pd.DataFrame(covariance_diagnostic_rows).set_index(["fit", "method"]).sort_index()
display(covariance_diagnostic_table)

uncertainty_tables_nonempty = {label: table for label, table in uncertainty_tables.items() if not table.empty}
if uncertainty_tables_nonempty:
    display(pd.concat(uncertainty_tables_nonempty, names=["fit", "method", "parameter"]))

covariance_method_comparison = build_method_comparison_table(covariance_results, uncertainty_tables)
if not covariance_method_comparison.empty:
    display(covariance_method_comparison)
else:
    print("Covariance method comparison unavailable: fewer than two covariance methods returned matrices for each fit.")

parameter_uncertainty_by_method, parameter_uncertainty_method_comparison = build_parameter_method_variability(
    uncertainty_tables
)
if not parameter_uncertainty_method_comparison.empty:
    display(parameter_uncertainty_method_comparison)

problematic_parameter_diagnostics = build_problem_parameter_diagnostics(
    bound_tables, parameter_uncertainty_method_comparison
)
if not problematic_parameter_diagnostics.empty:
    display(problematic_parameter_diagnostics)

if DISPLAY_COVARIANCE_MATRICES:
    for (fit_label, method), cov in covariance_results.items():
        if cov is not None:
            print(f"Covariance matrix from {method} for {fit_label}:")
            display(cov)


## Residual and FIM-based estimability diagnostics

This section consolidates the first two checks before deeper objective-function exploration: residual contributions by state and parameter estimability from the covariance/FIM information. Large residual fractions point to outputs driving the fit; large covariance eigenvalues and dominant loadings identify weak parameter directions.

In [ ]:
if str(FERMENTATION_DIR) not in sys.path:
    sys.path.insert(0, str(FERMENTATION_DIR))

from estimability_tools import (
    build_profile_grid,
    covariance_estimability_summary,
    plot_profile_likelihood_profiles,
    summarize_profile_likelihood,
    summarize_residual_contributions,
)

residual_contribution_summary = summarize_residual_contributions(state_fit_comparison)
if not residual_contribution_summary.empty:
    display(residual_contribution_summary)

reliable_covariance_results = {
    key: cov
    for key, cov in covariance_results.items()
    if key[1] in {"finite_difference", "automatic_differentiation_kaug"} and cov is not None
}
fim_eigen_summary, weak_covariance_directions = covariance_estimability_summary(reliable_covariance_results)
if not fim_eigen_summary.empty:
    display(fim_eigen_summary)
if not weak_covariance_directions.empty:
    display(weak_covariance_directions)

if "problematic_parameter_diagnostics" in globals() and not problematic_parameter_diagnostics.empty:
    display(problematic_parameter_diagnostics)


## Profile likelihood estimability screening in effective-parameter space

The profile likelihood logic mirrors the class example: fix one parameter at a grid value, re-estimate all other parameters, and track the objective increase. Because `profile_likelihood` is not available in the installed ParmEst API, the loop is implemented explicitly. The default now profiles the reduced identifiable parameter subset selected above; set `PYOMO_DOE_PROFILE_MAX_PARAMETERS` only for quick debugging runs.

In [ ]:
RUN_PROFILE_LIKELIHOOD = True
RUN_PROFILE_OPTIMIZATION = True

PROFILE_OBJECTIVE = "SSE_weighted"
PROFILE_FIT_LABEL = "WSSE_profile"
PROFILE_BATCHES = Q_ANALYSIS_BATCHES if "Q_ANALYSIS_BATCHES" in globals() else BASE_FIT_BATCHES
PROFILE_EXCLUDED_PARAMETERS = set(Q_EXCLUDED_PARAMETERS) if "Q_EXCLUDED_PARAMETERS" in globals() else set()
if "BASE_FIT_FIXED_PARAMETERS" in globals():
    PROFILE_EXCLUDED_PARAMETERS.update(BASE_FIT_FIXED_PARAMETERS)
_profile_parameter_source = list(Q_ANALYSIS_PARAMETERS) if "Q_ANALYSIS_PARAMETERS" in globals() else list(DEFAULT_THETA)
if "problematic_parameter_diagnostics" in globals() and not problematic_parameter_diagnostics.empty:
    _profile_problem_parameters = []
    for _fit_label, _parameter, _issue in problematic_parameter_diagnostics.index:
        if _parameter in _profile_parameter_source and _parameter not in PROFILE_EXCLUDED_PARAMETERS:
            _profile_problem_parameters.append(_parameter)
    if _profile_problem_parameters:
        _profile_parameter_source = list(dict.fromkeys(_profile_problem_parameters))

PROFILE_MAX_PARAMETERS = int(os.environ.get("PYOMO_DOE_PROFILE_MAX_PARAMETERS", "0"))
PROFILE_PARAMETERS = [p for p in _profile_parameter_source if p not in PROFILE_EXCLUDED_PARAMETERS]
if PROFILE_MAX_PARAMETERS > 0:
    PROFILE_PARAMETERS = PROFILE_PARAMETERS[:PROFILE_MAX_PARAMETERS]
PROFILE_ESTIMATED_PARAMETERS = [p for p in DEFAULT_THETA if p not in PROFILE_EXCLUDED_PARAMETERS]

PROFILE_N_GRID = int(os.environ.get("PYOMO_DOE_PROFILE_N_GRID", "7"))
PROFILE_CONFIDENCE = 0.95
PROFILE_BASELINE_TOL = 1e-6
PROFILE_STOP_ON_BASELINE_FAILURE = True
PROFILE_TEE = False
PROFILE_USE_EXISTING_WSSE_BASELINE = True
PROFILE_VALIDATE_OBJECTIVE = True
PROFILE_OBJECTIVE_VALIDATION_RTOL = 1e-4
PROFILE_OBJECTIVE_VALIDATION_ATOL = 1e-5
PROFILE_AUTO_UPDATE_BASELINE_FROM_PROFILE = True
PROFILE_MAX_BASELINE_RESTARTS = 2
PROFILE_UPDATE_WSSE_BASELINE = True
PROFILE_DIAGNOSTIC_RELATIVE_THRESHOLD_PCT = 100.0  # Practical threshold: objective doubled.


def profile_std_guess(parameter: str, fit_label: str = "WSSE_fit"):
    if "parameter_uncertainty_method_comparison" not in globals() or parameter_uncertainty_method_comparison.empty:
        return None
    key = (fit_label, parameter)
    if key not in parameter_uncertainty_method_comparison.index:
        return None
    value = parameter_uncertainty_method_comparison.loc[key, "std_dev_max"]
    return float(value) if pd.notna(value) else None


def profile_objective_column(obj_function: str):
    if obj_function == "SSE_weighted":
        return "WSSE_raw_sum"
    if obj_function == "SSE":
        return "SSE_raw_sum"
    raise ValueError(f"No direct objective validation column is defined for {obj_function!r}.")


def reconcile_profile_objective_scale(obj_value, theta, obj_function=PROFILE_OBJECTIVE, batch_ids=PROFILE_BATCHES, context="profile"):
    obj_value = float(obj_value)
    check = {
        "objective_direct": np.nan,
        "objective_gap": np.nan,
        "objective_scale": "not_checked",
        "objective_consistent": True,
    }
    if not PROFILE_VALIDATE_OBJECTIVE:
        return obj_value, check

    objective_summary, _, solve_summary = evaluate_fit_objectives(theta, batch_ids=batch_ids)
    direct_col = profile_objective_column(obj_function)
    direct_obj = float(objective_summary.get(direct_col, np.nan))
    n_batches = len(normalize_batch_ids(batch_ids)) if "normalize_batch_ids" in globals() else len(list(batch_ids))
    check["objective_direct"] = direct_obj

    if np.isfinite(direct_obj) and np.isclose(
        obj_value,
        direct_obj,
        rtol=PROFILE_OBJECTIVE_VALIDATION_RTOL,
        atol=PROFILE_OBJECTIVE_VALIDATION_ATOL,
    ):
        check["objective_gap"] = direct_obj - obj_value
        check["objective_scale"] = "total"
        return obj_value, check

    scaled_obj = obj_value * n_batches
    if n_batches > 1 and np.isfinite(direct_obj) and np.isclose(
        scaled_obj,
        direct_obj,
        rtol=PROFILE_OBJECTIVE_VALIDATION_RTOL,
        atol=PROFILE_OBJECTIVE_VALIDATION_ATOL,
    ):
        check["objective_gap"] = direct_obj - scaled_obj
        check["objective_scale"] = "parmest_per_scenario_rescaled"
        print(
            f"[PROFILE] {context}: rescaled ParmEst objective from per-scenario average "
            f"{obj_value:.6g} to total {scaled_obj:.6g} using {n_batches} batches.",
            flush=True,
        )
        return float(scaled_obj), check

    check["objective_gap"] = direct_obj - obj_value if np.isfinite(direct_obj) else np.nan
    check["objective_scale"] = "mismatch"
    check["objective_consistent"] = False
    print(
        f"WARNING: {context} objective mismatch: ParmEst={obj_value:.6g}, "
        f"direct={direct_obj:.6g}. Direct solve statuses: "
        f"{solve_summary.reset_index().to_dict('records') if not solve_summary.empty else []}",
        flush=True,
    )
    return obj_value, check


def run_profile_likelihood_parameter(parameter, theta_hat, obj_hat, obj_function=PROFILE_OBJECTIVE, batch_ids=PROFILE_BATCHES, n_grid=PROFILE_N_GRID):
    grid = build_profile_grid(theta_hat, PARAMETER_BOUNDS, parameter, std_dev=profile_std_guess(parameter), n_grid=n_grid, relative_span=0.75)
    lb, ub = PARAMETER_BOUNDS[parameter]
    value_hat = float(theta_hat[parameter])
    grid_extras = []
    if value_hat > float(lb):
        grid_extras.append(max(float(lb), value_hat * 0.25))
    if value_hat < float(ub):
        grid_extras.append(min(float(ub), value_hat * 4.0))
    grid_candidates = np.sort(np.asarray(grid.tolist() + grid_extras + [value_hat], dtype=float))
    unique_grid = []
    for candidate in grid_candidates:
        if not unique_grid or not np.isclose(candidate, unique_grid[-1], rtol=1e-10, atol=1e-12):
            unique_grid.append(float(candidate))
    grid = np.asarray(unique_grid, dtype=float)
    free_parameters = [name for name in PROFILE_ESTIMATED_PARAMETERS if name != parameter]
    rows = []
    for grid_idx, value in enumerate(grid, start=1):
        print(f"[PROFILE] {parameter} grid {grid_idx}/{len(grid)} fixed={float(value):.6g}", flush=True)
        theta_start = dict(theta_hat)
        theta_start[parameter] = float(value)
        row = {
            "profiled_theta": parameter,
            "theta_value": float(value),
            "theta_hat": float(theta_hat[parameter]),
            "objective": np.nan,
            "lr_stat": np.nan,
            "success": False,
            "status": "failed",
            "error": "",
        }
        if np.isclose(float(value), float(theta_hat[parameter]), rtol=1e-8, atol=1e-10):
            theta_profile = dict(theta_hat)
            row.update(
                {
                    "objective": float(obj_hat),
                    "objective_direct": float(obj_hat),
                    "objective_gap": 0.0,
                    "objective_scale": "baseline",
                    "objective_consistent": True,
                    "lr_stat": 0.0,
                    "success": True,
                    "status": "theta_hat_baseline",
                }
            )
            row.update({f"estimated_{name}": theta_profile.get(name, np.nan) for name in free_parameters})
            print(f"[PROFILE] {parameter} grid {grid_idx}/{len(grid)} done | objective={float(obj_hat):.6g} | lr=0", flush=True)
            rows.append(row)
            continue
        try:
            obj_value, theta_raw, _ = run_parmest_estimation(
                obj_function,
                batch_ids=batch_ids,
                theta_initial=theta_start,
                parameters_to_estimate=free_parameters,
                tee=PROFILE_TEE,
            )
            theta_solution = dict(theta_start)
            theta_solution.update(theta_raw)
            theta_profile, _ = clip_theta_to_bounds(theta_solution)
            obj_value, objective_check = reconcile_profile_objective_scale(
                obj_value,
                theta_profile,
                obj_function=obj_function,
                batch_ids=batch_ids,
                context=f"{parameter} grid {grid_idx}/{len(grid)}",
            )
            row.update(objective_check)
            row["objective"] = float(obj_value)
            row["lr_stat"] = 2.0 * (float(obj_value) - float(obj_hat))
            row["success"] = bool(objective_check.get("objective_consistent", True))
            row["status"] = "optimal" if row["success"] else "objective_mismatch"
            row.update({f"estimated_{name}": theta_profile.get(name, np.nan) for name in free_parameters})
            if not row["success"]:
                row["error"] = "ParmEst objective did not match direct fixed-theta objective."
            print(f"[PROFILE] {parameter} grid {grid_idx}/{len(grid)} done | objective={float(obj_value):.6g} | lr={row['lr_stat']:.6g}", flush=True)
        except Exception as err:
            row["error"] = f"{type(err).__name__}: {err}"
        rows.append(row)
    return pd.DataFrame(rows)


def profile_lower_objective_row(profile_frame, obj_hat, tol=PROFILE_BASELINE_TOL):
    if profile_frame.empty or "success" not in profile_frame.columns:
        return np.nan, None
    successful = profile_frame[profile_frame["success"] & profile_frame["objective"].notna()]
    if successful.empty:
        return np.nan, None
    best_idx = successful["objective"].astype(float).idxmin()
    min_objective = float(successful.loc[best_idx, "objective"])
    if min_objective < float(obj_hat) - float(tol):
        return min_objective, successful.loc[best_idx]
    return min_objective, None


def theta_from_profile_row(row, theta_reference):
    theta = dict(theta_reference)
    profiled_name = str(row["profiled_theta"])
    theta[profiled_name] = float(row["theta_value"])
    for name in DEFAULT_THETA:
        col = f"estimated_{name}"
        if col in row.index and pd.notna(row[col]):
            theta[name] = float(row[col])
    return clip_theta_to_bounds(theta)


def polish_profile_baseline(theta_start, context="profile baseline restart"):
    obj_value, theta_raw, _ = run_parmest_estimation(
        PROFILE_OBJECTIVE,
        batch_ids=PROFILE_BATCHES,
        theta_initial=theta_start,
        parameters_to_estimate=PROFILE_ESTIMATED_PARAMETERS,
        tee=PROFILE_TEE,
    )
    theta_complete = dict(theta_start)
    theta_complete.update(theta_raw)
    theta_hat, theta_adjustments = clip_theta_to_bounds(theta_complete)
    obj_value, objective_check = reconcile_profile_objective_scale(
        obj_value,
        theta_hat,
        obj_function=PROFILE_OBJECTIVE,
        batch_ids=PROFILE_BATCHES,
        context=context,
    )
    if not objective_check.get("objective_consistent", True):
        raise RuntimeError(f"{context} objective is inconsistent with direct fixed-theta simulation.")
    return float(obj_value), theta_hat, theta_adjustments, objective_check


def add_profile_diagnostic_columns(profile_frame, obj_hat):
    frame = profile_frame.copy()
    obj_hat = float(obj_hat)
    denom = max(abs(obj_hat), np.finfo(float).eps)
    frame["objective_increase"] = frame["objective"] - obj_hat
    frame["relative_objective_increase_pct"] = 100.0 * frame["objective_increase"] / denom
    frame["relative_objective_increase_pct"] = frame["relative_objective_increase_pct"].clip(lower=0.0)
    return frame


def summarize_profile_diagnostics(profile_frame, relative_threshold_pct=PROFILE_DIAGNOSTIC_RELATIVE_THRESHOLD_PCT):
    ok = profile_frame[profile_frame["success"]].copy() if profile_frame is not None and not profile_frame.empty else pd.DataFrame()
    if ok.empty:
        return pd.DataFrame()
    rows = []
    for parameter, group in ok.groupby("profiled_theta"):
        theta_hat = float(group["theta_hat"].iloc[0])
        left = group[group["theta_value"] < theta_hat]
        right = group[group["theta_value"] > theta_hat]
        rows.append(
            {
                "profiled_theta": parameter,
                "n_success": len(group),
                "theta_hat": theta_hat,
                "min_profile_objective": group["objective"].min(),
                "max_relative_objective_increase_pct": group["relative_objective_increase_pct"].max(),
                "crosses_left_practical": bool((left["relative_objective_increase_pct"] >= relative_threshold_pct).any()),
                "crosses_right_practical": bool((right["relative_objective_increase_pct"] >= relative_threshold_pct).any()),
                "relative_threshold_pct": float(relative_threshold_pct),
            }
        )
    return pd.DataFrame(rows).set_index("profiled_theta").sort_index()


def plot_profile_diagnostic_profiles(profile_frame, relative_threshold_pct=PROFILE_DIAGNOSTIC_RELATIVE_THRESHOLD_PCT):
    ok = profile_frame[profile_frame["success"]].copy() if profile_frame is not None and not profile_frame.empty else pd.DataFrame()
    if ok.empty:
        return None, None
    parameters = list(ok["profiled_theta"].drop_duplicates())
    ncols = 2
    nrows = int(np.ceil(len(parameters) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3.8 * nrows), squeeze=False)
    axes_flat = axes.ravel()
    for ax, parameter in zip(axes_flat, parameters):
        group = ok[ok["profiled_theta"].eq(parameter)].sort_values("theta_value")
        ax.plot(group["theta_value"], group["relative_objective_increase_pct"], marker="o")
        ax.axhline(relative_threshold_pct, color="tab:red", linestyle="--", label=f"{relative_threshold_pct:g}% practical")
        ax.axvline(float(group["theta_hat"].iloc[0]), color="black", linestyle=":", label="theta_hat")
        ax.set_title(parameter)
        ax.set_xlabel("fixed parameter value")
        ax.set_ylabel("objective increase (%)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
    for ax in axes_flat[len(parameters):]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes


if RUN_PROFILE_OPTIMIZATION or "profile_results" not in globals():
    profile_results = {"profiles": pd.DataFrame(), "theta_hat": {}, "obj_hat": np.nan}

if RUN_PROFILE_LIKELIHOOD:
    profile_display_columns = [
        "profiled_theta",
        "theta_value",
        "theta_hat",
        "objective",
        "objective_direct",
        "objective_gap",
        "objective_scale",
        "objective_consistent",
        "lr_stat",
        "success",
        "status",
        "error",
    ]
    profile_can_update_wsse = False

    if RUN_PROFILE_OPTIMIZATION:
        if "theta_WSSE" not in globals():
            raise RuntimeError("Run the Gaussian WSSE local polish block before profile likelihood.")
        profile_initial_theta = dict(DEFAULT_THETA)
        profile_initial_theta.update(theta_WSSE)
        profile_initial_theta = {name: float(profile_initial_theta[name]) for name in DEFAULT_THETA}

        print("Profile initial theta source:", globals().get("theta_WSSE_source", "theta_WSSE"))
        print("Profile batches:", PROFILE_BATCHES)
        print("Profile profiled parameters:", PROFILE_PARAMETERS)
        print("Profile fixed/excluded parameters:", sorted(PROFILE_EXCLUDED_PARAMETERS))
        print("Profile estimated parameters:", PROFILE_ESTIMATED_PARAMETERS)

        same_batches_as_base = "BASE_FIT_BATCHES" in globals() and list(PROFILE_BATCHES) == list(BASE_FIT_BATCHES)
        same_parameters_as_base = "BASE_FIT_PARAMETERS" in globals() and set(PROFILE_ESTIMATED_PARAMETERS) == set(BASE_FIT_PARAMETERS)
        profile_can_update_wsse = bool(same_batches_as_base and same_parameters_as_base)
        if PROFILE_USE_EXISTING_WSSE_BASELINE and same_batches_as_base and same_parameters_as_base and "obj_WSSE" in globals() and np.isfinite(obj_WSSE):
            profile_obj_hat = float(obj_WSSE)
            profile_theta_hat = dict(profile_initial_theta)
            profile_theta_adjustments = pd.DataFrame()
            print("Profile baseline source: Gaussian WSSE local polish objective")
        else:
            profile_obj_hat, profile_theta_raw, _ = run_parmest_estimation(
                PROFILE_OBJECTIVE,
                batch_ids=PROFILE_BATCHES,
                theta_initial=profile_initial_theta,
                parameters_to_estimate=PROFILE_ESTIMATED_PARAMETERS,
            )
            profile_theta_complete = dict(profile_initial_theta)
            profile_theta_complete.update(profile_theta_raw)
            profile_theta_hat, profile_theta_adjustments = clip_theta_to_bounds(profile_theta_complete)
            print("Profile baseline source: fresh ParmEst solve")

        profile_obj_hat, profile_baseline_objective_check = reconcile_profile_objective_scale(
            profile_obj_hat,
            profile_theta_hat,
            obj_function=PROFILE_OBJECTIVE,
            batch_ids=PROFILE_BATCHES,
            context="profile baseline",
        )
        if not profile_baseline_objective_check.get("objective_consistent", True):
            raise RuntimeError("Profile baseline objective is inconsistent with direct fixed-theta simulation.")

        profile_results["theta_hat"] = profile_theta_hat
        profile_results["obj_hat"] = float(profile_obj_hat)
        profile_results["baseline_objective_check"] = profile_baseline_objective_check
        print("Profile objective at theta_hat:", profile_obj_hat)
        if not profile_theta_adjustments.empty:
            display(profile_theta_adjustments)
        profile_profiles = pd.concat(
            [run_profile_likelihood_parameter(parameter, profile_theta_hat, profile_obj_hat) for parameter in PROFILE_PARAMETERS],
            ignore_index=True,
        )
        profile_results["profiles"] = profile_profiles
    else:
        if "profile_profiles" not in globals() or "profile_results" not in globals():
            raise RuntimeError("Cached profile likelihood results are not available. Set RUN_PROFILE_OPTIMIZATION = True.")
        profile_profiles = profile_profiles.copy()
        profile_obj_hat = float(profile_results.get("obj_hat", np.nan))
        profile_theta_hat = profile_results.get("theta_hat", {})
        if not np.isfinite(profile_obj_hat):
            raise RuntimeError("Cached profile_results does not contain a valid obj_hat.")
        print("Reusing cached profile likelihood results; no ParmEst/Ipopt solves were run.")

    profile_baseline_restart_rows = []
    baseline_restart_count = 0
    while True:
        min_profile_objective, best_profile_row = profile_lower_objective_row(
            profile_profiles,
            profile_obj_hat,
            tol=PROFILE_BASELINE_TOL,
        )
        profile_results["profiles"] = profile_profiles
        profile_results["theta_hat"] = profile_theta_hat
        profile_results["obj_hat"] = float(profile_obj_hat)
        if best_profile_row is None:
            break

        restart_row = {
            "restart": baseline_restart_count + 1,
            "old_obj_hat": float(profile_obj_hat),
            "min_profile_objective": float(min_profile_objective),
            "objective_gap": float(profile_obj_hat - min_profile_objective),
            "best_profiled_theta": best_profile_row["profiled_theta"],
            "best_theta_value": float(best_profile_row["theta_value"]),
        }
        display(pd.DataFrame([restart_row]))
        baseline_message = (
            "Profile likelihood found a lower objective than obj_hat: "
            f"{min_profile_objective:.6g} < {float(profile_obj_hat):.6g}. "
            f"Best profiled point: {best_profile_row['profiled_theta']}={best_profile_row['theta_value']:.6g}."
        )

        can_restart = (
            RUN_PROFILE_OPTIMIZATION
            and PROFILE_AUTO_UPDATE_BASELINE_FROM_PROFILE
            and baseline_restart_count < int(PROFILE_MAX_BASELINE_RESTARTS)
        )
        if not can_restart:
            final_message = (
                baseline_message
                + " Baseline could not be auto-updated further; do not interpret these profiles yet."
            )
            if PROFILE_STOP_ON_BASELINE_FAILURE:
                raise RuntimeError(final_message)
            print("WARNING:", final_message)
            print("Profile plots below are diagnostic only; confidence intervals are not valid until obj_hat is updated.")
            break

        print("WARNING:", baseline_message)
        print("Updating profile baseline from the best profiled point and rerunning the profiles.")
        restart_theta_start, restart_theta_adjustments = theta_from_profile_row(best_profile_row, profile_theta_hat)
        if not restart_theta_adjustments.empty:
            display(pd.concat({"profile_restart_start": restart_theta_adjustments}, names=["fit", "row"]))
        updated_obj_hat, updated_theta_hat, updated_adjustments, updated_check = polish_profile_baseline(
            restart_theta_start,
            context=f"profile baseline restart {baseline_restart_count + 1}",
        )
        restart_row.update(
            {
                "updated_obj_hat": float(updated_obj_hat),
                "updated_objective_direct": float(updated_check.get("objective_direct", np.nan)),
                "updated_objective_gap": float(updated_check.get("objective_gap", np.nan)),
                "updated_objective_scale": updated_check.get("objective_scale", ""),
            }
        )
        profile_baseline_restart_rows.append(restart_row)
        profile_obj_hat = float(updated_obj_hat)
        profile_theta_hat = dict(updated_theta_hat)
        profile_baseline_objective_check = updated_check
        profile_results["theta_hat"] = profile_theta_hat
        profile_results["obj_hat"] = float(profile_obj_hat)
        profile_results["baseline_objective_check"] = profile_baseline_objective_check

        if PROFILE_UPDATE_WSSE_BASELINE and profile_can_update_wsse:
            theta_WSSE = dict(profile_theta_hat)
            obj_WSSE = float(profile_obj_hat)
            print("Updated theta_WSSE/obj_WSSE from profile baseline restart.")
        print("Updated profile objective at theta_hat:", profile_obj_hat)
        if not updated_adjustments.empty:
            display(pd.concat({"profile_restart": updated_adjustments}, names=["fit", "row"]))

        baseline_restart_count += 1
        profile_profiles = pd.concat(
            [run_profile_likelihood_parameter(parameter, profile_theta_hat, profile_obj_hat) for parameter in PROFILE_PARAMETERS],
            ignore_index=True,
        )

    profile_results["baseline_restarts"] = pd.DataFrame(profile_baseline_restart_rows)
    profile_results["baseline_updated_from_profile"] = bool(profile_baseline_restart_rows)
    if profile_baseline_restart_rows:
        display(profile_results["baseline_restarts"])
        print(
            "Profile baseline was updated from a lower profiled objective. "
            "Rerun the fit diagnostics/post-fit Q/UQ cells if you need those tables at the updated theta_WSSE."
        )
    profile_profiles = add_profile_diagnostic_columns(profile_profiles, profile_obj_hat)
    profile_results["profiles"] = profile_profiles
    for col in ["objective_increase", "relative_objective_increase_pct"]:
        if col not in profile_display_columns:
            insert_at = profile_display_columns.index("lr_stat") if "lr_stat" in profile_display_columns else len(profile_display_columns)
            profile_display_columns.insert(insert_at, col)
    display(profile_profiles[[col for col in profile_display_columns if col in profile_profiles.columns]])
    profile_summary = summarize_profile_likelihood(profile_profiles, alpha=PROFILE_CONFIDENCE)
    profile_diagnostic_summary = summarize_profile_diagnostics(profile_profiles)
    profile_objective_is_gaussian = bool(
        OBJECTIVE_SCALE_MODE in {"measurement_error", "gaussian"}
        and not OBJECTIVE_NORMALIZE_BY_STATE_COUNT
        and not OBJECTIVE_NORMALIZE_BY_EXPERIMENT_STATE_COUNT
    )
    if not profile_summary.empty:
        if profile_objective_is_gaussian:
            print("Formal chi-square profile summary:")
        else:
            print("Formal chi-square profile summary (diagnostic only with the current scaled objective):")
        display(profile_summary)
    if not profile_diagnostic_summary.empty:
        print(f"Practical profile summary: {PROFILE_DIAGNOSTIC_RELATIVE_THRESHOLD_PCT:g}% relative objective increase threshold.")
        display(profile_diagnostic_summary)
    PROFILE_RESULTS_DIR = FERMENTATION_DIR / "results" / "identifiability_reduction"
    PROFILE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    profile_profiles.to_csv(PROFILE_RESULTS_DIR / "notebook_profile_profiles.csv", index=False)
    profile_summary.to_csv(PROFILE_RESULTS_DIR / "notebook_profile_summary.csv")
    profile_diagnostic_summary.to_csv(PROFILE_RESULTS_DIR / "notebook_profile_diagnostic_summary.csv")
    if "fim_relative" in globals() and not profile_summary.empty:
        bayes_parameters = [name for name in PROFILE_PARAMETERS if name in fim_relative.index]
        prior_theta = _theta_for_batch(load_batch(PROFILE_BATCHES[0]), DEFAULT_THETA)
        prior_log_sd = float(os.environ.get("PYOMO_DOE_BAYES_PRIOR_LOG_SD", "1.0"))
        fim_log = fim_relative.loc[bayes_parameters, bayes_parameters].to_numpy(dtype=float)
        prior_precision = np.eye(len(bayes_parameters)) / prior_log_sd**2
        posterior_cov = np.linalg.inv(fim_log + prior_precision)
        mle_log = np.array([np.log(float(profile_theta_hat[name])) for name in bayes_parameters], dtype=float)
        prior_log = np.array([np.log(float(prior_theta[name])) for name in bayes_parameters], dtype=float)
        posterior_log_mean = posterior_cov @ (fim_log @ mle_log + prior_precision @ prior_log)
        posterior_log_sd = np.sqrt(np.diag(posterior_cov))
        bayesian_laplace_summary = pd.DataFrame(
            {
                "theta_mle": [float(profile_theta_hat[name]) for name in bayes_parameters],
                "prior_median_nominal": [float(prior_theta[name]) for name in bayes_parameters],
                "posterior_median_laplace": np.exp(posterior_log_mean),
                "posterior_log_sd": posterior_log_sd,
                "posterior_95_lower": np.exp(posterior_log_mean - 1.96 * posterior_log_sd),
                "posterior_95_upper": np.exp(posterior_log_mean + 1.96 * posterior_log_sd),
                "posterior_var_over_prior_var": np.diag(posterior_cov) / prior_log_sd**2,
            },
            index=bayes_parameters,
        )
        bayesian_laplace_summary["data_dominated"] = bayesian_laplace_summary["posterior_var_over_prior_var"] < 0.10
        print("Basic Bayesian/Laplace diagnostic with weak lognormal priors:")
        display(bayesian_laplace_summary)
        bayesian_laplace_summary.to_csv(PROFILE_RESULTS_DIR / "notebook_bayesian_laplace_summary.csv")
    if profile_objective_is_gaussian:
        plot_profile_likelihood_profiles(profile_profiles, alpha=PROFILE_CONFIDENCE);
    else:
        print("NOTE: The chi-square threshold is not a formal confidence cutoff because the objective is scaled by max/state counts, not measurement noise.")
        plot_profile_diagnostic_profiles(profile_profiles);
else:
    print("Profile likelihood skipped. Set RUN_PROFILE_LIKELIHOOD = True to execute the screening profiles.")


## Post-profile PSO on estimable parameters

After a valid profile-likelihood pass, optionally run the custom PSO only on parameters whose profiles cross the chi-square threshold on both sides. Non-estimable parameters remain fixed at the current profile baseline, and the PSO result can be polished with ParmEst/IPOPT.

In [ ]:
RUN_POST_PROFILE_PSO = False
POST_PROFILE_PSO_REQUIRE_VALID_BASELINE = True
POST_PROFILE_PSO_POLISH_WITH_PARMEST = True
POST_PROFILE_PSO_RESULTS_PREFIX = "post_profile_pso"
POST_PROFILE_PSO_BATCHES = PROFILE_BATCHES if "PROFILE_BATCHES" in globals() else BASE_FIT_BATCHES

post_profile_pso_diagnostic = {
    "run_requested": bool(RUN_POST_PROFILE_PSO),
    "status": "not_started",
    "reason": "",
}

if "profile_summary" not in globals() or profile_summary.empty:
    post_profile_pso_diagnostic.update({"status": "skipped", "reason": "profile_summary is not available"})
    display(pd.Series(post_profile_pso_diagnostic, name="post_profile_pso_diagnostic"))
elif "run_custom_pso" not in globals():
    post_profile_pso_diagnostic.update({"status": "skipped", "reason": "run the deferred custom PSO helper cell first"})
    display(pd.Series(post_profile_pso_diagnostic, name="post_profile_pso_diagnostic"))
else:
    post_profile_estimability = profile_summary.copy()
    post_profile_estimability["profile_estimable"] = post_profile_estimability["crosses_left"] & post_profile_estimability["crosses_right"]
    POST_PROFILE_PSO_PARAMETERS = [
        name for name in post_profile_estimability.index[post_profile_estimability["profile_estimable"]].tolist()
        if name in DEFAULT_THETA
    ]
    display(post_profile_estimability)

    baseline_valid = True
    if "min_profile_objective" in globals() and "profile_obj_hat" in globals():
        baseline_valid = pd.isna(min_profile_objective) or float(min_profile_objective) >= float(profile_obj_hat) - PROFILE_BASELINE_TOL

    if POST_PROFILE_PSO_REQUIRE_VALID_BASELINE and not baseline_valid:
        post_profile_pso_diagnostic.update({
            "status": "skipped",
            "reason": "profile baseline is not valid; rebase/reoptimize theta_hat before PSO",
            "n_estimable_parameters": len(POST_PROFILE_PSO_PARAMETERS),
        })
        display(pd.Series(post_profile_pso_diagnostic, name="post_profile_pso_diagnostic"))
    elif not POST_PROFILE_PSO_PARAMETERS:
        post_profile_pso_diagnostic.update({"status": "skipped", "reason": "no parameters crossed both profile sides"})
        display(pd.Series(post_profile_pso_diagnostic, name="post_profile_pso_diagnostic"))
    elif not RUN_POST_PROFILE_PSO:
        post_profile_pso_diagnostic.update({
            "status": "ready",
            "reason": "set RUN_POST_PROFILE_PSO = True to launch PSO on profile-estimable parameters",
            "n_estimable_parameters": len(POST_PROFILE_PSO_PARAMETERS),
            "estimable_parameters": ", ".join(POST_PROFILE_PSO_PARAMETERS),
        })
        display(pd.Series(post_profile_pso_diagnostic, name="post_profile_pso_diagnostic"))
    else:
        if "profile_theta_hat" in globals() and profile_theta_hat:
            post_profile_reference_theta = dict(profile_theta_hat)
        else:
            post_profile_reference_theta = dict(theta_WSSE)

        CUSTOM_PSO_PARAMETERS = list(POST_PROFILE_PSO_PARAMETERS)
        CUSTOM_PSO_FIXED_PARAMETERS = set(DEFAULT_THETA) - set(CUSTOM_PSO_PARAMETERS)
        CUSTOM_PSO_BATCHES = normalize_batch_ids(POST_PROFILE_PSO_BATCHES)
        CUSTOM_PSO_LOG_PARAMETERS = [name for name in CUSTOM_PSO_PARAMETERS if PARAMETER_BOUNDS[name][0] > 0.0]
        CUSTOM_PSO_REFERENCE_THETA = dict(post_profile_reference_theta)
        CUSTOM_PSO_INCLUDE_WSSE_SEED = True
        CUSTOM_PSO_RESULT_PREFIX = POST_PROFILE_PSO_RESULTS_PREFIX

        print("Post-profile PSO batches:", CUSTOM_PSO_BATCHES)
        print("Post-profile PSO estimated parameters:", CUSTOM_PSO_PARAMETERS)
        print("Post-profile PSO fixed parameters:", sorted(CUSTOM_PSO_FIXED_PARAMETERS))
        post_profile_pso_results = run_custom_pso()
        theta_post_profile_pso_best = post_profile_pso_results["best_theta"]
        obj_post_profile_pso_best = float(post_profile_pso_results["best_objective"])
        display(pd.Series(theta_post_profile_pso_best, name="theta_post_profile_pso_best"))
        display(pd.Series({
            "obj_post_profile_pso_best": obj_post_profile_pso_best,
            "n_evaluations": post_profile_pso_results["n_evaluations"],
            "n_successful_evaluations": post_profile_pso_results["n_successful_evaluations"],
            "n_failed_evaluations": post_profile_pso_results["n_failed_evaluations"],
            "n_cache_hits": post_profile_pso_results["n_cache_hits"],
        }, name="post_profile_pso_summary"))

        if POST_PROFILE_PSO_POLISH_WITH_PARMEST and np.isfinite(obj_post_profile_pso_best):
            print("Polishing post-profile PSO point with ParmEst/Ipopt...")
            obj_post_profile_pso_polished, theta_post_profile_pso_polished_raw, _ = run_parmest_estimation(
                PROFILE_OBJECTIVE,
                batch_ids=CUSTOM_PSO_BATCHES,
                theta_initial=theta_post_profile_pso_best,
                parameters_to_estimate=CUSTOM_PSO_PARAMETERS,
            )
            theta_post_profile_pso_polished = dict(theta_post_profile_pso_best)
            theta_post_profile_pso_polished.update(theta_post_profile_pso_polished_raw)
            theta_post_profile_pso_polished, theta_post_profile_pso_adjustments = clip_theta_to_bounds(theta_post_profile_pso_polished)
            display(pd.Series(theta_post_profile_pso_polished, name="theta_post_profile_pso_polished"))
            print("Polished post-profile PSO objective:", obj_post_profile_pso_polished)
            if not theta_post_profile_pso_adjustments.empty:
                display(theta_post_profile_pso_adjustments)

            CUSTOM_PSO_RESULTS_DIR.mkdir(exist_ok=True)
            pd.Series(theta_post_profile_pso_polished, name="theta_post_profile_pso_polished").to_csv(
                CUSTOM_PSO_RESULTS_DIR / f"{POST_PROFILE_PSO_RESULTS_PREFIX}_polished_theta.csv"
            )
            pd.Series({"objective": float(obj_post_profile_pso_polished)}, name="post_profile_pso_polished_objective").to_csv(
                CUSTOM_PSO_RESULTS_DIR / f"{POST_PROFILE_PSO_RESULTS_PREFIX}_polished_objective.csv"
            )


## Next implementation steps

1. Run the initial Gaussian WSSE calibration directly with ParmEst/IPOPT from the nominal/reference point in effective-parameter space.
2. Use Q/eigen, covariance, and profile-likelihood diagnostics to separate estimable from weak/non-estimable effective parameters.
3. If the profile baseline is valid, optionally run post-profile PSO on only the profile-estimable effective parameters and polish with ParmEst.
4. Back-transform only the well-behaved effective fits to the physical Zenteno/MATLAB parameter names for interpretation.
